# FLP Definitive — GA-ALNS Recovery (NSGA-II + ALNS)

This is the **definitive** notebook using **NSGA-II multi-objective evolutionary recovery**.
Boosted ALNS/Tabu hyperparameters. Bugs fixed: `_calculate_weighted_score` was
called by Phase 2/3's ranking but never defined (guaranteed crash); the ZPF
crater was 6x6 instead of the intended 3x3; `_place_order`'s machine-relocation
search expanded past the disruption zone's r2 boundary instead of stopping there;
`connect_deep_polish`'s stash-fallback stage and Phase 3's `all_placed`
recomputation both regressed a confirmed feasibility bug; `ensure_full_connectivity`
spawned an unneeded stash for machines with no revenue-requiring outgoing flow.


---

## POOL REWORK (work in progress)

Fork of the `_fixed` notebook. Three coupled changes:

1. **Global logistics pool** — material deposited in any stash / depot_loader /
   main_unit input is available from any depot_unloader / main_unit output.
   Transfer is instant and free; loaders and unloaders are edge-only.
2. **Self-fuelling power** — thermal-bank fuel demand feeds back through the
   BOM expansion, so the factory manufactures the batteries it burns.
3. **Real build costs** for the logistics machines, so depots are not free.

The `_fixed` notebooks remain the reference implementation.


## 1. Imports & Constants

In [ ]:
from __future__ import annotations

import heapq
import math
import random
from collections import defaultdict, deque
from dataclasses import dataclass, field
from enum import Enum, auto
from typing import Optional

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("[WARN] matplotlib not available — visualisation will be skipped.")

# Materials that are NOT physically conveyed on belts. `power` is produced by
# ther_bank as a recipe output, but electricity does not ride a conveyor: leaving
# it in the flow matrix created a fake belt with intensity=1100 that accounted for
# ~98% of total MHC, so the GA's third objective was effectively "minimise the
# distance from the thermal bank to the main unit".
NON_CONVEYED_MATERIALS: set[str] = {'power'}

# Battery power output table (from database 'machines prod' sheet)
BATTERY_POWER: dict[str, int] = {
    'power':      0,
    'LC_bat':   220,
    'SC_bat':   420,
    'HC_bat':  1100,
    'LC_W_bat': 1600,
    'SC_W_bat': 3200,
    'ori_ore':    50,
}

print(f"Machine data tables loaded.")


## 1.5  Control Panel — every tunable parameter

**This is the only cell you need to edit to retune a run.** Everything below it
reads from `CFG` (or the module constants defined here); nothing downstream
hard-codes a parameter.

| Group | Drives | Typical reason to touch it |
|---|---|---|
| `RIGOUR` | the whole run | `"quick"` ≈ 5–10 min smoke test, `"full"` for reported results |
| Geometry & time | grid, BLF placement, belt travel | bigger grid, tighter machine spacing |
| Product mix & BOM | `derive_rate_balanced_bom` | **how large and how complex layouts get** |
| GA / NSGA-II | `run_nsga2` (Phase 1) | search effort, convergence |
| Layout repair | `evaluate_layout` (ALNS + Tabu) | routing completeness vs runtime |
| Feasibility gates | `_feasibility_ratio`, `evaluate_layout` | how strict "feasible" is |
| ZPF disruption | `zpf_recovery` (Phase 4) | blast size, zone radii, recovery budget |
| Simulation & power | `simulate_layout` (Phase 5) | tick budget, power economy |
| Phase widths | Phases 2–6 | how many candidates survive to each stage |

**Not here** (model data, not tuning): `MACHINE_CATALOGUE` and `REVENUE_TABLE`
in §4, `BATTERY_POWER` and `NON_CONVEYED_MATERIALS` in §1, `TARGET_PRODUCTS` and
`ALL_RECIPES` in §17 (both derived from the catalogue).

### Calibration notes, measured on this code

- **BOM size vs rate is a step function with a flat floor.** Every product gives
  an identical BOM from rate 0.02 to 0.10; the first step is ~0.15, then ~0.25,
  ~0.40, ~0.60. Setting `MAX_TERMINAL_RATE` below 0.15 makes the rate gene inert.
- **`MIN_TERMINAL_RATE` only affects generation 0.** `custom_mutate` clamps to
  `max(0.0, ...)`, not to `MIN`, so rates drift freely below it afterwards.
  `MAX_TERMINAL_RATE` is the enforced ceiling.
- **`MAX_MACHINES` is usually the binding constraint.** Over-cap individuals
  return a flat penalty *without ever building a layout*. At `0.15–0.45` with
  `PRODUCT_ACTIVATION_PROB=0.4`, **58%** of an initial population is over a cap
  of 160 (median BOM 183). Raising the cap to 300 drops that to 12%.
- **Complexity is limited by routing, not space.** At 50 machines the grid is
  only 17% occupied and nothing fails to place, but routing completeness falls
  to 23%. `IMPORT_DEPOTS` is the lever that recovers it.


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CONTROL PANEL — every tunable parameter for every phase of the pipeline
# ═══════════════════════════════════════════════════════════════════════════
#  Phase map:
#    Phase 1  run_nsga2()            -> GA / NSGA-II group
#    Phase 2  evaluate_layout(repair=True) -> Layout repair + Phase widths
#    Phase 3  connect_deep_polish / ensure_full_connectivity -> Layout repair
#    Phase 4  zpf_recovery()         -> ZPF disruption group
#    Phase 5  simulate_layout()      -> Simulation & power group
#    Phase 6  plot_zpf_sequence()    -> Phase widths
# ═══════════════════════════════════════════════════════════════════════════
import numpy as np

# ── FACTORY GEOMETRY & TIME ──────────────────────────────────────────────────
# Used by FactoryGrid, blf_decode (placement) and every belt-travel calculation.
GRID_W              = 70     # grid width  in cells
GRID_H              = 70     # grid height in cells
MIN_MACHINE_SPACING = 1      # clearance kept around every machine, in cells
TICK_SECONDS        = 2.0    # wall-clock seconds represented by one tick
BELT_SPEED          = 1      # ticks to traverse one 1x1 belt cell
SIM_EQ_MAX_TICKS    = 300    # Gate 2 budget; overridden by CFG below

# Power is not charged for the first CFG.POWER_GRACE_TICKS ticks of a
# simulation, letting the pipeline fill before the bill starts. The grace
# period is deliberately a FIXED TICK COUNT, not "until equilibrium": gating a
# cost on equilibrium is exploitable in the worst direction, since a layout that
# never stabilises would never start paying, and starving machines is precisely
# how a layout avoids stabilising. A fixed window cannot be manipulated by the
# layout being evaluated.
STARTUP_FREE        = True

@dataclass
class FLPConfig:
    # --- Sorting Weights ---
    WEIGHT_FEASIBILITY: float = 100.0
    WEIGHT_REVENUE: float = 0.2
    # --- Multiprocessing Toggle ---
    MULTIPROCESSING: bool = True
    N_JOBS: int = -1  # -1 uses all available cores
    # --- reproducibility --------------------------------------------------
    RANDOM_SEED: int = 42          # seeds population init, crossover, mutation,
                                    # AND (deterministically, per genotype) the
                                    # ZPF disruption-recovery search below.

    # --- GA (NSGA-II) -------------------------------------------------------
    POP_SIZE: int = 200
    NGEN: int = 50
    CXPB: float = 0.7              # crossover probability
    MUTPB: float = 0.2             # mutation probability
    MAX_MACHINES: int = 160        # hard cap on machines/individual
    # (minimise build_cost, maximise terminal_rate, minimise MHC)
    #
    # Regret used to be a fourth objective here, permanently pinned at 0.0
    # because it is BY DESIGN a post-bombing quantity: Phase 1 generates
    # layouts, Phase 4 bombs them, and only then does regret exist. Declaring
    # it during evolution did not make the search resilience-aware -- it just
    # handed NSGA-II a constant axis on which every individual ties, which
    # inflates the non-dominated front and flattens crowding distance, diluting
    # selection pressure on the three objectives that are actually measured.
    # Regret is still computed and reported in Phase 4 (and, from Phase 5, also
    # against simulated throughput); it is simply no longer pretending to steer
    # a search that never sees it.
    OBJ_WEIGHTS: tuple = (-1.0, 1.0, -1.0)

    # --- ZPF disruption zone (Four-zone Euclidean model) ---------------------
    ZPF_X: int = 22
    ZPF_Y: int = 22
    ZPF_W: int = 3   # bomb footprint -- must stay completely empty
    ZPF_H: int = 3
    ZPF_R1: float = 5.0
    ZPF_R2: float = 12.0
    # Budget shared by BOTH recovery variants -- they must match or the
    # comparison measures the budget rather than the method. 300 x 5 scenarios
    # x 5 layouts = 7,500 evaluations per Phase 4.
    #
    # Prefer spending a cut on DEPTH rather than on scenarios: the regret
    # distribution is the evidence, the search depth is only how hard each
    # point in it was worked. ZPF_EARLY_STOP below makes the depth choice
    # self-justifying -- if recoveries routinely stop early, 300 is generous;
    # if they routinely run to the wall, raise it.
    ZPF_ITERATIONS: int = 300   # total recovery evaluations
    # Evaluations without improving the RECORD before a recovery gives up.
    # 0 disables. Reported per run as recovery_log["stopped_early"], so the
    # budget actually used can be quoted rather than assumed.
    ZPF_EARLY_STOP: int = 100
    ZPF_POP_SIZE:   int = 20    # NSGA-II recovery population

    # --- ZPF: keep the supply plan fixed across the disruption ----------------
    # zpf_recovery used to call derive_flow_matrix() from scratch on the
    # surviving machines. That matcher is greedy and ORDER-DEPENDENT, and
    # recovery returns `placed` in a different order (stable machines first,
    # then movable), so the pairings changed even when the bomb destroyed
    # nothing and moved nothing. Demonstrated on a 14-machine layout with a
    # crater that touched no machine at all:
    #
    #   identical machine positions : True   (0 moved, 0 destroyed)
    #   flow pairings identical     : False  (4 dropped, 4 new)
    #   routed 14 -> 12,  feasibility 100% -> 96%
    #
    # With this True the pre-bomb pairings are carried through and only the
    # BELTS are re-routed, so pre- and post-bomb are the same graph and regret
    # measures the disruption instead of a reshuffle. Flows whose endpoints did
    # not survive are dropped. Set False if you want the factory to be allowed
    # to re-plan which machine feeds which after damage -- but then pre/post
    # numbers are not directly comparable.
    ZPF_PRESERVE_FLOWS: bool = True

    # --- bombing scenarios (Phase 4) -------------------------------------------
    # A single crater at a fixed coordinate is one sample, and usually a null
    # one: measured on real layouts, a 3x3 crater at (22,22) destroyed NOTHING
    # in three of five layouts. Uniform-random placement is unbiased but wastes
    # most of the budget -- layouts occupy 6-12% of the grid, so only 12-24% of
    # uniform craters hit anything, and ~73 of 100 recoveries would be computed
    # from no damage.
    #
    # Positions are therefore sampled uniformly and REJECTED unless they destroy
    # at least one machine. Sampling is free; only the recovery is expensive, so
    # every recovery run does real work. This keeps the unbiased geometry --
    # dense clusters and large machines are hit more often, as in a real
    # accident -- unlike centring on a uniformly-chosen machine, which
    # over-samples isolated ones.
    #
    # The scenario set is generated from a seed derived from the LAYOUT, so GA
    # and SA face identical craters on identical layouts. Comparing recovery
    # methods across different disasters would be meaningless.
    ZPF_SCENARIOS:        int  = 5      # craters per layout; 1 = single-bomb behaviour
    ZPF_REQUIRE_DAMAGE:   bool = True   # reject craters that destroy nothing
    ZPF_SAMPLE_ATTEMPTS:  int  = 4000   # rejection-sampling cap per scenario

    # A 3x3 crater is 0.18% of a 70x70 grid, and layouts are sparse and spread
    # over the whole board, so it destroys about one machine however it is
    # aimed. Setting this > 0 sizes the crater as that FRACTION of the layout's
    # own bounding-box extent, and scales r1/r2 with it, so the disruption is
    # meaningful relative to the factory rather than to the grid.
    #
    # Measured (12 samples per configuration, crater centred on a real machine):
    #
    #   rate machines  crater   r1   r2  destroyed  movable  % hit
    #   0.25       30     3x3    5   12        1.0      8.0    27%   <- default
    #                     5x5   10   20        1.6     11.2    38%
    #                     7x7   13   26        1.8     14.1    47%
    #   0.40       40     3x3    5   12        1.0      6.8    17%   <- default
    #                     5x5   10   21        1.8     14.9    37%
    #                     7x7   14   28        2.2     22.2    55%
    #
    # 0.0 keeps the fixed ZPF_W/ZPF_H/ZPF_R1/ZPF_R2 above. Every number
    # downstream moves when this is enabled, so it is off by default: run it as
    # a deliberate sensitivity experiment ("does the recovery result hold as the
    # disruption grows?"), which is a far stronger claim than one crater size.
    ZPF_SCALE_TO_LAYOUT: float = 0.0    # e.g. 0.10 -> crater ~10% of layout extent

    # --- ZPF recovery cost terms ---------------------------------------------
    # NOTE: three simulated-annealing knobs (initial temperature, cooling rate,
    # destroy-operator weights) used to sit here. They are left over from the SA
    # sibling notebook and were never read by anything in THIS one -- its ZPF
    # recovery is NSGA-II, budgeted by ZPF_ITERATIONS / ZPF_POP_SIZE above.
    # --- ZPF recovery neighbourhood (R4) --------------------------------------
    # Both recoveries used to search a PERMUTATION of the movable machines. The
    # repair they hand it to puts every undamaged machine straight back at its
    # original coordinates, and those coordinates were already collision-free,
    # so they always succeed -- in any order. The permutation therefore only
    # mattered for machines that were actually destroyed, which for a 3x3 crater
    # is typically one. Measured: 20 evaluations produced 1-4 DISTINCT costs,
    # and the best was usually found on evaluation 1, i.e. by the deterministic
    # repair before the metaheuristic did anything. Both variants were wrappers
    # around the same heuristic and the comparison could not show a difference.
    #
    # The neighbourhood is now destroy-and-reinsert over COORDINATES, shared
    # identically by the NSGA-II and the SA recovery. See ZPFNeighbourhood.
    ZPF_POSITION_SEARCH: bool = True   # False = permutation-only (the old behaviour)
    ZPF_DESTROY_MAX:     int  = 3      # machines destroyed-and-reinserted per move
    ZPF_GREEDY_SAMPLES:  int  = 8      # candidate cells scored per greedy insertion
    # A destroyed depot loader/unloader is edge_only, and no grid edge exists
    # inside a crater in the middle of the factory -- so under the r2 rule it
    # could never be re-placed and was permanently lost. With this True it may
    # return to any edge instead.
    ZPF_EDGE_RELOCATION: bool = True

    PENALTY_PER_MOVED_MACHINE: float = 100.0
    INFEASIBLE_PENALTY: float = 1_000_000.0

    # --- feasibility fitness shaping -----------------------------------------
    # terminal_rate and regret are graded by these weighted gate ratios instead
    # of snapping to 0/1 the moment a single gate (placement, routing, power,
    # self-sustainability) is missed.
    W_PLACEMENT: float = 0.25
    W_ROUTING:   float = 0.25
    # W_POWER is 0.0 by default. With a single thermal bank the power ratio is
    # 1.00 or 0.00 -- the bank is fuelled or it is not -- so under an equal
    # -weighted mean one belt landing or missing was worth 25 feasibility
    # points, more than any other single fact about the layout. Whether that
    # belt lands is close to arbitrary: it competes for main_unit's six output
    # ports with everything else. It was the largest single source of noise in
    # the reported number and it moved independently of layout quality.
    #
    # The gate is NOT gone: `power_ok` is still computed, still printed in the
    # per-gate breakdown, and still part of the hard `feasible` boolean. It is
    # simply no longer allowed to dominate a graded score it cannot vary
    # smoothly. Set W_POWER back to 0.25 to restore the old behaviour; the
    # weights are renormalised by their sum, so the remaining three share the
    # score equally.
    W_POWER:     float = 0.0
    W_SUSTAIN:   float = 0.25

    # How the four gate ratios combine into the headline feasibility number.
    #   "mean"      -- weighted arithmetic mean (the original rule)
    #   "geometric" -- weighted geometric mean: a weak gate hurts
    #                  multiplicatively instead of costing a fixed slice, so a
    #                  layout cannot buy back a failed gate with three good ones
    #   "min"       -- worst gate binds. Honest, but it flattens the search:
    #                  every layout with the same worst gate ties.
    FEAS_AGGREGATION: str = "mean"

    # --- revenue: what an unsupplied input is worth ---------------------------
    # _calculate_revenue multiplies a machine's efficiency by this for every
    # required input no routed flow delivers, and the result propagates
    # downstream. It was a bare 0.01 literal in the middle of the function and
    # it dominates the objective: measured on a 24-machine layout, breaking the
    # worst single flow cost 43% of revenue while route_ratio moved 4 points.
    # terminal_rate = revenue x feasibility, so this constant -- not the gates --
    # sets how brutally the search is punished for an unrouted upstream flow.
    #
    # It is a knob, not a derived quantity, and it is named here so the write-up
    # can say what it is and what it does. The principled replacement is to make
    # the penalty proportional to the fraction of required input actually
    # delivered, which needs the rate information the _pool fork's
    # derive_flow_matrix now carries (see P1/P3 in the problem register).
    STARVED_INPUT_EFF: float = 0.01

    # --- revenue: how output is credited (P2) ---------------------------------
    # _calculate_revenue used to credit every machine's output at production,
    # regardless of destination, and REVENUE_TABLE prices intermediates
    # (steel_part 3, ame_bottle 2, ferri_part 1, ame_part 1, ori_crust 1). So a
    # part earned revenue AND the product built from it earned revenue again.
    # Measured: an HC_bat factory drew 53% of its gross from steel_part it
    # consumes itself; an SC_bat factory 25% from ferri_part. A battery burned
    # by a thermal bank likewise earned full revenue AND supplied power. That
    # inverts the incentive the model exists to express -- processing further is
    # meant to be MORE profitable. Ranked by terminal-rate-per-machine, the
    # deepest chain came LAST and the three batteries took the top three places.
    #
    #   "net"   (default) credit produced minus consumed, per material. A
    #           material the factory consumes itself nets against its own
    #           production, so it is paid for exactly once, at the point it
    #           stops being an input to something. Structural: it does not
    #           depend on which belts happened to route.
    #
    #   "sink"  credit only output a routed flow delivers to an io_sink. The
    #           most literal reading of "what is actually sold", and it is what
    #           was implemented first -- but it makes revenue hinge on one belt.
    #           Measured on the SA quick profile: layout #2 scored 2.6273
    #           pre-bomb and 0.0251 after nothing but re-routing the SAME
    #           undamaged layout on a fresh grid. Under this fork's 1:1 greedy
    #           matching a product reaches a sink through a single Stage-3 flow
    #           with a round-robin sink assignment, so the objective goes
    #           near-binary on one A* result. Worth revisiting on the _pool
    #           fork, where Stage 3 deposits every unallocated supply by rate.
    #
    #   "gross" the original behaviour: credit everything at production,
    #           double-count included. Kept for comparison.
    REVENUE_MODE: str = "net"

    # --- routing: honour the docstring, within a stated budget -----------------
    # route_belts' fallback broke out of the OUTER loop as soon as any out-port
    # found a path, so it returned that port's best path rather than the
    # shortest over the (out, in) pairs, which is what it claims to do. It now
    # keeps searching across out-ports -- but with a cap, because the fallback
    # is reached on exactly the flows a congested layout has most of, and the
    # full product is |outs| x |ins| A* searches per flow: 6 x 14 = 84 for a 9x9
    # main_unit, on a 70x70 grid, inside every ALNS iteration and every recovery
    # evaluation. Uncapped it took the quick profile from ~300s to ~4600s on the
    # pool notebooks.
    #
    # So the docstring's promise is now "shortest over the pairs tried", and
    # this is how many get tried. 0 restores the old first-out-port behaviour.
    ROUTE_FALLBACK_BEST: bool = True
    ROUTE_FALLBACK_MAX_PAIRS: int = 12

    # --- rotation search (P7) -------------------------------------------------
    # Every chromosome is built Gene(m.id, Rotation.R0, r) -- in evaluate_nsga2,
    # in Phase 2 and in the recovery -- so rotation, which decides which face
    # carries the I/O ports, was fixed at zero everywhere and an entire
    # dimension of the layout problem was never searched.
    # greedy_rotation_search() sits in the legacy appendix and is called by
    # nothing; at 4 x n_genes x passes full evaluations it is unaffordable at
    # MAX_MACHINES=160.
    #
    # Two contained fixes instead:
    #   * the recovery neighbourhood gets a `rotate` operator, so rotation IS
    #     searched where this thesis actually makes its claim (see
    #     ZPFNeighbourhood);
    #   * Phase 2 runs a BUDGETED greedy rotation pass over the machines that
    #     carry the most handling cost. 0 disables it.
    # Cost: TOP x 3 rotations x PASSES full evaluate_layout calls per Phase 2
    # candidate. On a 100+ machine BOM one evaluation is seconds, so keep TOP
    # small; MAX_BOM skips the pass entirely on layouts where it would dominate
    # the phase.
    ROTATION_POLISH_TOP:    int = 4    # machines considered per pass; 0 = off
    ROTATION_POLISH_PASSES: int = 1
    ROTATION_POLISH_MAX_BOM: int = 80  # skip entirely above this BOM size
    # MHC is one of the three NSGA-II objectives and Phase 2's Tabu pass exists
    # specifically to cut it, so rotation must not be allowed to spend it
    # freely. Phase 2 ranks candidates on feasibility-dominated weighted score
    # with MHC nowhere in it, so without a bound a rotation was accepted that
    # bought 2.4 points of feasibility for +386 MHC (625 -> 1011, +62%),
    # undoing Tabu. A rotation may now cost at most this fraction more MHC than
    # the incumbent.
    ROTATION_POLISH_MHC_TOLERANCE: float = 0.05

    # --- sustain gate: normalise out the part the layout cannot control -------
    # sustain used to be SIM_EQ_MAX_TICKS / critical_path. The critical path
    # grows with the number of machines on it, so KEEPING machines scored worse
    # and destroying them scored better -- a bombed layout could improve on this
    # gate by more than it lost on routing, which is exactly how the _pool fork
    # produced regret=0.0000 with the bomb "improving" the factory in 4 of 4
    # layouts.
    #
    # With SUSTAIN_NORMALISE the budget scales with the layout's own irreducible
    # processing time (the same critical path with every belt treated as
    # instantaneous), so the gate measures what the LAYOUT adds -- belt travel
    # and detours -- rather than how deep the BOM happens to be. Removing a
    # machine now shortens the numerator and the denominator together.
    SUSTAIN_NORMALISE: bool = True
    SUSTAIN_SLACK: float = 2.0     # allowed multiple of the irreducible time

    # --- GA genes / mutation --------------------------------------------------
    # Calibrated against derive_rate_balanced_bom() directly: machine count vs
    # rate is a step function dominated by ceil() rounding, and it was FLAT
    # across the entire old 0.005-0.02 range for every product (e.g. buck_cap_A
    # -> 65 machines at rate=0.005 AND at rate=0.1 - identical BOM either way).
    # That, combined with init_individual() only ever activating one product,
    # is why layouts stayed small: the gene ranges couldn't reach the part of
    # the search space where rate/product-count actually change the BOM.
    # 0.15-0.45 sits past that dead zone for every product's response curve.
    MIN_TERMINAL_RATE: float = 0.15
    MAX_TERMINAL_RATE: float = 0.45
    PRODUCT_ACTIVATION_PROB: float = 0.4  # per-product odds of being active at init
    MULT_INIT_MIN: float = 0.1        # initial range for recipe-multiplier genes
    MULT_INIT_MAX: float = 1.0
    MUT_RATE_SIGMA: float = 0.08       # scaled up to match the new, wider rate range
    MUT_GENE_SIGMA: float = 0.15

    # --- chain terminators: where material enters and leaves the factory -------
    # Raw material enters at an io_source (main_unit / depot_unloader); finished
    # goods leave at an io_sink (main_unit / depot_loader / stash).
    # derive_flow_matrix has always recognised all of these -- but the BOM built
    # exactly ONE main_unit and nothing else, so EVERY boundary flow in the
    # factory funnelled through its 6 output and 14 input cells.
    #
    # Measured on a 34-machine BOM (rate 0.40), routed flows:
    #    depots stashes | overall | imports | exports | chain
    #         0       0 |  19/53  |    0/21 |    4/16 | 15/16
    #         6       0 |  26/53  |    9/21 |    3/16 | 14/16
    #         0       6 |  30/53  |    3/21 |   11/16 | 16/16
    #         6       6 |  40/53  |   11/21 |   13/16 | 16/16
    # The production chain itself was already routing at ~94% -- essentially
    # every failure was at the boundary, not in the chain.
    #
    # NOTE: stash costs 10 power each, so adding many can flip the power gate
    # if the thermal bank's own fuel belt loses the competition for ports.
    #
    # A value >= 1 is an absolute count; a value in (0,1) is a fraction of the
    # BOM size (0.15 -> 6 terminators on a 40-machine BOM). 0 disables.
    #
    # ON BY DEFAULT. On a real quick run (seed 60) these took feasibility from
    # 46% to 67%, and the biggest single effect was NOT routing -- the
    # `sustain` gate went 0.14 -> 1.00, because giving the chain real endpoints
    # collapses the critical path that _chain_time measures. Set both to 0.0 to
    # reproduce the old single-main_unit behaviour.
    IMPORT_DEPOTS:  float = 0.15  # depot_unloader, edge-only raw-material sources
    EXPORT_STASHES: float = 0.15  # stash, 3x3 placeable anywhere, product sinks
    EXPORT_LOADERS: float = 0.0   # depot_loader, edge-only product sinks


    # --- logistics machine build costs ----------------------------------------
    # stash / depot_loader / depot_unloader all ship with build_cost=[] in the
    # catalogue, i.e. they are FREE. The design relies on them costing money to
    # discourage sprinkling depots everywhere; without a price the layout
    # objective weakens badly once the pool exists (any machine can just sit
    # next to its own depot).
    #
    # depot_loader/unloader are edge_only, which limits them geometrically.
    # `stash` is interior-placeable, so it is the one that most needs a price.
    #
    # PROVENANCE. database.xlsx prices machines in its `machine_mat` column,
    # and that column covers PRODUCTION machines only -- there is no build cost
    # in the source for stash, depot_loader or depot_unloader. The sheet's
    # `rules` tab prices their POWER ("all utilities use 10 power each except
    # batteries, belts, depot unloader and loader") and says nothing about
    # construction. So these figures cannot be "corrected" from the database;
    # there is nothing there to copy.
    #
    # They are therefore an ANALOGY, and should be described as one in the
    # write-up: a stash is 3x3, the same footprint class as the Refining /
    # Shredding / Fitting / Moulding units, which the database prices at 5-10
    # units of an early-chain material (ori ore / ori crust). The depots get
    # half a stash on the grounds that they are edge-only and hold nothing.
    #
    # Set LOGISTICS_BUILD_COST = () to make them free again, which is what the
    # source model literally specifies.
    LOGISTICS_BUILD_COST: tuple = (
        ("stash",          (("ori_crust", 10), ("ferri_part", 2))),
        ("depot_loader",   (("ori_crust",  5), ("ferri_part", 2))),
        ("depot_unloader", (("ori_crust",  5), ("ferri_part", 2))),
    )

    # --- self-fuelling power ---------------------------------------------------
    # A thermal bank burns 1 battery per process_time ticks, but the BOM appends
    # banks AFTER the expansion loop, so that demand never re-enters the queue
    # and nothing ever makes a battery. With this True the fuel demand is fed
    # back through the expansion until the machine count settles, so the factory
    # manufactures the batteries it burns. Measured growth: +50% to +130% BOM.
    SELF_FUEL_POWER:   bool = True
    BOM_POWER_PASSES:  int  = 6      # fixed-point iterations for the fuel loop

    # --- power / battery --------------------------------------------------------
    # --- power: which battery the thermal banks burn ------------------------
    # ther_bank has one recipe per battery type. The BOM used to hardcode the
    # SC_bat recipe, so four of five were unreachable and the factory was
    # forced onto the most chain-expensive fuel in the catalogue.
    #
    #   "auto"  -- highest power per machine of supporting production chain
    #   "<material>" -- pin a specific fuel, e.g. "HC_bat" (reproduces the old
    #              behaviour exactly with "SC_bat")
    POWER_FUEL: str = "auto"
    # Whether a bank may burn ORE directly (ther_bank[0]: ori_ore -> 50 power).
    # This is a real option in the source model. It opens no free-power hole:
    # ore accrues at a fixed rate and 50 power does not out-score HC_bat's 1100
    # even against an 8-machine chain.
    #
    # This flag used to also admit LC_W_bat and SC_W_bat, which have no producer
    # and are not ores -- i.e. are unobtainable. They are excluded structurally
    # now (see UNOBTAINABLE), not by this switch.
    POWER_FUEL_ALLOW_RAW: bool = True

    BATTERY_CAPACITY: int = 1100
    THER_BANK_RECIPE: int = 2      # which ther_bank recipe the BOM builds   # power supplied per Thermal Bank added by derive_bom

    # --- simulation --------------------------------------------------------------
    SIM_EQ_MAX_TICKS: int = 300      # self-sustainability tick budget (Gate 2)
    # Phase 5 discrete-time simulation. The GA's Gate 2 stays the cheap static
    # _chain_time() DP -- running a full SimEngine inside the fitness function
    # would multiply an already-expensive evaluation. The simulation instead
    # VALIDATES the top layouts after the fact and produces the production
    # timeline (Gantt) and throughput curves, and Phase 5 reports where the
    # static gate and the simulation disagree.
    SIM_TOP_N:        int   = 3      # layouts simulated pre-bomb and post-recovery
    SIM_MAX_TICKS:    int   = 400
    SIM_EQ_WINDOW:    int   = 30
    SIM_EQ_THRESHOLD: float = 0.08

    # Equilibrium must mean "running steadily", not merely "unchanging".
    # is_equilibrium only ever inspected production_log, and a machine that
    # never completes a recipe contributes no entries -- so it was invisible,
    # and a factory with most of its machines permanently starved counted as
    # converged. It was stably broken. This is the minimum fraction of
    # recipe-bearing machines that must have produced something inside the
    # window before equilibrium is granted.
    SIM_MIN_LIVE_FRACTION: float = 0.5

    # --- when does the simulation CONTRADICT the analytical model? ------------
    # Phase 5 used to compare only equilibrium timing, so a layout with an
    # analytic rate of 7.0000 and 0.000 simulated revenue/tick was reported as
    # agreeing. Throughput is now compared too: the simulation must reach this
    # fraction of the analytic rate before the two are called consistent.
    # Below it, Phase 5 names a cause instead of staying silent.
    SIM_AGREE_FRACTION: float = 0.5
    # Share of the POST-GRACE ticks spent browned out above which power, not
    # starvation, is named as the cause. Measured against post-grace ticks
    # because power is free during the grace window by construction -- a layout
    # that browns out on every tick it is actually charged for scores 1.0 here,
    # not 0.6.
    SIM_BROWNOUT_FRACTION: float = 0.25

    # --- NSGA-II selection pressure (S2) --------------------------------------
    # Two compounding faults froze the search at generation 4:
    #
    # 1. NO MATING SELECTION. varAnd() was handed the population directly, so
    #    every individual reproduced regardless of quality and the only
    #    selection pressure in the entire loop was the environmental step.
    #    Textbook NSGA-II picks parents by binary tournament on (rank, crowding)
    #    first -- selTournamentDCD.
    #
    # 2. A SATURATED FIRST FRONT. derive_rate_balanced_bom is a step function of
    #    the genome (ceil() rounding), so large regions of genotype space map to
    #    the SAME objective vector. A real run held 200 individuals carrying
    #    about 9 distinct (cost, rate, MHC) triples. Individuals with identical
    #    objectives cannot dominate each other, so they all land on front 1;
    #    once the whole population is rank 1, selNSGA2 degenerates into pure
    #    crowding distance, which rewards spread rather than quality. It keeps
    #    the extremes -- including the cheap-and-useless corner -- which is
    #    exactly why the best rate froze while the average collapsed.
    #
    # GA_MAX_CLONES caps how many individuals sharing one objective vector may
    # survive a generation. GA_FRESH_BLOOD is the per-generation budget for new
    # random individuals used to fill the slots that frees up; the remainder is
    # refilled from the ranked overflow, so population size and character are
    # preserved. Deduplicating outright and refilling at random would be a
    # restart every generation, not a fix.
    GA_MATING_SELECTION: bool  = True
    GA_MAX_CLONES:       int   = 2
    GA_FRESH_BLOOD:      float = 0.10   # fraction of POP_SIZE per generation
    GA_STAGNATION_GENS:  int   = 5
    # NSGA-II is Pareto-elitist, which guarantees nothing about any ONE
    # objective. When the first front is larger than the population, survival is
    # decided by crowding distance, and a high-rate individual in a crowded
    # region loses to a worse one at the edge of the front. Measured on this
    # notebook: the best rate went 3.9768 -> 2.8571 and stayed there, i.e. the
    # best layout found was thrown away. Phase 2 and Phase 4 both select by
    # rate, so that is the pipeline losing its subject matter.
    #
    # With this on, the best-by-rate individual ever seen is retained and
    # re-injected if it drops out, replacing the worst by rate. The reported
    # best rate is then monotone by construction. Set False for textbook
    # NSGA-II behaviour.
    GA_RATE_ELITISM:     bool  = True

    # --- power economy (simulation) --------------------------------------------
    SIM_ENFORCE_POWER: bool = True   # machines cannot START a recipe unpowered
    # Must cover bootstrap, or the factory browns out before it can generate:
    # first battery arrives ~1/rate ticks, then the burn itself takes
    # process_time (100) ticks, so the first power lands around tick 120.
    POWER_GRACE_TICKS: int  = 150    # free power for this many ticks (see STARTUP_FREE)

    # Thermal banks are generators RATED at their recipe's power output while
    # burning (CFG.BATTERY_CAPACITY is "power supplied per Thermal Bank"), and
    # power_cost is a rated draw -- so `output > demand` compares like with
    # like. Setting this True instead treats 1100 as an energy batch and
    # divides by process_time, which multiplies the bank count ~17x; those
    # extra banks cannot be fuelled through main_unit's six output ports, so
    # routing gets substantially WORSE. Kept only for comparison.
    POWER_RATE_CORRECT: bool = False

    # --- base-layout macro-repair (ALNS + Tabu) ---------------------------------
    # evaluate_layout() used to run a single greedy BLF decode with no
    # repair, which silently dropped any machine that didn't fit on the
    # first pass. This closes that gap the same way the v3-architecture
    # notebook does: destroy-and-reinsert a subset of machines to reduce the
    # unplaced count, then a short Tabu polish pass to cut MHC once
    # placement/routing are complete.
    # NOTE: raising these further has sharply diminishing returns above
    # ~10 iterations — tested directly on a 158-machine BOM, routing only
    # moved from 63/115 to 67/115 going from 5 to 40 iterations. Missing
    # routes at high machine counts are predominantly a grid-congestion
    # problem (no physical space left for A* to route through), not a
    # search-depth problem, so more iterations mostly just cost more time.
    ALNS_MAX_ITERS: int = 30
    ALNS_DESTROY_PROB: float = 0.4
    TABU_MAX_ITERS: int = 15
    TABU_LIST_SIZE: int = 10

    # --- rip-up and reroute (Phase 3) ------------------------------------------
    # Belts are placed greedily and never removed, so the first flow to claim a
    # corridor keeps it and a later flow with no alternative just fails. This
    # pass lets a blocked flow evict belts crossing its path, route into the
    # freed space, then re-lay the evicted ones -- keeping the move only if the
    # total routed count strictly increases, so it can never make things worse.
    #
    # Measured (SA notebook, terminators at 0.15): 21/27 -> 23/27, 30/40 ->
    # 32/40, 38/53 -> 41/53. In every case it drove the number of unrouted
    # MACHINE-TO-MACHINE flows to ZERO -- everything still unrouted afterwards
    # is a factory-boundary flow. ON BY DEFAULT; set to 0 to disable.
    RIPUP_ROUNDS:    int = 2   # 0 disables
    RIPUP_MAX_EVICT: int = 5   # belts a single blocked flow may displace

    # --- final connect-or-remove cleanup -----------------------------------------
    # After ALNS+Tabu, any machine that still isn't connected to anything gets
    # one more targeted shot at connecting (trying every port combination, not
    # just the one derive_flow_matrix assigned) before being removed outright.
    # This trades "matches the original BOM size" for "everything left is
    # actually working" -- set PRUNE_UNCONNECTABLE=False to keep the strict,
    # original behaviour (dangling unrouted flows just stay unrouted).
    PRUNE_UNCONNECTABLE: bool = True
    RECONNECT_MAX_ATTEMPTS: int = 6   # port combinations tried per stuck flow
    RECONNECT_MAX_PASSES: int = 3     # remove-and-retry cycles (removing one
                                       # machine can free space or orphan
                                       # another, so this repeats a few times)

    # --- feasibility gate strictness -----------------------------------------
    # The v3-architecture notebook's actual evaluate_fitness (verified
    # directly in its source) never required 100% routing completeness for
    # `feasible` either -- only all_placed, power_ok, and sustain_ok. An
    # unrouted flow costs an MHC penalty instead of blocking feasibility
    # outright. That's the main reason it reports "feasible" layouts readily
    # even on complex/dense ones, and evaluate_layout now matches it by
    # default. Set this True to restore the previous, stricter requirement
    # that every single flow must route.
    REQUIRE_FULL_ROUTING: bool = True   # was False -- "feasible"/[FEASIBLE] was reachable
    # with routing well under 100% (main_feas could be e.g. 88-96% and still print
    # [FEASIBLE], since the boolean never actually checked paths_found==flows_total
    # unless this flag was on). Matches the same fix applied to the v5 lineage.

    # --- how many layouts each downstream phase processes ----------------------
    PHASE2_CANDIDATES: int = 20   # top-N from the ranked population into Phase 2/3
    # Phase 2 used to take the top-ranked individuals regardless of BOM size.
    # evaluate_nsga2 does not evaluate an over-cap individual at all: it returns
    # a graded penalty plus `raw_terminal_rate * 0.1` in place of a measured
    # rate, and that discounted raw figure is still large for an ambitious BOM,
    # so over-cap individuals outranked properly evaluated ones on objective 1 --
    # the very key Phase 2 sorts by. Measured on the quick profile at
    # MAX_MACHINES=40: Phase 2 ran the heavy repair on candidates of 148, 105
    # and 94 genes and handed them to Phase 4.
    #
    # With this on, such individuals are skipped and the selector walks further
    # down the ranked list. If NOTHING is under the cap it falls back to the
    # smallest BOMs and says so -- an empty Phase 2 would break every phase
    # after it, and a whole population over the cap means MAX_MACHINES or the
    # terminal-rate band is mis-set rather than that those layouts are fine.
    PHASE2_ENFORCE_CAP: bool = True
    # --- Phase 4 eligibility ------------------------------------------------
    # W_POWER = 0.0 keeps power out of the GRADED score, where a binary gate is
    # a bad signal. It should not also keep power out of ELIGIBILITY, where
    # binary is exactly right: a factory that browns out cannot be a resilience
    # result, however well the recovery re-routed its belts.
    #
    # Measured, GA full results 7 Layout #1: regret 0.0000 and recovered
    # fraction 100.0% on a layout the simulation browned out for 250 of 250
    # post-grace ticks.
    PHASE4_REQUIRE_POWER: bool = True

    PHASE4_TOP_N:      int = 5    # layouts bombed in Phase 4 and plotted in Phase 6

    # --- logging ---------------------------------------------------------------
    SHOW_BLF_WARNINGS: bool = False  # print every BLF "could not place" line;
                                      # off by default (very noisy over a GA
                                      # run). Failures are still tallied in
                                      # BLF_UNPLACED_COUNT regardless.

# ─── RIGOUR PROFILE ───────────────────────────────────────────────────────────
#   "full"  — thesis-grade run. Hours of CPU. Use this for reported results.
#   "quick" — smoke test, roughly 5 minutes. Verifies that the pipeline EXECUTES
#             end to end and that every phase emits its output.
#
# A "quick" run tells you the plumbing is correct. It does NOT tell you anything
# about solution quality: POP_SIZE/NGEN are far below what NSGA-II needs to
# converge, and the terminal-rate band is deliberately narrowed so the BOM stays
# tiny (BOM size is the dominant runtime driver -- a 10-machine layout evaluates
# in ~3s, an 18-machine one in ~17s). Never quote numbers from a quick run.
RIGOUR = "full"

# Sized so a whole run lands around 5-10 minutes on one core. The dominant
# cost is evaluate_layout: ~3s per 10-machine BOM at these rates, and Phase 2's
# repair=True path costs roughly 60s per candidate. Budget accordingly if you
# raise anything here.
QUICK_PROFILE = dict(
    # --- search effort ---
    POP_SIZE=10, NGEN=2, MAX_MACHINES=40,
    ALNS_MAX_ITERS=2, TABU_MAX_ITERS=2, RECONNECT_MAX_PASSES=1,
    # --- how many layouts each downstream phase touches ---
    PHASE2_CANDIDATES=3, PHASE4_TOP_N=2, SIM_TOP_N=2, SIM_MAX_TICKS=150,
    # --- keeps the BOM small; this is what actually buys the speed ---
    MIN_TERMINAL_RATE=0.02, MAX_TERMINAL_RATE=0.06,
    # --- single process: tiny populations gain nothing from workers, and
    #     joblib swallows tracebacks, which is the opposite of what a smoke
    #     test needs. Also makes the run bit-reproducible from RANDOM_SEED.
    MULTIPROCESSING=False,
)
# 3 craters, not 10: a smoke test only needs to prove the scenario loop
# runs and aggregates. Quality claims never come from this profile.
_QUICK_ZPF = dict(ZPF_ITERATIONS=18, ZPF_POP_SIZE=6, ZPF_SCENARIOS=3)


CFG = FLPConfig()
if RIGOUR == "quick":
    for _k, _v in QUICK_PROFILE.items():
        setattr(CFG, _k, _v)
    for _k, _v in _QUICK_ZPF.items():
        setattr(CFG, _k, _v)
    print("=" * 74)
    print("[RIGOUR] QUICK smoke-test profile ACTIVE — results are NOT meaningful.")
    print("         Set RIGOUR = \"full\" above for a thesis-grade run.")
    print("=" * 74)
elif RIGOUR == "full":
    print("[RIGOUR] FULL profile — expect a long run.")
else:
    raise ValueError(f"RIGOUR must be 'quick' or 'full', got {RIGOUR!r}")


# Seed every RNG the notebook touches, once, for reproducible end-to-end runs.
random.seed(CFG.RANDOM_SEED)
np.random.seed(CFG.RANDOM_SEED)

# Single source of truth: override the Section-1 defaults with the config knobs.
SIM_EQ_MAX_TICKS = CFG.SIM_EQ_MAX_TICKS
BLF_VERBOSE = CFG.SHOW_BLF_WARNINGS

# Back-compat aliases — later cells refer to these bare names directly.
POP_SIZE     = CFG.POP_SIZE
NGEN         = CFG.NGEN
CXPB         = CFG.CXPB
MUTPB        = CFG.MUTPB
MAX_MACHINES = CFG.MAX_MACHINES

# ── SUMMARY ──────────────────────────────────────────────────────────────────
print(f"Grid          : {GRID_W}x{GRID_H}   spacing={MIN_MACHINE_SPACING}")
print(f"Belt speed    : {BELT_SPEED} tick/cell   tick={TICK_SECONDS}s")
print(f"Search        : POP={CFG.POP_SIZE} NGEN={CFG.NGEN} MAX_MACHINES={CFG.MAX_MACHINES} "
      f"rate band={CFG.MIN_TERMINAL_RATE}-{CFG.MAX_TERMINAL_RATE} "
      f"activation={CFG.PRODUCT_ACTIVATION_PROB}")
print(f"Repair        : ALNS={CFG.ALNS_MAX_ITERS} TABU={CFG.TABU_MAX_ITERS} "
      f"prune={CFG.PRUNE_UNCONNECTABLE} full_routing={CFG.REQUIRE_FULL_ROUTING}")
print(f"Phase widths  : phase2={CFG.PHASE2_CANDIDATES} phase4={CFG.PHASE4_TOP_N} "
      f"sim={CFG.SIM_TOP_N}")
print(f"ZPF           : crater {CFG.ZPF_W}x{CFG.ZPF_H} @({CFG.ZPF_X},{CFG.ZPF_Y}) "
      f"r1={CFG.ZPF_R1} r2={CFG.ZPF_R2} budget={CFG.ZPF_ITERATIONS}")
print(f"Power         : enforce={CFG.SIM_ENFORCE_POWER} grace={CFG.POWER_GRACE_TICKS} "
      f"rate_correct={CFG.POWER_RATE_CORRECT} import_depots={CFG.IMPORT_DEPOTS}")

## 2. Enums

In [ ]:
class Rotation(Enum):
    R0   = 0    # original orientation
    R90  = 90   # 90 deg clockwise
    R180 = 180  # 180 deg
    R270 = 270  # 270 deg clockwise


class TileState(Enum):
    EMPTY   = auto()
    MACHINE = auto()
    BELT    = auto()
    BLOCKED = auto()  # disruption zone (Phase 2)

print("Enums defined:", list(Rotation), list(TileState))


## 3. Data Classes

### 3.1 Recipe
One input → output transformation a machine can perform.

### 3.2 MachineType
Blueprint for a machine type (not a placed instance).  
Port layout convention (before rotation, R0):  
- `input_face` = which face has the input nodes  
- `output_face` = which face has the output nodes (always opposite)  
- The face with I/O is always the **wider** side.  
- Each node is 1 cell wide.

### 3.3 PlacedMachine
A machine placed on the grid.  
**I/O model:**
- I/O nodes are **inside** the machine, on the final tile row/col of the rectangle
- Belt connect points are **outside** the machine, adjacent to the I/O nodes
- At least 1 belt tile is always required to connect machines

### 3.4 FlowPair
Directed material flow between two machine instances.  
`src_id`/`dst_id` are instance IDs (e.g. `refining_unit_0`), not type IDs.


In [ ]:
@dataclass
class Recipe:
    """One input -> output transformation a machine can perform."""
    inputs:       list[tuple[str, int]]  # [(material_name, qty), ...]
    outputs:      list[tuple[str, int]]  # [(material_name, qty), ...]
    process_time: int                    # ticks
    common_extra: list[tuple[str, int]] = field(default_factory=list)


@dataclass
class MachineType:
    """Blueprint for a machine type (not a placed instance)."""
    id:            str
    display_name:  str
    base_w:        int
    base_h:        int
    recipes:       list[Recipe]
    build_cost:    list[tuple[str, int]]
    power_cost:    int
    input_face_r0: str = "bottom"
    edge_only:     bool = False
    singleton:     bool = False
    is_power_src:  bool = False

    def size(self, rot: Rotation = Rotation.R0) -> tuple[int, int]:
        """Return (w, h) after rotation."""
        if rot in (Rotation.R0, Rotation.R180):
            return (self.base_w, self.base_h)
        return (self.base_h, self.base_w)

    def _rotate_face(self, face: str, rot: Rotation) -> str:
        """Rotate a face name by the given rotation (clockwise)."""
        order = ["top", "right", "bottom", "left"]
        steps = {Rotation.R0: 0, Rotation.R90: 1,
                 Rotation.R180: 2, Rotation.R270: 3}
        return order[(order.index(face) + steps[rot]) % 4]

    def io_faces(self, rot: Rotation = Rotation.R0) -> tuple[str, str]:
        """Return (input_face, output_face) after rotation."""
        if self.id == "main_unit":
            # main_unit has multi-face I/O; return comma-joined full face names
            in_faces = [self._rotate_face("left", rot), self._rotate_face("right", rot)]
            out_faces = [self._rotate_face("top", rot), self._rotate_face("bottom", rot)]
            return (",".join(in_faces), ",".join(out_faces))
        opposite = {"top": "bottom", "bottom": "top",
                    "left": "right",  "right": "left"}
        base_out = opposite[self.input_face_r0]
        in_face  = self._rotate_face(self.input_face_r0, rot)
        out_face = self._rotate_face(base_out, rot)
        return (in_face, out_face)

    def wider_face_len(self, rot: Rotation = Rotation.R0) -> int:
        """Number of cells on the wider (I/O) face after rotation."""
        w, h = self.size(rot)
        in_face, _ = self.io_faces(rot)
        if in_face in ('top', 'bottom'):
            return w
        return h



@dataclass
class PlacedMachine:
    """A machine placed on the grid."""
    machine_type:  MachineType
    x:             int
    y:             int
    rotation:      Rotation
    instance_id:   str = ""
    active_recipe: Optional[int] = None

    @property
    def w(self) -> int:
        return self.machine_type.size(self.rotation)[0]

    @property
    def h(self) -> int:
        return self.machine_type.size(self.rotation)[1]

    def cells(self) -> list[tuple[int, int]]:
        return [(self.x + dx, self.y + dy)
                for dy in range(self.h)
                for dx in range(self.w)]

    def _get_io_cells(self, face: str) -> list[tuple[int, int]]:
        """I/O node cells INSIDE the machine on the given face."""
        w, h = self.w, self.h
        if face == "top":    return [(self.x + dx, self.y)           for dx in range(w)]
        if face == "bottom": return [(self.x + dx, self.y + h - 1)  for dx in range(w)]
        if face == "left":   return [(self.x,       self.y + dy)     for dy in range(h)]
        if face == "right":  return [(self.x + w - 1, self.y + dy)  for dy in range(h)]
        raise ValueError(f"Unknown face: {face}")

    def _get_main_unit_io_nodes(self, face: str) -> list[tuple[int, int]]:
        w, h = self.w, self.h
        if face == "top":    return [(self.x + dx, self.y)           for dx in (1, 4, 7)]
        if face == "bottom": return [(self.x + dx, self.y + h - 1)  for dx in (1, 4, 7)]
        if face == "left":   return [(self.x,       self.y + dy)     for dy in range(1, 8)]
        if face == "right":  return [(self.x + w - 1, self.y + dy)  for dy in range(1, 8)]
        return []

    def input_nodes(self)  -> list[tuple[int, int]]:
        if self.machine_type.id == "main_unit":
            faces = [self.machine_type._rotate_face("left", self.rotation),
                     self.machine_type._rotate_face("right", self.rotation)]
            return [n for f in faces for n in self._get_main_unit_io_nodes(f)]
        in_face, _ = self.machine_type.io_faces(self.rotation)
        return self._get_io_cells(in_face)

    def output_nodes(self) -> list[tuple[int, int]]:
        if self.machine_type.id == "main_unit":
            faces = [self.machine_type._rotate_face("top", self.rotation),
                     self.machine_type._rotate_face("bottom", self.rotation)]
            return [n for f in faces for n in self._get_main_unit_io_nodes(f)]
        _, out_face = self.machine_type.io_faces(self.rotation)
        return self._get_io_cells(out_face)

    def input_node_centre(self)  -> tuple[int, int]:
        nodes = self.input_nodes();  return nodes[len(nodes) // 2]

    def output_node_centre(self) -> tuple[int, int]:
        nodes = self.output_nodes(); return nodes[len(nodes) // 2]

    def _get_belt_connect_cells(self, face: str) -> list[tuple[int, int]]:
        """Cells OUTSIDE the machine, adjacent to the given face."""
        w, h = self.w, self.h
        if face == "top":    return [(self.x + dx, self.y - 1)  for dx in range(w)]
        if face == "bottom": return [(self.x + dx, self.y + h)  for dx in range(w)]
        if face == "left":   return [(self.x - 1,  self.y + dy) for dy in range(h)]
        if face == "right":  return [(self.x + w,  self.y + dy) for dy in range(h)]
        raise ValueError(f"Unknown face: {face}")

    def _get_main_unit_points(self, face: str) -> list[tuple[int, int]]:
        w, h = self.w, self.h
        if face == "top":    return [(self.x + dx, self.y - 1)  for dx in (1, 4, 7)]
        if face == "bottom": return [(self.x + dx, self.y + h)  for dx in (1, 4, 7)]
        if face == "left":   return [(self.x - 1,  self.y + dy) for dy in range(1, 8)]
        if face == "right":  return [(self.x + w,  self.y + dy) for dy in range(1, 8)]
        return []

    def input_belt_points(self)  -> list[tuple[int, int]]:
        if self.machine_type.id == "main_unit":
            faces = [self.machine_type._rotate_face("left", self.rotation),
                     self.machine_type._rotate_face("right", self.rotation)]
            return [n for f in faces for n in self._get_main_unit_points(f)]
        in_face, _ = self.machine_type.io_faces(self.rotation)
        return self._get_belt_connect_cells(in_face)

    def output_belt_points(self) -> list[tuple[int, int]]:
        if self.machine_type.id == "main_unit":
            faces = [self.machine_type._rotate_face("top", self.rotation),
                     self.machine_type._rotate_face("bottom", self.rotation)]
            return [n for f in faces for n in self._get_main_unit_points(f)]
        _, out_face = self.machine_type.io_faces(self.rotation)
        return self._get_belt_connect_cells(out_face)


    def input_belt_centre(self)  -> tuple[int, int]:
        pts = self.input_belt_points();  return pts[len(pts) // 2]

    def output_belt_centre(self) -> tuple[int, int]:
        pts = self.output_belt_points(); return pts[len(pts) // 2]


@dataclass
class FlowPair:
    """Directed material flow between two machine instances."""
    src_id:       str
    dst_id:       str
    material:     str
    intensity:    float
    src_node_idx: int = 0   # which output node on the source machine
    dst_node_idx: int = 0   # which input node slot on the destination machine

print("Data classes defined: Recipe, MachineType, PlacedMachine, FlowPair")




## 4. Machine Catalogue

Built from `database.xlsx` (yellow rows excluded).  
Port schematics:
- **3×3** (R0): 3 wide × 3 tall, row 0 = output (top), row 2 = input (bottom)
- **6×4** (R0): 6 wide × 4 tall, row 0 = input (top), row 3 = output (bottom)
- **5×5** (R0): Seed-picker: bottom = input; Planting: top = input
- **2×2** (R0): row 0 = input (top), row 1 = output (bottom)


In [ ]:
def _build_machine_catalogue() -> dict[str, MachineType]:
    machines = {}

    machines["refining_unit"] = MachineType(
        id="refining_unit", display_name="Refining Unit",
        base_w=3, base_h=3, input_face_r0="bottom",
        recipes=[
            Recipe([("ori_ore", 1)],             [("ori_crust", 1)],            2),
            Recipe([("ame_ore", 1)],             [("ame_fiber", 1)],            2),
            Recipe([("ferri_ore", 1)],           [("ferrium", 1)],              2),
            Recipe([("citrome", 1)],             [("carbon", 1)],               2),
            Recipe([("sandleaf", 1)],            [("carbon", 1)],               2),
            Recipe([("buckflower", 1)],          [("carbon", 1)],               2),
            Recipe([("dense_ori_pow", 1)],       [("dense_ori_crust_pow", 1)],  2),
            Recipe([("cryston_pow", 1)],         [("cryston_fiber", 1)],        2),
            Recipe([("dense_carbon_pow", 1)],    [("stabilized_carbon", 1)],    2),
            Recipe([("dense_ori_crust_pow", 1)], [("packed_origo_crust", 1)],   2),
            Recipe([("dense_ferri_pow", 1)],     [("steel", 1)],                2),
            Recipe([("sandleaf_pow", 3)],        [("carbon_pow", 2)],           2),
            Recipe([("ori_pow", 1)],             [("ori_crust_pow", 1)],        2),
        ],
        build_cost=[("ori_ore", 5)], power_cost=10,
    )

    machines["shredding_unit"] = MachineType(
        id="shredding_unit", display_name="Shredding Unit",
        base_w=3, base_h=3, input_face_r0="bottom",
        recipes=[
            Recipe([("ori_ore", 1)],    [("ori_pow", 1)],       2),
            Recipe([("ferrium", 1)],    [("ferri_pow", 1)],     2),
            Recipe([("ame_fiber", 1)],  [("ame_pow", 1)],       2),
            Recipe([("carbon", 1)],     [("carbon_pow", 2)],    2),
            Recipe([("ori_crust", 1)],  [("ori_crust_pow", 1)], 2),
            Recipe([("buckflower", 1)], [("buck_pow", 2)],      2),
            Recipe([("citrome", 1)],    [("citrome_pow", 2)],   2),
            Recipe([("sandleaf", 1)],   [("sandleaf_pow", 2)],  2),
            Recipe([("aketine", 1)],    [("ake_pow", 2)],       2),
        ],
        build_cost=[("ori_crust", 5)], power_cost=10,
    )

    machines["fitting_unit"] = MachineType(
        id="fitting_unit", display_name="Fitting Unit",
        base_w=3, base_h=3, input_face_r0="bottom",
        recipes=[
            Recipe([("steel", 1)],         [("steel_part", 1)],   2),
            Recipe([("ferrium", 1)],       [("ferri_part", 1)],   2),
            Recipe([("cryston_fiber", 1)], [("cryston_part", 1)], 2),
            Recipe([("ame_fiber", 1)],     [("ame_part", 1)],     2),
        ],
        build_cost=[("ori_crust", 10)], power_cost=10,
    )

    machines["moulding_unit"] = MachineType(
        id="moulding_unit", display_name="Moulding Unit",
        base_w=3, base_h=3, input_face_r0="bottom",
        recipes=[
            Recipe([("steel", 2)],         [("steel_bottle", 1)],    2),
            Recipe([("ferrium", 2)],       [("ferrium_bottle", 1)],  2),
            Recipe([("ame_fiber", 2)],     [("ame_bottle", 1)],      2),
            Recipe([("cryston_fiber", 2)], [("cryston_bottle", 1)],  2),
        ],
        build_cost=[("ori_crust", 10)], power_cost=10,
    )

    machines["grinding_unit"] = MachineType(
        id="grinding_unit", display_name="Grinding Unit",
        base_w=6, base_h=4, input_face_r0="top",
        recipes=[
            Recipe([("buck_pow", 2)],      [("ground_buck_pow", 1)],     2, common_extra=[("sandleaf_pow", 1)]),
            Recipe([("ori_pow", 2)],       [("dense_ori_pow", 1)],       2),
            Recipe([("ame_pow", 2)],       [("cryston_pow", 1)],         2),
            Recipe([("ferri_pow", 2)],     [("dense_ferri_pow", 1)],     2),
            Recipe([("carbon_pow", 2)],    [("dense_carbon_pow", 1)],    2),
            Recipe([("citrome_pow", 2)],   [("ground_citrome_pow", 1)],  2),
            Recipe([("ori_crust_pow", 2)], [("dense_ori_crust_pow", 1)], 2),
        ],
        build_cost=[("ferri_part", 10)], power_cost=20,
    )

    machines["filling_unit"] = MachineType(
        id="filling_unit", display_name="Filling Unit",
        base_w=6, base_h=4, input_face_r0="top",
        recipes=[
            Recipe([("steel_bottle", 10),   ("ground_buck_pow", 10)],   [("buck_cap_A", 1)], 10),
            Recipe([("steel_bottle", 10),   ("ground_citrome_pow", 10)], [("can_cit_A", 1)], 10),
            Recipe([("ferrium_bottle", 10), ("ground_buck_pow", 10)],   [("buck_cap_B", 1)], 10),
            Recipe([("ferrium_bottle", 10), ("ground_citrome_pow", 10)], [("can_cit_B", 1)], 10),
            Recipe([("ame_bottle", 10),     ("ground_buck_pow", 10)],   [("buck_cap_C", 1)], 10),
            Recipe([("ame_bottle", 10),     ("ground_citrome_pow", 10)], [("can_cit_C", 1)], 10),
        ],
        build_cost=[("ame_part", 20)], power_cost=20,
    )

    machines["packaging_unit"] = MachineType(
        id="packaging_unit", display_name="Packaging Unit",
        base_w=6, base_h=4, input_face_r0="top",
        recipes=[
            Recipe([("steel_part", 10), ("dense_ori_pow", 15)],  [("HC_bat", 1)], 10),
            Recipe([("ame_part", 5),    ("ori_pow", 10)],        [("LC_bat", 1)], 10),
            Recipe([("ferri_part", 10), ("ori_pow", 15)],        [("SC_bat", 1)], 10),
        ],
        build_cost=[("ame_part", 20)], power_cost=20,
    )

    machines["planting_unit"] = MachineType(
        id="planting_unit", display_name="Planting Unit",
        base_w=5, base_h=5, input_face_r0="top",
        recipes=[Recipe([(f"{p}_seed", 1)], [(p, 2)], 1) for p in ["citrome", "sandleaf", "buckflower", "aketine"]],
        build_cost=[("ame_part", 20), ("carbon", 10)], power_cost=30,
    )

    machines["seed_picking_unit"] = MachineType(
        id="seed_picking_unit", display_name="Seed-Picking Unit",
        base_w=5, base_h=5, input_face_r0="bottom",
        recipes=[Recipe([(p, 1)], [(f"{p}_seed", 2)], 1) for p in ["citrome", "sandleaf", "buckflower", "aketine"]],
        build_cost=[("ame_part", 20)], power_cost=30,
    )

    machines["ther_bank"] = MachineType(
        id="ther_bank", display_name="Thermal Bank",
        base_w=2, base_h=2, input_face_r0="top",
        recipes=[
            Recipe([("ori_ore", 1)],  [("power", 50)],   8),
            Recipe([("LC_bat", 1)],   [("power", 220)],  100),
            Recipe([("HC_bat", 1)],   [("power", 1100)], 100),
            Recipe([("SC_bat", 1)],   [("power", 420)],  100),
            Recipe([("LC_W_bat", 1)], [("power", 1600)], 100),
            Recipe([("SC_W_bat", 1)], [("power", 3200)], 100),
        ],
        build_cost=[("ori_crust", 10), ("ame_part", 10)],
        power_cost=0, is_power_src=True,
    )

    machines["stash"] = MachineType(
        id="stash", display_name="Stash",
        base_w=3, base_h=3, input_face_r0="bottom",
        recipes=[], build_cost=[], power_cost=10,
    )

    machines["depot_loader"] = MachineType(
        id="depot_loader", display_name="Depot Loader",
        base_w=3, base_h=1, input_face_r0="top",
        recipes=[], build_cost=[], power_cost=0, edge_only=True,
    )

    machines["depot_unloader"] = MachineType(
        id="depot_unloader", display_name="Depot Unloader",
        base_w=3, base_h=1, input_face_r0="bottom",
        recipes=[], build_cost=[], power_cost=0, edge_only=True,
    )

    machines["main_unit"] = MachineType(
        id="main_unit", display_name="Main Unit",
        base_w=9, base_h=9, input_face_r0="bottom",
        recipes=[], build_cost=[], power_cost=0, singleton=True,
    )

    return machines


MACHINE_CATALOGUE = _build_machine_catalogue()

REVENUE_TABLE: dict[str, int] = {
    "buck_cap_A": 70, "can_cit_A": 70,
    "buck_cap_B": 27, "can_cit_B": 27,
    "buck_cap_C": 10, "can_cit_C": 10,
    "HC_bat": 27, "SC_bat": 30, "LC_bat": 16,
    "ori_crust": 1, "ame_bottle": 2,
    "ame_part": 1, "ferri_part": 1, "steel_part": 3,
}

print(f"Machine types registered: {len(MACHINE_CATALOGUE)}")
print(f"Revenue items tracked   : {len(REVENUE_TABLE)}")

# Quick catalogue summary
print("\n--- Machine Catalogue ---")
for mid, mt in MACHINE_CATALOGUE.items():
    extra = []
    if mt.edge_only:    extra.append("EDGE-ONLY")
    if mt.singleton:    extra.append("SINGLETON")
    if mt.is_power_src: extra.append("POWER-SRC")
    tags   = f" [{', '.join(extra)}]" if extra else ""
    in_f, out_f = mt.io_faces(Rotation.R0)
    print(f"  {mt.display_name:25s}  {mt.base_w}x{mt.base_h}  "
          f"power={mt.power_cost:3d}  in={in_f:6s} out={out_f:6s}  "
          f"recipes={len(mt.recipes)}{tags}")

# Apply the configurable logistics build costs from the Control Panel. The
# catalogue ships these machines with build_cost=[]; see CFG.LOGISTICS_BUILD_COST.
for _mid, _cost in CFG.LOGISTICS_BUILD_COST:
    if _mid in MACHINE_CATALOGUE:
        MACHINE_CATALOGUE[_mid].build_cost = [tuple(c) for c in _cost]
print("Logistics build costs applied:",
      {m: MACHINE_CATALOGUE[m].build_cost for m, _ in CFG.LOGISTICS_BUILD_COST})


# ─── Catalogue invariants ────────────────────────────────────────────────────
# The factory boundary. derive_flow_matrix withdraws raw material from a SOURCE
# and delivers finished product to a SINK; _calculate_revenue credits output
# only once it reaches one of these (see CFG.REVENUE_AT_SINK_ONLY).
IO_SOURCE_IDS = ("main_unit", "depot_unloader")
IO_SINK_IDS   = ("main_unit", "depot_loader", "stash")

# `singleton` marks the hub, and it is read in exactly one place: the recovery
# sorts it to the front so a destroyed main_unit gets first claim on space. That
# is only correct if there is exactly one such machine type, which nothing
# checked. Asserted here rather than discovered as a silent mis-ordering.
_SINGLETON_TYPES = sorted(mid for mid, mt in MACHINE_CATALOGUE.items() if mt.singleton)
assert _SINGLETON_TYPES == ["main_unit"], (
    f"exactly one singleton machine type expected, got {_SINGLETON_TYPES}. "
    "_zpf_place sorts singletons first; more than one makes that order arbitrary.")
for _sid in IO_SOURCE_IDS + IO_SINK_IDS:
    assert _sid in MACHINE_CATALOGUE, f"boundary machine {_sid!r} missing from the catalogue"
print(f"Catalogue invariants OK: singleton={_SINGLETON_TYPES[0]}, "
      f"sources={IO_SOURCE_IDS}, sinks={IO_SINK_IDS}")

# ─── What actually counts as a raw material ──────────────────────────────────
# Ores, and only ores. They accumulate in inventory at a fixed rate per unit
# time rather than being manufactured, which is why they are the one class of
# input a layout may draw from the boundary without building anything.
#
# Stated explicitly rather than derived as "nothing in the catalogue produces
# it". That derivation is what made LC_W_bat and SC_W_bat look raw: nothing
# produces them and they are not ores, so they are UNOBTAINABLE, and the two
# ther_bank recipes that burn them are dead. Inferring "free" from "no producer"
# ranked those two dead recipes as the best power sources available.
#
# Plants are not raw either: planting_unit and seed_picking_unit form a closed
# doubling loop (seed -> 2 plant, plant -> 2 seed), so they are produced.
RAW_MATERIALS: frozenset = frozenset({"ame_ore", "ferri_ore", "ori_ore"})

def _unobtainable_materials():
    """Consumed somewhere, produced nowhere, and not an ore."""
    produced, consumed = set(), set()
    for mt in MACHINE_CATALOGUE.values():
        for r in mt.recipes:
            produced.update(o for o, _ in r.outputs)
            consumed.update(m for m, _ in list(r.inputs) + list(r.common_extra))
    return (consumed - produced) - RAW_MATERIALS - NON_CONVEYED_MATERIALS


UNOBTAINABLE: frozenset = frozenset(_unobtainable_materials())
print(f"RAW_MATERIALS  : {sorted(RAW_MATERIALS)}")
print(f"UNOBTAINABLE   : {sorted(UNOBTAINABLE)}  "
      f"(consumed, never produced, not an ore -- recipes needing these are dead)")



## 5. Material Flow Matrix

The flow matrix is **derived dynamically** from placed machines' assigned recipes.  
Each machine has exactly one active recipe; an output material from machine A is
connected to machine B only if B's recipe lists that material as an input.

Special rules:
- `packaging_unit` outputs batteries → `ther_bank`
- `main_unit` → `depot_loader`


In [ ]:
# ── Logistics roles ──────────────────────────────────────────────────────────
# Material deposited into ANY sink is available from ANY source: one global
# pool, transfer instant and free. Loaders/unloaders are edge_only; stashes are
# interior-placeable. This is what makes the BOM's parallel lines work -- a
# surplus producer deposits, and a consumer short of input withdraws.
POOL_SINKS   = ("main_unit", "depot_loader", "stash")
POOL_SOURCES = ("main_unit", "depot_unloader")

# Materials nothing in the catalogue can produce: these genuinely arrive from
# outside and may be withdrawn without ever being deposited. Everything else is
# INTERNAL and must be manufactured before it can be drawn from the pool.
RAW_MATERIALS = frozenset(
    m for m in {mat for mt in MACHINE_CATALOGUE.values() for r in mt.recipes
                for mat, _ in (list(r.inputs) + list(r.common_extra))}
    if not any(mat_out == m
               for mt in MACHINE_CATALOGUE.values() for r in mt.recipes
               for mat_out, _ in r.outputs)
)


def _machine_rates(pm):
    """(supply, demand) as {material: units per tick} for one placed machine."""
    if pm.active_recipe is None or not pm.machine_type.recipes:
        return {}, {}
    if pm.active_recipe >= len(pm.machine_type.recipes):
        return {}, {}
    r  = pm.machine_type.recipes[pm.active_recipe]
    pt = max(1, r.process_time)
    sup = {m: q / pt for m, q in r.outputs if m not in NON_CONVEYED_MATERIALS}
    dem = {}
    for m, q in list(r.inputs) + list(r.common_extra):
        dem[m] = dem.get(m, 0.0) + q / pt
    return sup, dem


def _nearest(pm, candidates):
    """Closest machine to pm by centre-to-centre Manhattan distance."""
    if not candidates:
        return None
    return min(candidates, key=lambda q: abs(q.x + q.w // 2 - pm.x - pm.w // 2)
                                        + abs(q.y + q.h // 2 - pm.y - pm.h // 2))


def derive_flow_matrix(placed_machines: list[PlacedMachine]) -> list[FlowPair]:
    """Rate-aware supply/demand matching over a global logistics pool.

    Stage 1 matches producers to consumers DIRECTLY, by rate: a consumer keeps
    drawing from further producers until its per-tick demand is met, so a
    packaging unit needing 1 ferri_part/tick is fed by the two fitting units
    that supply 0.5 each. The previous version emitted one flow per
    (material, consumer) and stopped, which left the BOM's parallel lines
    orphaned and made them look like dead weight to the pruner.

    Whatever demand is still unmet is WITHDRAWN from the pool at the nearest
    source; whatever supply is unallocated is DEPOSITED at the nearest sink.
    Flow `intensity` is the actual per-tick rate carried.
    """
    flows: list[FlowPair] = []
    by_id = {pm.instance_id: pm for pm in placed_machines}
    used_out = defaultdict(int)
    used_in  = defaultdict(int)

    sources = [pm for pm in placed_machines if pm.machine_type.id in POOL_SOURCES]
    sinks   = [pm for pm in placed_machines if pm.machine_type.id in POOL_SINKS]

    supply_left: dict[str, dict[str, float]] = defaultdict(dict)   # material -> id -> rate
    demand_left: dict[str, dict[str, float]] = defaultdict(dict)
    ing_pos: dict[tuple[str, str], int] = {}
    for pm in placed_machines:
        sup, dem = _machine_rates(pm)
        for m, r in sup.items():
            supply_left[m][pm.instance_id] = r
        for i, (m, r) in enumerate(dem.items()):
            demand_left[m][pm.instance_id] = r
            ing_pos[(m, pm.instance_id)] = i

    def _emit(src_pm, dst_pm, material, rate, dst_slot=None):
        outs = src_pm.output_belt_points()
        ins  = dst_pm.input_belt_points()
        s_idx = used_out[src_pm.instance_id] % max(1, len(outs))
        used_out[src_pm.instance_id] += 1
        if dst_slot is None:
            d_idx = used_in[dst_pm.instance_id] % max(1, len(ins))
            used_in[dst_pm.instance_id] += 1
        else:
            d_idx = dst_slot % max(1, len(ins))
        flows.append(FlowPair(src_id=src_pm.instance_id, dst_id=dst_pm.instance_id,
                              material=material, intensity=round(rate, 6),
                              src_node_idx=s_idx, dst_node_idx=d_idx))

    # ── Stage 1: direct producer -> consumer, matched by rate ────────────────
    for material in sorted(set(demand_left) & set(supply_left)):
        for con_id, need in sorted(demand_left[material].items()):
            if need <= 1e-9:
                continue
            dst_pm = by_id.get(con_id)
            if dst_pm is None:
                continue
            slot = ing_pos.get((material, con_id), 0)
            for sup_id in sorted(supply_left[material]):
                if need <= 1e-9:
                    break
                if sup_id == con_id:
                    continue
                avail = supply_left[material][sup_id]
                if avail <= 1e-9:
                    continue
                src_pm = by_id.get(sup_id)
                if src_pm is None:
                    continue
                take = min(avail, need)
                _emit(src_pm, dst_pm, material, take, slot)
                supply_left[material][sup_id] -= take
                need -= take
            demand_left[material][con_id] = need

    # ── Stage 2: withdraw unmet demand from the pool ─────────────────────────
    for material, consumers in demand_left.items():
        for con_id, need in consumers.items():
            if need <= 1e-9 or not sources:
                continue
            dst_pm = by_id.get(con_id)
            if dst_pm is None:
                continue
            src_pm = _nearest(dst_pm, [s for s in sources if s.instance_id != con_id])
            if src_pm is None:
                continue
            _emit(src_pm, dst_pm, material, need, ing_pos.get((material, con_id), 0))

    # ── Stage 3: deposit unallocated supply into the pool ────────────────────
    for material, suppliers in supply_left.items():
        for sup_id, spare in suppliers.items():
            if spare <= 1e-9 or not sinks:
                continue
            src_pm = by_id.get(sup_id)
            if src_pm is None:
                continue
            dst_pm = _nearest(src_pm, [s for s in sinks if s.instance_id != sup_id])
            if dst_pm is None:
                continue
            _emit(src_pm, dst_pm, material, spare)

    return flows


def pool_balance(placed_machines, flows, routed=None):
    """Per-material {deposited, withdrawn, deficit} across the logistics pool.

    Transfer is free and instant, so the only real constraint is conservation:
    an INTERNAL material cannot be withdrawn unless something deposited it.
    Raw materials are exempt -- they arrive from outside the factory.

    Pass `routed` to count only flows that actually have a belt.
    """
    by_id = {pm.instance_id: pm for pm in placed_machines}
    ok = None
    if routed is not None:
        ok = {(fp.src_id, fp.dst_id, fp.material) for fp, p in routed if p is not None}

    dep, wit = defaultdict(float), defaultdict(float)
    for fp in flows:
        if ok is not None and (fp.src_id, fp.dst_id, fp.material) not in ok:
            continue
        s, d = by_id.get(fp.src_id), by_id.get(fp.dst_id)
        if d is not None and d.machine_type.id in POOL_SINKS:
            dep[fp.material] += fp.intensity
        if s is not None and s.machine_type.id in POOL_SOURCES:
            wit[fp.material] += fp.intensity

    out = {}
    for m in set(dep) | set(wit):
        if m in RAW_MATERIALS:
            continue                       # imported from outside; always available
        deficit = max(0.0, wit[m] - dep[m])
        out[m] = dict(deposited=round(dep[m], 4), withdrawn=round(wit[m], 4),
                      deficit=round(deficit, 4))
    return out


print("derive_flow_matrix() rewritten: rate-aware matching + global logistics pool.")
print(f"  pool sinks   : {POOL_SINKS}")
print(f"  pool sources : {POOL_SOURCES}")
print(f"  raw (importable) materials: {len(RAW_MATERIALS)} -> {sorted(RAW_MATERIALS)}")


## 6. Factory Grid

Discrete grid, `GRID_W × GRID_H` from the Control Panel. Machine footprints and belt cells are mutually exclusive.

> **Deviation from the source model.** `database.xlsx`, sheet `rules`, specifies *"The total area for the map is 50x50"*. This notebook runs **70×70**. Nothing in the code assumes 50×50 — every reference goes through `GRID_W`/`GRID_H` — but any result quoted from it is computed on a board 1.96× the specified area, which makes placement easier and routing much easier. Either justify the enlargement in the write-up or set `GRID_W = GRID_H = 50` and re-run.

| Symbol | Meaning |
|---|---|
| `.` | Empty |
| `#` | Machine |
| `~` | Belt |
| `X` | Blocked (disruption zone) |


In [ ]:
class FactoryGrid:
    """GRID_W x GRID_H discrete grid -- machine footprints and belt cells are
    mutually exclusive.

    The source model (database.xlsx, sheet `rules`) specifies a 50x50
    map. The Control Panel sets 70x70. Nothing here assumes either --
    both dimensions come from GRID_W/GRID_H -- but the difference is a
    factor of ~2 in area and it makes routing substantially easier.
    """

    def __init__(self, width: int = GRID_W, height: int = GRID_H):
        self.width   = width
        self.height  = height
        self.grid: list[list[tuple[TileState, Optional[str]]]] = [
            [(TileState.EMPTY, None) for _ in range(width)]
            for _ in range(height)
        ]
        self.placed_machines: list[PlacedMachine] = []
        self.belt_paths: list[tuple[str, list[tuple[int, int]]]] = []
        self._components = None      # lazy routable-connectivity labels

    def in_bounds(self, x: int, y: int) -> bool:
        return 0 <= x < self.width and 0 <= y < self.height

    def is_free(self, x: int, y: int) -> bool:
        return self.in_bounds(x, y) and self.grid[y][x][0] == TileState.EMPTY

    def is_routable(self, x: int, y: int) -> bool:
        return self.in_bounds(x, y) and self.grid[y][x][0] in (TileState.EMPTY, TileState.BELT)

    def can_place(self, x: int, y: int, w: int, h: int) -> bool:
        if x < 0 or y < 0 or x + w > self.width or y + h > self.height:
            return False
        return all(self.is_free(x + dx, y + dy)
                   for dy in range(h) for dx in range(w))

    def is_on_edge(self, x: int, y: int, w: int, h: int) -> bool:
        if w > h:   return y == 0 or y + h == self.height
        elif h > w: return x == 0 or x + w == self.width
        return x == 0 or y == 0 or x + w == self.width or y + h == self.height


    def check_clearance(self, x: int, y: int, w: int, h: int, clearance: int) -> bool:
        """Check if an area, expanded by clearance, is free of other machines.
        Clearance cells that fall outside the grid bounds are ignored."""
        if x < 0 or y < 0 or x + w > self.width or y + h > self.height:
            return False

        cx_start = max(0, x - clearance)
        cy_start = max(0, y - clearance)
        cx_end   = min(self.width, x + w + clearance)
        cy_end   = min(self.height, y + h + clearance)

        for cy in range(cy_start, cy_end):
            for cx in range(cx_start, cx_end):
                if self.grid[cy][cx][0] != TileState.EMPTY:
                    return False
        return True

    def place_machine(self, pm: PlacedMachine) -> bool:
        """Occupy the footprint and register the machine. Returns True on success.

        Tiles store the machine's INSTANCE id, not its type id. remove_machine()
        matches on instance_id, so storing the type id here meant the comparison
        never matched and removed machines left their footprint permanently
        marked MACHINE -- invisible ghosts that blocked A* forever. Nothing reads
        a MACHINE tile's payload except remove_machine (belt tiles store an axis
        set in the same slot), so instance_id is the correct thing to store.
        """
        w, h = pm.w, pm.h
        if not self.can_place(pm.x, pm.y, w, h):
            return False
        if pm.machine_type.edge_only and not self.is_on_edge(pm.x, pm.y, w, h):
            return False
        for (cx, cy) in pm.cells():
            self.grid[cy][cx] = (TileState.MACHINE, pm.instance_id)
        self.placed_machines.append(pm)
        self._invalidate_components()
        return True

    def remove_machine(self, pm: PlacedMachine, drop_belts: bool = True) -> bool:
        """Free the footprint, drop the machine, and optionally erase its belts.

        Belts are removed too: a belt whose endpoint machine is gone is dead
        weight that still blocks routing, which defeats the whole point of
        removing the machine (connect_or_remove_unlinked removes machines
        specifically to free space for the next routing pass).
        """
        freed = 0
        for dy in range(pm.h):
            for dx in range(pm.w):
                cx, cy = pm.x + dx, pm.y + dy
                if not self.in_bounds(cx, cy):
                    continue
                if self.grid[cy][cx][1] == pm.instance_id:
                    self.grid[cy][cx] = (TileState.EMPTY, None)
                    freed += 1
        if pm in self.placed_machines:
            self.placed_machines.remove(pm)

        self._invalidate_components()
        if drop_belts:
            keep = []
            for label, path in self.belt_paths:
                if pm.instance_id in label.split('->'):
                    for (bx, by) in path:
                        if self.in_bounds(bx, by) and self.grid[by][bx][0] == TileState.BELT:
                            self.grid[by][bx] = (TileState.EMPTY, None)
                else:
                    keep.append((label, path))
            self.belt_paths = keep
            # Belts that survived may have shared cells with the erased ones;
            # repaint them so a crossing isn't accidentally left empty.
            for label, path in self.belt_paths:
                self._paint_belt(path)
        return freed > 0

    def place_belt(self, path: list[tuple[int, int]], label: str = "belt") -> bool:
        """Place belt, allowing perpendicular intersections."""
        if any(not self.in_bounds(x, y) or not self.is_routable(x, y) for x, y in path):
            return False
        self._paint_belt(path)
        self.belt_paths.append((label, list(path)))
        return True

    def _paint_belt(self, path: list[tuple[int, int]]) -> None:
        """Mark every cell of `path` as BELT, recording which axes it carries."""
        for i, (x, y) in enumerate(path):
            axes = set()
            if i > 0:
                dx, dy = abs(x - path[i-1][0]), abs(y - path[i-1][1])
                if dx > 0: axes.add('H')
                if dy > 0: axes.add('V')
            if i < len(path) - 1:
                dx, dy = abs(path[i+1][0] - x), abs(path[i+1][1] - y)
                if dx > 0: axes.add('H')
                if dy > 0: axes.add('V')

            if self.grid[y][x][0] == TileState.EMPTY:
                self.grid[y][x] = (TileState.BELT, axes)
            elif self.grid[y][x][0] == TileState.BELT:
                existing = self.grid[y][x][1]
                if not isinstance(existing, set):
                    existing = set()
                self.grid[y][x] = (TileState.BELT, existing.union(axes))



    # ── routable-connectivity cache (performance only) ───────────────────────
    # A* returns None when no path exists, but only after expanding every
    # reachable cell -- and most routing attempts in a congested layout fail.
    # Two cells in different connected components of the routable set can never
    # be joined, so that case is answered here in O(1) after one flood fill.
    #
    # Soundness: this is computed on is_routable() alone and IGNORES the belt
    # axis-crossing rules, so it is an OPTIMISTIC upper bound on connectivity.
    # "different component" therefore implies "A* would fail" (no false
    # negatives); "same component" decides nothing and A* still runs. Results
    # are unchanged, only the hopeless searches are skipped.
    #
    # Invalidation: place_belt turns EMPTY into BELT and both are routable, so
    # belt placement cannot change connectivity and does not invalidate. Only
    # machine placement/removal does.

    def _invalidate_components(self):
        self._components = None

    def routable_component_of(self, x: int, y: int) -> int:
        """Component id of (x, y), or -1 if the cell is not routable."""
        if getattr(self, '_components', None) is None:
            self._build_components()
        if not self.in_bounds(x, y):
            return -1
        return self._components[y][x]

    def _build_components(self) -> None:
        from collections import deque as _deque
        W, H, g = self.width, self.height, self.grid
        routable = (TileState.EMPTY, TileState.BELT)
        lab = [[-1] * W for _ in range(H)]
        cid = 0
        for sy in range(H):
            row = lab[sy]
            for sx in range(W):
                if row[sx] != -1 or g[sy][sx][0] not in routable:
                    continue
                q = _deque([(sx, sy)])
                row[sx] = cid
                while q:
                    cx, cy = q.popleft()
                    for nx, ny in ((cx + 1, cy), (cx - 1, cy), (cx, cy + 1), (cx, cy - 1)):
                        if 0 <= nx < W and 0 <= ny < H and lab[ny][nx] == -1 \
                                and g[ny][nx][0] in routable:
                            lab[ny][nx] = cid
                            q.append((nx, ny))
                cid += 1
        self._components = lab

    def rebuild_belts(self, labelled_paths) -> None:
        """Replace the entire belt layer with `labelled_paths` [(label, path), ...].

        Belts are otherwise write-only: place_belt() adds them and nothing ever
        takes one away, so a flow that cannot find a corridor has no way to
        displace a belt sitting in its path. This makes the belt layer
        rewritable so rip-up-and-reroute can evict and re-lay.
        """
        for y in range(self.height):
            row = self.grid[y]
            for x in range(self.width):
                if row[x][0] == TileState.BELT:
                    row[x] = (TileState.EMPTY, None)
        self.belt_paths = []
        for label, path in labelled_paths:
            self._paint_belt(path)
            self.belt_paths.append((label, list(path)))
        self._invalidate_components()

    def render(self) -> str:
        sym_map = {TileState.EMPTY: ".", TileState.MACHINE: "#",
                   TileState.BELT: "~",  TileState.BLOCKED: "X"}
        hdr   = "   " + "".join(f"{c % 10}" for c in range(self.width))
        lines = [hdr]
        for y in range(self.height):
            row = [sym_map.get(self.grid[y][x][0], "?") for x in range(self.width)]
            lines.append(f"{y:2d} {''.join(row)}")
        return "\n".join(lines)

print("FactoryGrid defined.")



## 7. Chromosome Encoding

Each **gene** encodes:
- `machine_id` — which machine type to place
- `rotation` — one of R0/R90/R180/R270
- `recipe_idx` — which recipe the machine runs (index into `MachineType.recipes`)

A **chromosome** is a permutation of genes; the order determines BLF placement order.


In [ ]:
@dataclass
class Gene:
    machine_id: str
    rotation:   Rotation
    recipe_idx: Optional[int] = None
@dataclass
class Chromosome:
    genes: list[Gene]
    @staticmethod
    def random(
        machine_ids: list[str],
        recipe_assignments: Optional[list[Optional[int]]] = None,
    ) -> "Chromosome":
        """Create a random chromosome with shuffled gene order."""
        genes = []
        for i, mid in enumerate(machine_ids):
            rot  = random.choice(list(Rotation))
            ridx = (recipe_assignments[i] if recipe_assignments is not None
                    else (0 if MACHINE_CATALOGUE[mid].recipes else None))
            genes.append(Gene(mid, rot, ridx))
        random.shuffle(genes)
        return Chromosome(genes)
    def copy(self) -> "Chromosome":
        return Chromosome([Gene(g.machine_id, g.rotation, g.recipe_idx)
                           for g in self.genes])
print("Gene and Chromosome defined.")


## 8. BLF Placement Decoder

**Bottom-Left Fill (BLF):** for each gene, scan the grid left-to-right, top-to-bottom
and place the machine at the first valid (empty, in-bounds, edge-rules-satisfied) location.  
Instance IDs are generated as `{type_id}_{counter}` to distinguish multiple machines
of the same type.


In [ ]:
# BLF_VERBOSE is owned by the Control Panel (CFG.SHOW_BLF_WARNINGS). It used
# to be re-declared here, which was harmless while the config cell ran AFTER
# this one -- but the panel now runs first, so an unconditional re-declaration
# silently reset the knob and SHOW_BLF_WARNINGS had no effect.
BLF_VERBOSE = globals().get('BLF_VERBOSE', False)
BLF_UNPLACED_COUNT = 0

def derive_topology(chromosome):
    """Run a mock flow match to get topological neighbors before placement."""
    mock_placed = []
    instance_counters = defaultdict(int)
    for gene in chromosome.genes:
        idx = instance_counters[gene.machine_id]
        instance_counters[gene.machine_id] += 1
        inst_id = f"{gene.machine_id}_{idx}"
        mtype = MACHINE_CATALOGUE[gene.machine_id]
        pm = PlacedMachine(machine_type=mtype, x=0, y=0, rotation=gene.rotation,
                           instance_id=inst_id, active_recipe=gene.recipe_idx)
        mock_placed.append(pm)

    flows = derive_flow_matrix(mock_placed)
    neighbors = defaultdict(set)
    for f in flows:
        neighbors[f.src_id].add(f.dst_id)
        neighbors[f.dst_id].add(f.src_id)
    return neighbors

def generate_spiral(cx, cy, max_r):
    """Yields (x, y) coordinates radiating outward from cx, cy."""
    yield cx, cy
    for r in range(1, max_r + 1):
        for x in range(cx - r, cx + r): yield x, cy - r
        for y in range(cy - r, cy + r): yield cx + r, y
        for x in range(cx + r, cx - r, -1): yield x, cy + r
        for y in range(cy + r, cy - r, -1): yield cx - r, y

def blf_decode(chromosome: Chromosome, grid: FactoryGrid,
               rng: Optional[random.Random] = None) -> list[PlacedMachine]:
    """Decode a chromosome onto the grid: adjacency-biased spiral, then fallback sweep.

    `rng` MUST be threaded through by callers that care about reproducibility.
    This used to do a bare `import random; random.shuffle(...)` inside the gene
    loop, which (a) made every seeded rng handed to evaluate_layout cosmetic --
    the decode itself drew from the global stream, so the same chromosome
    decoded differently on consecutive calls -- and (b) perturbed the very
    stream DEAP uses for crossover and mutation.

    The fallback sweep is also cached per (bounds) key instead of being rebuilt
    and reshuffled for EVERY gene: it is a ~4,700-element list over the whole
    grid, and reshuffling it 160 times per individual dominated decode runtime.
    Machines of the same footprint now share one shuffled order, which is
    harmless -- earlier genes still claim their cells first.
    """
    global BLF_UNPLACED_COUNT
    rng = rng if rng is not None else random
    placed: list[PlacedMachine] = []
    instance_counters: dict[str, int] = defaultdict(int)

    topology = derive_topology(chromosome)
    placed_by_id = {}
    fallback_cache: dict[tuple[int, int, int, int], list[tuple[int, int]]] = {}

    for gene in chromosome.genes:
        mtype = MACHINE_CATALOGUE[gene.machine_id]
        w, h  = mtype.size(gene.rotation)

        idx     = instance_counters[gene.machine_id]
        instance_counters[gene.machine_id] += 1
        inst_id = f"{gene.machine_id}_{idx}"

        y_start, x_start = (0, 0) if mtype.edge_only else (1, 1)
        y_end   = grid.height - h + 1
        x_end   = grid.width  - w + 1

        search_sequence = []
        placed_neighbors = [placed_by_id[nid] for nid in topology[inst_id] if nid in placed_by_id]
        if placed_neighbors and not mtype.edge_only:
            cx = int(sum(p.x + p.w/2 for p in placed_neighbors) / len(placed_neighbors))
            cy = int(sum(p.y + p.h/2 for p in placed_neighbors) / len(placed_neighbors))
            cx, cy = int(cx - w/2), int(cy - h/2)

            for nx, ny in generate_spiral(cx, cy, 25):
                if x_start <= nx < x_end and y_start <= ny < y_end:
                    search_sequence.append((nx, ny))

        cache_key = (x_start, y_start, x_end, y_end)
        fallback_sequence = fallback_cache.get(cache_key)
        if fallback_sequence is None:
            fallback_sequence = [(x, y)
                                 for y in range(y_start, y_end)
                                 for x in range(x_start, x_end)]
            rng.shuffle(fallback_sequence)
            fallback_cache[cache_key] = fallback_sequence
        search_sequence.extend(fallback_sequence)

        seen = set()
        unique_search = []
        for nx, ny in search_sequence:
            if (nx, ny) not in seen:
                seen.add((nx, ny))
                unique_search.append((nx, ny))

        success = False
        for clearance_val in [MIN_MACHINE_SPACING]:
            if success:
                break
            for x, y in unique_search:
                if mtype.edge_only and not grid.is_on_edge(x, y, w, h):
                    continue
                if not grid.check_clearance(x, y, w, h, clearance=clearance_val):
                    continue
                pm = PlacedMachine(machine_type=mtype, x=x, y=y,
                                   rotation=gene.rotation,
                                   instance_id=inst_id,
                                   active_recipe=gene.recipe_idx)
                if not mtype.edge_only:
                    ports_ok = all(grid.in_bounds(px, py)
                                   for px, py in
                                   pm.input_belt_points() + pm.output_belt_points())
                    if not ports_ok:
                        continue
                if not grid.place_machine(pm):
                    continue          # never append a machine the grid rejected
                placed.append(pm)
                placed_by_id[inst_id] = pm
                success = True
                break

        if not success:
            BLF_UNPLACED_COUNT += 1
            if BLF_VERBOSE:
                print(f"  [BLF] WARNING: Could not place {mtype.display_name} ({w}x{h})")

    return placed

print("blf_decode() defined (seeded rng, cached fallback sweep).")




## 9. A\* Pathfinding & Belt Routing

### 9.1 A\* Search
- Cardinal movement only (N/E/S/W)
- Every cell in the path must be **EMPTY** (machines and placed belts are impassable)
- Returns shortest path `[(x0,y0), ..., (xn,yn)]` or `None`

### 9.2 Belt Routing
- 1:1 mapping: one belt from one output connect point to one input connect point
- Tries all `(output_belt_point, input_belt_point)` pairs, picks the shortest path
- Once placed, belt cells become `BELT` state and block future routes


In [ ]:
def astar(
    grid:  FactoryGrid,
    start: tuple[int, int],
    goal:  tuple[int, int],
) -> Optional[list[tuple[int, int]]]:
    """A* on routable cells, cardinal moves only, honouring belt axis rules.

    Functionally identical to the original -- same expansion order, same
    tie-breaking, same returned path -- but the interior uses flat arrays
    indexed by (y * width + x) instead of dicts and sets keyed by (x, y)
    tuples. Tuple construction and hashing dominated the profile: 638k dict
    lookups, 489k set inserts and ~670k tuple allocations for a single
    34-machine evaluation.

    Two cheap exits come first:
      * either endpoint not routable -> None;
      * endpoints in different connected components of the routable set ->
        None, decided in O(1) from FactoryGrid's cached labels rather than by
        expanding every reachable cell. (The label map ignores belt axis rules,
        so it is an optimistic bound: it can only prove impossibility, never
        possibility, which is exactly what is needed here.)
    """
    sx, sy = start
    gx, gy = goal
    W, H, g = grid.width, grid.height, grid.grid
    if not (0 <= sx < W and 0 <= sy < H and 0 <= gx < W and 0 <= gy < H):
        return None
    EMPTY, BELT = TileState.EMPTY, TileState.BELT
    if g[sy][sx][0] not in (EMPTY, BELT) or g[gy][gx][0] not in (EMPTY, BELT):
        return None
    if (sx, sy) == (gx, gy):
        return [(sx, sy)]
    if grid.routable_component_of(sx, sy) != grid.routable_component_of(gx, gy):
        return None

    size      = W * H
    g_score   = [-1] * size          # -1 == not reached (infinity)
    came_from = [-1] * size
    closed    = bytearray(size)

    s_idx, g_idx = sy * W + sx, gy * W + gx
    g_score[s_idx] = 0
    counter = 0
    open_set = [(abs(sx - gx) + abs(sy - gy), 0, s_idx)]
    heappush, heappop = heapq.heappush, heapq.heappop

    while open_set:
        _, _, cur = heappop(open_set)
        if cur == g_idx:
            path = []
            node = cur
            while node != -1:
                path.append((node % W, node // W))
                node = came_from[node]
            path.reverse()
            return path
        if closed[cur]:
            continue
        closed[cur] = 1

        cy, cx = divmod(cur, W)
        cell = g[cy][cx]
        cur_axes = cell[1] if (cell[0] is BELT and isinstance(cell[1], set)) else None
        if cur_axes is not None and 'H' in cur_axes and 'V' in cur_axes:
            continue
        cur_h = 'H' in cur_axes if cur_axes is not None else False
        cur_v = 'V' in cur_axes if cur_axes is not None else False
        base = g_score[cur] + 1

        for dx, dy in ((0, 1), (0, -1), (1, 0), (-1, 0)):
            nx, ny = cx + dx, cy + dy
            if nx < 0 or nx >= W or ny < 0 or ny >= H:
                continue
            nidx = ny * W + nx
            if closed[nidx]:
                continue
            ncell = g[ny][nx]
            nstate = ncell[0]
            if nstate is not EMPTY and nstate is not BELT:
                continue
            if (cur_h and dx != 0) or (cur_v and dy != 0):
                continue
            if nstate is BELT:
                axes = ncell[1] if isinstance(ncell[1], set) else ()
                a_h, a_v = 'H' in axes, 'V' in axes
                if (a_h and a_v) or (a_h and dx != 0) or (a_v and dy != 0):
                    continue
            prev = g_score[nidx]
            if prev < 0 or base < prev:
                g_score[nidx] = base
                came_from[nidx] = cur
                counter += 1
                heappush(open_set, (base + abs(nx - gx) + abs(ny - gy), counter, nidx))
    return None


def route_belts(
    grid:          FactoryGrid,
    placed_lookup: dict[str, PlacedMachine],
    flow_matrix:   list[FlowPair],
) -> list[tuple[FlowPair, Optional[list[tuple[int, int]]]]]:
    """Route each FlowPair using its specific assigned src/dst node indices.

    For single-ingredient machines, the assigned node is typically the centre.
    For multi-ingredient machines (Filling, Packaging), each ingredient gets a
    dedicated node slot on the input face.
    For multi-output machines (seed_picking_unit), each output belt uses a
    different output node slot.
    """
    results = []
    for fp in flow_matrix:
        src_pm = placed_lookup.get(fp.src_id)
        dst_pm = placed_lookup.get(fp.dst_id)
        if src_pm is None or dst_pm is None:
            results.append((fp, None))
            continue

        # Get ALL external belt points (outside machine, adjacent to face)
        all_out_bps = src_pm.output_belt_points()
        all_in_bps  = dst_pm.input_belt_points()

        if not all_out_bps or not all_in_bps:
            results.append((fp, None))
            continue

        # Pick the specific node assigned to this FlowPair
        src_node_idx = fp.src_node_idx % len(all_out_bps)
        dst_node_idx = fp.dst_node_idx % len(all_in_bps)

        # Preferred start/end points
        preferred_out = all_out_bps[src_node_idx]
        preferred_in  = all_in_bps[dst_node_idx]

        # Try the preferred pairing first, then fall back to any routable pair
        best_path, best_len = None, float('inf')

        candidates_out = [preferred_out] + [p for p in all_out_bps
                          if p != preferred_out and grid.in_bounds(*p) and grid.is_routable(*p)]
        candidates_in  = [preferred_in]  + [p for p in all_in_bps
                          if p != preferred_in  and grid.in_bounds(*p) and grid.is_routable(*p)]

        # Try preferred pair first
        if grid.in_bounds(*preferred_out) and grid.is_routable(*preferred_out) and            grid.in_bounds(*preferred_in)  and grid.is_routable(*preferred_in):
            path = astar(grid, preferred_out, preferred_in)
            if path is not None:
                best_path, best_len = path, len(path)

        # Fall back to other port pairs if the preferred one fails. The search
        # continues across OUT-PORTS rather than stopping at the first one that
        # works -- that was the bug -- but it is capped at
        # CFG.ROUTE_FALLBACK_MAX_PAIRS A* calls, because this branch is reached
        # on exactly the flows a congested layout has most of. See the note in
        # the Control Panel for the measured cost of leaving it uncapped.
        if best_path is None:
            tried = {(preferred_out, preferred_in)}   # already attempted above
            budget = CFG.ROUTE_FALLBACK_MAX_PAIRS if CFG.ROUTE_FALLBACK_BEST else 0
            for op in candidates_out:
                if not (grid.in_bounds(*op) and grid.is_routable(*op)):
                    continue
                for ip in candidates_in:
                    if (op, ip) in tried:
                        continue
                    tried.add((op, ip))
                    if not (grid.in_bounds(*ip) and grid.is_routable(*ip)):
                        continue
                    path = astar(grid, op, ip)
                    budget -= 1
                    if path is not None and len(path) < best_len:
                        best_path, best_len = path, len(path)
                    if budget <= 0 and best_path is not None:
                        break
                if best_path is not None and budget <= 0:
                    break

        if best_path is not None and grid.place_belt(best_path, f"{fp.src_id}->{fp.dst_id}"):
            results.append((fp, best_path))
        else:
            results.append((fp, None))
    return results

print("astar() and route_belts() defined.")



## 10.5  Feasibility Repair — Starved Machine Recovery

After belt routing, some machines may be **starved** (no incoming belt path
for required inputs). This repair pass fixes feasibility in priority order:

1. **Thermal banks** (power) — highest priority. Must be connected to a
   battery producer or a depot unloader.
2. **Production machines** — reconnect to nearest supplier, or place a
   depot unloader beside them as a last resort.


In [ ]:
# ── Feasibility repair ────────────────────────────────────────────────────────

def find_starved_connections(placed_machines, flow_matrix, belt_results):
    """
    Identify flow pairs that failed routing (no belt path).
    Returns list of (FlowPair, src_pm, dst_pm) for failed connections.
    """
    placed_lookup = {pm.instance_id: pm for pm in placed_machines}
    starved = []
    for fp, path in belt_results:
        if path is None:
            src_pm = placed_lookup.get(fp.src_id)
            dst_pm = placed_lookup.get(fp.dst_id)
            if src_pm and dst_pm:
                starved.append((fp, src_pm, dst_pm))
    return starved


def find_unconnected_ther_banks(placed_machines, belt_results):
    """
    Find thermal banks that have NO incoming belt with a battery material.
    These are the critical power failures.
    """
    battery_mats = set(BATTERY_POWER.keys()) - {'power'}
    ther_banks = [pm for pm in placed_machines if pm.machine_type.id == 'ther_bank']

    # Which ther_banks got a successful incoming belt?
    connected_tb_ids = set()
    for fp, path in belt_results:
        if path is not None and fp.material in battery_mats:
            if fp.dst_id.startswith('ther_bank_'):
                connected_tb_ids.add(fp.dst_id)

    return [tb for tb in ther_banks if tb.instance_id not in connected_tb_ids]


def try_place_depot_unloader_near(grid, target_pm, material=None):
    # None means 'whatever this machine actually burns'. The old default
    # spawned an SC_bat depot next to a bank running the HC_bat recipe.
    if material is None:
        material = _bank_fuel(target_pm)
    """
    Try to place a depot_unloader adjacent to target_pm on a grid edge.
    Returns the PlacedMachine if successful, None otherwise.
    """
    mt = MACHINE_CATALOGUE['depot_unloader']

    # Try all 4 rotations and positions near the target
    candidates = []
    for rot in list(Rotation):
        w, h = mt.size(rot)
        # Try positions around the target machine
        for dy in range(-h, target_pm.h + 1):
            for dx in range(-w, target_pm.w + 1):
                x = target_pm.x + dx
                y = target_pm.y + dy
                if not grid.can_place(x, y, w, h):
                    continue
                if mt.edge_only and not grid.is_on_edge(x, y, w, h):
                    continue
                # Check distance to target
                dist = abs(x + w//2 - target_pm.x - target_pm.w//2) + \
                       abs(y + h//2 - target_pm.y - target_pm.h//2)
                candidates.append((dist, x, y, rot))

    if not candidates:
        return None

    # Pick closest valid position
    candidates.sort(key=lambda item: (item[0], item[1], item[2], item[3].value))
    dist, x, y, rot = candidates[0]
    w, h = mt.size(rot)

    # Create a unique instance id
    existing_ids = {pm.instance_id for pm in grid.placed_machines}
    idx = 0
    while f'depot_unloader_{idx}' in existing_ids:
        idx += 1

    pm = PlacedMachine(
        machine_type=mt, x=x, y=y, rotation=rot,
        instance_id=f'depot_unloader_{idx}',
        active_recipe=None)
    grid.place_machine(pm)
    return pm


def repair_flow_intensity(dst_pm, material, default=1.0):
    """Per-tick demand `dst_pm` has for `material`, for a repair belt's
    intensity. Repair belts used to be created with a flat intensity=1.0, which
    is the one number MHC weights by -- so a repaired connection carrying 10
    units per run and one carrying 0.05 cost the search the same."""
    if dst_pm is None or dst_pm.active_recipe is None or not dst_pm.machine_type.recipes:
        return default
    if dst_pm.active_recipe >= len(dst_pm.machine_type.recipes):
        return default
    rec = dst_pm.machine_type.recipes[dst_pm.active_recipe]
    qty = sum(q for m, q in list(rec.inputs) + list(rec.common_extra) if m == material)
    if qty <= 0:
        return default
    return qty / max(1, rec.process_time)


def repair_feasibility(grid, placed_machines, flow_matrix, belt_results,
                       battery_item=None, verbose=True):   # None -> per-bank fuel
    """
    Post-routing repair pass.

    Priority order:
      1. Reconnect starved thermal banks (power is critical)
      2. Reconnect other starved production machines

    Strategies:
      A. Re-attempt routing with relaxed path (try all output/input point combos)
      B. Place a depot_unloader near the starved machine and route from it

    Returns (repaired_count, new_machines, new_belts).

    `new_machines` are NOT added to `placed_machines` -- the caller must do
    `placed.extend(new_machines)` itself (see the note at the append site).

    `new_belts` entries are (src_id, dst_id, material, path, existing_fp).
    `existing_fp` is the FlowPair this belt REPAIRS when the connection was
    already in the flow matrix and merely failed to route -- the caller must
    mark that pair routed rather than append a second FlowPair for the same
    connection. Appending was the old behaviour and it double-counted: the flow
    stayed in the matrix as unrouted AND a duplicate arrived routed, so
    flows_total grew by one on every repair while the original never went away.
    `existing_fp` is None for a genuinely new connection (the depot spawned to
    feed an unfuelled thermal bank), and those carry the destination's real
    per-tick demand for the material instead of a flat 1.0.
    """
    placed_lookup = {pm.instance_id: pm for pm in placed_machines}
    repaired = 0
    new_machines = []
    new_belts = []
    battery_mats = set(BATTERY_POWER.keys()) - {'power'}

    # ── Phase 1: Thermal banks ────────────────────────────────────────────
    unconnected_tbs = find_unconnected_ther_banks(placed_machines, belt_results)

    if verbose and unconnected_tbs:
        print(f"  [REPAIR] {len(unconnected_tbs)} thermal bank(s) without power supply")

    for tb in unconnected_tbs:
        fixed = False

        # Strategy A: find ANY battery producer and try to route
        for pm in placed_machines:
            if pm.machine_type.id == 'packaging_unit' and pm.active_recipe is not None:
                recipe = pm.machine_type.recipes[pm.active_recipe]
                produces_battery = any(mat in battery_mats for mat, _ in recipe.outputs)
                if not produces_battery:
                    continue

                # Try all output→input point combinations
                out_pts = [(x, y) for x, y in pm.output_belt_points()
                           if grid.in_bounds(x, y) and grid.is_free(x, y)]
                in_pts  = [(x, y) for x, y in tb.input_belt_points()
                           if grid.in_bounds(x, y) and grid.is_free(x, y)]

                best_path = None
                best_len = float('inf')
                for op in out_pts:
                    for ip in in_pts:
                        path = astar(grid, op, ip)
                        if path and len(path) < best_len:
                            best_path, best_len = path, len(path)

                if best_path and grid.place_belt(best_path, f"{pm.instance_id}->{tb.instance_id}"):
                    bat_mat = next(mat for mat, _ in recipe.outputs if mat in battery_mats)
                    new_belts.append((pm.instance_id, tb.instance_id, bat_mat,
                                      best_path, None))
                    repaired += 1
                    fixed = True
                    if verbose:
                        print(f"  [REPAIR] Connected {tb.instance_id} ← {pm.instance_id} "
                              f"({best_len} cells)")
                    break

        # Strategy B: place a depot_unloader near the thermal bank
        if not fixed:
            depot = try_place_depot_unloader_near(grid, tb, battery_item or _bank_fuel(tb))
            if depot:
                # NOTE: deliberately does NOT append to `placed_machines`.
                # It used to do both, while _decode_instances ALSO did
                # `placed.extend(rep_machines)` -- so every magic depot landed
                # in the caller's list twice, inflating len(placed) (hence
                # all_placed), build_cost and power demand, and letting ALNS's
                # `unplaced = len(instances) - len(placed)` go negative.
                # Callers now extend explicitly; this function only reports.
                new_machines.append(depot)
                placed_lookup[depot.instance_id] = depot

                # Route from depot to thermal bank
                out_pts = [(x, y) for x, y in depot.output_belt_points()
                           if grid.in_bounds(x, y) and grid.is_free(x, y)]
                in_pts  = [(x, y) for x, y in tb.input_belt_points()
                           if grid.in_bounds(x, y) and grid.is_free(x, y)]

                best_path = None
                best_len = float('inf')
                for op in out_pts:
                    for ip in in_pts:
                        path = astar(grid, op, ip)
                        if path and len(path) < best_len:
                            best_path, best_len = path, len(path)

                if best_path and grid.place_belt(best_path,
                        f"{depot.instance_id}->{tb.instance_id}"):
                    new_belts.append((depot.instance_id, tb.instance_id,
                                     (battery_item or _bank_fuel(tb)), best_path, None))
                    repaired += 1
                    fixed = True
                    if verbose:
                        print(f"  [REPAIR] Placed {depot.instance_id} at "
                              f"({depot.x},{depot.y}) -> {tb.instance_id} "
                              f"({best_len} cells)")
                elif verbose:
                    print(f"  [REPAIR] Placed depot but could not route to "
                          f"{tb.instance_id}")
            elif verbose:
                print(f"  [REPAIR] FAILED: no space for depot near {tb.instance_id}")

    # ── Phase 2: Other starved machines ───────────────────────────────────
    starved = find_starved_connections(placed_machines, flow_matrix, belt_results)
    # Filter out thermal bank connections (already handled)
    starved = [(fp, s, d) for fp, s, d in starved
               if not d.instance_id.startswith('ther_bank_')]

    if verbose and starved:
        print(f"  [REPAIR] {len(starved)} other starved connection(s)")

    for fp, src_pm, dst_pm in starved:
        # Strategy A: re-attempt routing
        out_pts = [(x, y) for x, y in src_pm.output_belt_points()
                   if grid.in_bounds(x, y) and grid.is_free(x, y)]
        in_pts  = [(x, y) for x, y in dst_pm.input_belt_points()
                   if grid.in_bounds(x, y) and grid.is_free(x, y)]

        best_path = None
        best_len = float('inf')
        for op in out_pts:
            for ip in in_pts:
                path = astar(grid, op, ip)
                if path and len(path) < best_len:
                    best_path, best_len = path, len(path)

        if best_path and grid.place_belt(best_path, f"{fp.src_id}->{fp.dst_id}"):
            # `fp` is already in the flow matrix -- it just failed to route.
            # Hand it back so the caller marks THAT pair routed instead of
            # appending a duplicate connection.
            new_belts.append((fp.src_id, fp.dst_id, fp.material, best_path, fp))
            repaired += 1
            if verbose:
                print(f"  [REPAIR] Reconnected {fp.src_id} -> {fp.dst_id} "
                      f"({fp.material}, {best_len} cells)")

    if verbose:
        print(f"  [REPAIR] Done: {repaired} connection(s) repaired, "
              f"{len(new_machines)} machine(s) added")

    return repaired, new_machines, new_belts


def apply_repair_belts(placed, flows, routed, rep_belts):
    """Fold repair_feasibility's belts into (flows, routed).

    A belt that repairs an EXISTING FlowPair marks that pair routed in place; a
    belt for a genuinely new connection is appended to BOTH lists with the
    destination's real per-tick demand as its intensity. Appending only to
    `routed` inflates paths_found without moving len(flows), which drives
    `unrouted` negative and rewards the search for spawning depots instead of
    routing real flows; appending a duplicate for an existing pair inflates both.
    """
    if not rep_belts:
        return flows, routed
    by_id = {pm.instance_id: pm for pm in placed}
    index = {id(fp): k for k, (fp, _) in enumerate(routed)}
    for entry in rep_belts:
        src_id, dst_id, mat, path = entry[0], entry[1], entry[2], entry[3]
        existing = entry[4] if len(entry) > 4 else None
        if existing is not None and id(existing) in index:
            k = index[id(existing)]
            routed[k] = (existing, path)
            continue
        fp_rep = FlowPair(src_id=src_id, dst_id=dst_id, material=mat,
                          intensity=repair_flow_intensity(by_id.get(dst_id), mat))
        flows.append(fp_rep)
        routed.append((fp_rep, path))
    return flows, routed


print("Feasibility repair tools defined:")
print("  find_starved_connections(), find_unconnected_ther_banks()")
print("  try_place_depot_unloader_near(), repair_feasibility()")


## 11. Matplotlib Visualisation

Renders the `GRID_W × GRID_H` grid (70×70 here; the source model specifies 50×50 — see section 6) with:
- Coloured machine rectangles with display names
- Input node markers (red square `■`)
- Output node markers (green triangle `▲`)
- Belt paths (orange lines)
- Flow arrows (red arrows between I/O nodes)
- Legend and title with key fitness metrics


In [ ]:
from types import SimpleNamespace

COLOUR_MAP = {
    "main_unit":         "#2C3E50",
    "ther_bank":         "#F1C40F",
    "refining_unit":     "#E74C3C",
    "shredding_unit":    "#E67E22",
    "fitting_unit":      "#27AE60",
    "moulding_unit":     "#2980B9",
    "grinding_unit":     "#F39C12",
    "filling_unit":      "#8E44AD",
    "packaging_unit":    "#16A085",
    "planting_unit":     "#2ECC71",
    "seed_picking_unit": "#1ABC9C",
    "stash":             "#95A5A6",
    "depot_loader":      "#3498DB",
    "depot_unloader":    "#9B59B6",
}

# ─── Material Belt Colour Palette ────────────────────────────────────────────

# ─── Production-Line Belt Colour System ──────────────────────────────────────
# Each belt is colored by the FINAL REVENUE PRODUCT its production chain
# leads to, so the entire chain for e.g. buck_cap_A is one unified color.

_PRODUCT_LINE_PALETTE = [
    '#e6194B',  # red
    '#3cb44b',  # green  
    '#4363d8',  # blue
    '#f58231',  # orange
    '#911eb4',  # purple
    '#42d4f4',  # cyan
    '#f032e6',  # magenta
    '#469990',  # teal
    '#9A6324',  # brown
    '#800000',  # maroon
    '#808000',  # olive
    '#000075',  # navy
    '#e6beff',  # lavender
    '#ffe119',  # yellow
    '#aaffc3',  # mint
    '#ffd8b1',  # apricot
]


def _trace_production_lines(flow_matrix, placed_machines):
    """Trace each FlowPair backward/forward through the production DAG
    to find the final revenue product(s) it contributes to.
    
    Returns a dict: flow_key (src_id, dst_id, material) -> set of revenue product names.
    """
    # Build adjacency: machine_id -> list of outgoing FlowPairs
    outgoing = defaultdict(list)
    for fp in flow_matrix:
        outgoing[fp.src_id].append(fp)
    
    # Find which machines produce revenue items
    machine_revenue = {}  # machine_id -> revenue product name
    for pm in placed_machines:
        if pm.active_recipe is None or not pm.machine_type.recipes:
            continue
        if pm.active_recipe >= len(pm.machine_type.recipes):
            continue
        recipe = pm.machine_type.recipes[pm.active_recipe]
        for mat, qty in recipe.outputs:
            if mat in REVENUE_TABLE:
                machine_revenue[pm.instance_id] = mat
                break
    
    # For each flow, trace forward through the DAG to find the revenue product
    # it ultimately contributes to. Use BFS from each machine.
    machine_to_product = {}  # machine_id -> revenue product name
    
    # First, directly assign machines that produce revenue items
    for mid, prod in machine_revenue.items():
        machine_to_product[mid] = prod
    
    # BFS backward: for machines that DON'T directly produce revenue items,
    # trace forward through flows to find what revenue product they feed into.
    # Build reverse adjacency for backward tracing
    incoming = defaultdict(list)
    for fp in flow_matrix:
        incoming[fp.dst_id].append(fp)
    
    # Forward trace from each machine to find the revenue product it feeds
    def _find_revenue_product(machine_id, visited=None):
        if visited is None:
            visited = set()
        if machine_id in visited:
            return None
        visited.add(machine_id)
        
        if machine_id in machine_to_product:
            return machine_to_product[machine_id]
        
        # Follow outgoing flows
        for fp in outgoing[machine_id]:
            result = _find_revenue_product(fp.dst_id, visited)
            if result is not None:
                return result
        return None
    
    # Assign every machine to a revenue product
    for pm in placed_machines:
        if pm.instance_id not in machine_to_product:
            prod = _find_revenue_product(pm.instance_id)
            if prod is not None:
                machine_to_product[pm.instance_id] = prod
    
    # Now map each flow to its revenue product
    flow_to_product = {}
    for fp in flow_matrix:
        key = (fp.src_id, fp.dst_id, fp.material)
        # Use the destination's product assignment, fallback to source's
        prod = machine_to_product.get(fp.dst_id) or machine_to_product.get(fp.src_id)
        flow_to_product[key] = prod
    
    return flow_to_product, machine_to_product


def _get_product_line_colour(product_name, colour_map):
    """Deterministic colour for a revenue product name."""
    if product_name is None:
        return '#a9a9a9'  # grey for unassigned
    if product_name not in colour_map:
        idx = len(colour_map) % len(_PRODUCT_LINE_PALETTE)
        colour_map[product_name] = _PRODUCT_LINE_PALETTE[idx]
    return colour_map[product_name]


def plot_layout(
    grid:        FactoryGrid,
    result,                       # FitnessResult-like: any object with
                                  # .mhc/.revenue/.power_feasible/.chain_time/
                                  # .paths_found/.paths_total/.feasible
    flow_matrix: list[FlowPair],
    save_path:   str = "layout_merged.png",
    zpf_zone=None,
    title_str=None,
    ax=None,
    show_legend=True,
):
    """Matplotlib visualisation of the factory layout.
    
    Belts are color-coded by the final revenue product their production
    chain leads to (not by the individual material they carry).
    """
    if not HAS_MPL:
        print("[SKIP] matplotlib not available.")
        return

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(13, 13))

    ax.set_xlim(0, grid.width)
    ax.set_ylim(0, grid.height)
    ax.set_aspect('equal')
    ax.invert_yaxis()

    for x in range(grid.width + 1):
        ax.axvline(x, color='#E0E0E0', linewidth=0.3, zorder=0)
    for y in range(grid.height + 1):
        ax.axhline(y, color='#E0E0E0', linewidth=0.3, zorder=0)

    seen_types    = {}
    placed_lookup = {}
    for pm in grid.placed_machines:
        placed_lookup[pm.instance_id] = pm
        color = COLOUR_MAP.get(pm.machine_type.id, '#BDC3C7')
        ax.add_patch(plt.Rectangle(
            (pm.x, pm.y), pm.w, pm.h,
            facecolor=color, edgecolor='#2C3E50',
            linewidth=1.5, alpha=0.85, zorder=2))
        ax.text(pm.x + pm.w / 2, pm.y + pm.h / 2,
                pm.machine_type.display_name.replace(' ', '\n'),
                ha='center', va='center',
                fontsize=5.5, color='white', fontweight='bold', zorder=3)
        slot_colors = ['#e74c3c', '#e67e22', '#9b59b6', '#1abc9c', '#3498db', '#e91e63']
        for slot_i, (nx, ny) in enumerate(pm.input_belt_points()):
            col = slot_colors[slot_i % len(slot_colors)]
            ax.plot(nx + 0.5, ny + 0.5, 's', color=col, markersize=3.5, zorder=4)
        for slot_i, (nx, ny) in enumerate(pm.output_belt_points()):
            alpha = 0.6 + 0.1 * (slot_i % 4)
            ax.plot(nx + 0.5, ny + 0.5, '^', color='#2ecc71', markersize=3.5,
                    alpha=min(alpha, 1.0), zorder=4)
        seen_types[pm.machine_type.display_name] = color

    # Trace production lines for color assignment
    flow_to_product, _ = _trace_production_lines(flow_matrix, grid.placed_machines)
    product_colours = {}  # Fresh per call -- no cross-plot leakage

    # Build belt label -> product mapping
    belt_product_map = {}
    for fp in flow_matrix:
        label = f"{fp.src_id}->{fp.dst_id}"
        key = (fp.src_id, fp.dst_id, fp.material)
        belt_product_map[label] = flow_to_product.get(key)

    # Where each belt physically starts and ends. route_belts may fall back to a
    # different (out-port, in-port) pair than the one derive_flow_matrix assigned
    # without updating fp.src_node_idx/dst_node_idx, so drawing arrows from those
    # indices made 4 of 27 arrows point at a port the belt never touched.
    belt_ends = {}
    for _lbl, _p in grid.belt_paths:
        if _p:
            belt_ends.setdefault(_lbl, (_p[0], _p[-1]))

    seen_products = set()
    for label, path in grid.belt_paths:
        product = belt_product_map.get(label)
        belt_col = _get_product_line_colour(product, product_colours)
        if product:
            seen_products.add(product)
        xs = [p[0] + 0.5 for p in path]
        ys = [p[1] + 0.5 for p in path]
        ax.plot(xs, ys, '-', color=belt_col, linewidth=1.5, alpha=0.8, zorder=1.5)

    if zpf_zone is not None:
        cx, cy = zpf_zone.x + zpf_zone.w / 2, zpf_zone.y + zpf_zone.h / 2
        ax.add_patch(plt.Rectangle(
            (zpf_zone.x, zpf_zone.y), zpf_zone.w, zpf_zone.h,
            facecolor='#c0392b', edgecolor='#c0392b', alpha=0.35, zorder=6,
            label='Crater (unusable)'))
        ax.add_patch(plt.Circle((cx, cy), zpf_zone.r1, fill=False,
                                 edgecolor='#e67e22', linewidth=1.5,
                                 linestyle='--', zorder=6, label='Free zone'))
        ax.add_patch(plt.Circle((cx, cy), zpf_zone.r2, fill=False,
                                 edgecolor='#f1c40f', linewidth=1.5,
                                 linestyle=':', zorder=6, label='Penalty zone'))

    # Flow arrows colored by production line
    for fp in flow_matrix:
        src_pm = placed_lookup.get(fp.src_id)
        dst_pm = placed_lookup.get(fp.dst_id)
        if src_pm is None or dst_pm is None:
            continue
        out_pts = src_pm.output_belt_points()
        in_pts  = dst_pm.input_belt_points()
        if not out_pts or not in_pts:
            continue
        ends = belt_ends.get(f"{fp.src_id}->{fp.dst_id}")
        if ends is not None:                 # routed: follow the real belt ports
            (snx, sny), (dnx, dny) = ends
            dash, alpha = '-', 0.5
        else:                                # unrouted: no belt exists at all
            snx, sny = out_pts[fp.src_node_idx % len(out_pts)]
            dnx, dny = in_pts[fp.dst_node_idx  % len(in_pts)]
            dash, alpha = (0, (4, 3)), 0.9
        key = (fp.src_id, fp.dst_id, fp.material)
        arrow_col = _get_product_line_colour(flow_to_product.get(key), product_colours)
        ax.annotate('', xy=(dnx + 0.5, dny + 0.5), xytext=(snx + 0.5, sny + 0.5),
                    arrowprops=dict(arrowstyle='->', color=arrow_col, linestyle=dash,
                                    lw=min(2.0, fp.intensity * 0.35), alpha=alpha), zorder=5)

    if show_legend:
        patches = [mpatches.Patch(color=c, label=name) for name, c in seen_types.items()]
        if zpf_zone is not None:
            patches.append(mpatches.Patch(color='#c0392b', alpha=0.35, label='Crater (unusable)'))
            patches.append(plt.Line2D([0], [0], color='#e67e22', linestyle='--', label='Free zone'))
            patches.append(plt.Line2D([0], [0], color='#f1c40f', linestyle=':', label='Penalty zone'))
        # Add production line legend entries
        for prod in sorted(seen_products):
            patches.append(plt.Line2D([0], [0], color=_get_product_line_colour(prod, product_colours),
                                       linewidth=2.5, label=f'Line: {prod}'))
        patches.append(plt.Line2D([0], [0], color='#a9a9a9',
                                   linewidth=2, label='Line: (power/utility)'))
        patches.append(plt.Line2D([0], [0], marker='s', color='w',
                                   markerfacecolor='#e74c3c', markersize=6, label='Input Node'))
        patches.append(plt.Line2D([0], [0], marker='^', color='w',
                                   markerfacecolor='#2ecc71', markersize=6, label='Output Node'))
        patches.append(plt.Line2D([0], [0], color='#555', linestyle=(0, (4, 3)),
                                   linewidth=1.5, label='Flow with NO belt (unrouted)'))
        ax.legend(handles=patches, loc='lower right', fontsize=5, framealpha=0.9, ncol=2)

    if hasattr(result, 'feasible'):
        feas_str = "FEASIBLE" if result.feasible else "INFEASIBLE"
    else:
        feas_str = "N/A"
    
    ax.set_title(
        title_str if title_str is not None else
        f'Valley IV Factory Layout\n'
        f'MHC={result.mhc:.0f}  |  Rev={result.revenue:.2f} cr/tick  |  '
        f'Power {"OK" if result.power_feasible else "DEFICIT"}  |  '
        f'Chain={result.chain_time:.1f}t/{SIM_EQ_MAX_TICKS}t  |  '
        f'Paths {result.paths_found}/{result.paths_total}  |  '
        f'{feas_str}',
        fontsize=9)
    ax.set_xlabel('Grid X')
    ax.set_ylabel('Grid Y')

    if own_fig:
        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Saved to {save_path}")


def plot_zpf_sequence(pre_result, post_result, rec_result, zpf_zone,
                      pre_feas=None, post_feas=None, rec_feas=None,
                      pre_rate=0.0, post_rate=0.0, rec_rate=0.0,
                      scenario_note=None,
                      save_path="zpf_sequence.png"):
    """Three-panel side-by-side: Pre-Bomb | Post-Bomb | Post-Recovery."""
    if not HAS_MPL:
        print("[SKIP] matplotlib not available.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(39, 13))

    panels = [
        (pre_result,  "PRE-BOMB",        pre_feas or 0.0, pre_rate),
        (post_result, "POST-BOMB",       post_feas or 0.0, post_rate),
        (rec_result,  "POST-RECOVERY",   rec_feas or 0.0, rec_rate),
    ]

    for ax_i, (res, label, feas, rate) in zip(axes, panels):
        fr = SimpleNamespace(
            mhc=res["mhc"], revenue=rate,
            power_feasible=res["power_ok"], chain_time=res["chain_time"],
            paths_found=res["paths_found"], paths_total=res["flows_total"],
            feasible=res["feasible"],
        )
        plot_layout(res["grid"], fr, res["flows"],
                    zpf_zone=zpf_zone, ax=ax_i, show_legend=(ax_i is axes[-1]),
                    title_str=f"{label}\nRate={rate:.2f}  |  Feasibility={feas:.1%}  |  MHC={res['mhc']:.0f}")

    # State the scenario on the figure. Without it the reader cannot tell
    # whether the drawn crater is the one the recovery was computed from.
    if scenario_note:
        fig.suptitle(scenario_note, fontsize=17, y=1.005)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Saved to {save_path}")





def plot_gantt(engine, eq_tick=None, max_ticks=160, title=None, save_path=None, ax=None):
    """Per-machine status over time.

    Draws one bar per CONTIGUOUS RUN of identical status rather than one bar per
    tick per machine (which was ~20k patches for a 160-machine, 120-tick run).
    """
    if not HAS_MPL:
        print("[SKIP] matplotlib not available"); return None

    STATUS_COLOUR = {
        'idle':    '#95A5A6',
        'running': '#27AE60',
        'starved': '#E74C3C',
        'blocked': '#F39C12',
        'unpowered': '#8E44AD',
    }
    machine_ids = sorted(engine.states.keys(),
                         key=lambda m: (m.rsplit('_', 1)[0], int(m.rsplit('_', 1)[-1])
                                        if m.rsplit('_', 1)[-1].isdigit() else 0))
    if not machine_ids:
        print("[SKIP] No machines to plot."); return None

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(15, max(4, len(machine_ids) * 0.28)))
    ticks = min(max_ticks, engine.tick_count)

    for row, mid in enumerate(machine_ids):
        history = engine.states[mid].status_history[:ticks]
        if not history:
            continue
        run_start, run_status = 0, history[0]
        for t in range(1, len(history) + 1):
            if t == len(history) or history[t] != run_status:
                ax.barh(row, t - run_start, left=run_start, height=0.8,
                        color=STATUS_COLOUR.get(run_status, '#BDC3C7'), edgecolor='none')
                if t < len(history):
                    run_start, run_status = t, history[t]

    if eq_tick is not None and eq_tick <= ticks:
        ax.axvline(eq_tick, color='#2C3E50', linestyle='--', linewidth=1.8,
                   label=f'Equilibrium @ tick {eq_tick}')

    ax.set_yticks(range(len(machine_ids)))
    ax.set_yticklabels(machine_ids, fontsize=5.5)
    ax.set_xlabel('Tick')
    ax.set_xlim(0, max(1, ticks))
    ax.set_title(title or 'Machine Status Gantt Chart', fontsize=11)
    ax.invert_yaxis()

    patches = [mpatches.Patch(color=c, label=s) for s, c in STATUS_COLOUR.items()]
    if eq_tick is not None and eq_tick <= ticks:
        patches.append(plt.Line2D([0], [0], color='#2C3E50', linestyle='--',
                                  label=f'Eq @ {eq_tick}'))
    ax.legend(handles=patches, loc='lower right', fontsize=7, ncol=5, framealpha=0.9)

    if own_fig:
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"  Saved {save_path}")
        plt.show()
    return ax


def plot_throughput(engine, title=None, save_path=None, ax=None, top_n=12):
    """Cumulative production over time, one line per material."""
    if not HAS_MPL:
        print("[SKIP] matplotlib not available"); return None
    if not engine.production_log:
        print("[SKIP] No production recorded - nothing ever completed a recipe."); return None

    materials = {}
    for tick, iid, mat, qty in engine.production_log:
        materials.setdefault(mat, {})
        materials[mat][tick] = materials[mat].get(tick, 0) + qty

    totals = {m: sum(d.values()) for m, d in materials.items()}
    keep = sorted(totals, key=lambda m: -totals[m])[:top_n]

    own_fig = ax is None
    if own_fig:
        fig, ax = plt.subplots(figsize=(12, 5))
    colors = plt.cm.tab20.colors
    for idx, mat in enumerate(keep):
        tick_data = materials[mat]
        ticks_sorted = sorted(tick_data)
        cumulative, total = [], 0
        for t in ticks_sorted:
            total += tick_data[t]
            cumulative.append(total)
        style = '-' if mat in REVENUE_TABLE else '--'
        ax.plot(ticks_sorted, cumulative, style, color=colors[idx % 20],
                linewidth=1.8 if mat in REVENUE_TABLE else 1.1,
                label=f"{mat}{' *' if mat in REVENUE_TABLE else ''}", alpha=0.9)

    ax.set_xlabel('Tick')
    ax.set_ylabel('Cumulative Units Produced')
    ax.set_title(title or 'Production Throughput  (* = revenue item)', fontsize=11)
    ax.legend(fontsize=7, ncol=3, loc='upper left')
    ax.grid(alpha=0.3)
    if own_fig:
        plt.tight_layout()
        if save_path:
            plt.savefig(save_path, dpi=150, bbox_inches='tight')
            print(f"  Saved {save_path}")
        plt.show()
    return ax


print("plot_layout(), plot_zpf_sequence(), plot_gantt(), plot_throughput() defined.")


### 14.1  Simulation data structures

In [ ]:


@dataclass
class BeltPath:
    """A routed belt connection between two machine instances.
    path is the actual sequence of (x,y) grid cells from source output
    connect point to destination input connect point."""
    src_id:    str              # instance_id of source machine
    dst_id:    str              # instance_id of destination machine
    material:  str              # material flowing on this belt
    path:      list             # [(x0,y0), ..., (xN,yN)]
    intensity: float

    @property
    def transit_ticks(self) -> int:
        """Ticks to traverse this belt path (1 cell = 1 tick)."""
        return max(1, len(self.path) - 1)


@dataclass
class InTransitItem:
    """One unit of material currently moving along a belt path."""
    material:      str
    qty:           float
    path_idx:      int      # index into SimEngine.belt_paths
    ticks_elapsed: int = 0

    def current_cell(self, belt_paths: list) -> tuple:
        bp  = belt_paths[self.path_idx]
        idx = min(self.ticks_elapsed, len(bp.path) - 1)
        return bp.path[idx]

    def is_delivered(self, belt_paths: list) -> bool:
        return self.ticks_elapsed >= belt_paths[self.path_idx].transit_ticks


@dataclass
class MachineSimState:
    """Runtime production state for one placed machine instance.

    status:
      'idle'    – no recipe running
      'running' – recipe countdown active
      'starved' – tried to start but input buffer insufficient
      'blocked' – output buffer full (not modelled; reserved)
      'unpowered' – power pool could not cover this tick's demand
    """
    instance_id:        str
    placed:             object            # PlacedMachine reference

    buffer_in:          dict = field(default_factory=dict)
    buffer_out:         dict = field(default_factory=dict)
    active_recipe_idx:  object = None    # int or None
    ticks_remaining:    int    = 0
    status:             str    = 'idle'
    status_history:     list   = field(default_factory=list)
    runs_completed:     int    = 0


print("Simulation data structures defined: BeltPath, InTransitItem, MachineSimState")


### 14.2  Simulation inputs — belt paths and boundary supply


In [ ]:
# ── Simulation inputs, built from the ACTUAL routed layout ──────────────────

def build_belt_paths_from_routed(routed) -> list:
    """One BeltPath per SUCCESSFULLY routed flow.

    Replaces build_belt_paths(grid, flow_matrix), which reconstructed belts by
    re-parsing grid.belt_paths labels back into (src, dst) and looking them up
    in the flow matrix. That was lossy in two ways: a {src->dst} label is not
    unique when two materials run between the same pair (the dict collapsed
    them), and grid.belt_paths also contains belts drawn by repair/retry passes
    that were later superseded. `routed` is the authoritative pairing of flow to
    path, so build straight from it.
    """
    return [
        BeltPath(src_id=fp.src_id, dst_id=fp.dst_id, material=fp.material,
                 path=list(path), intensity=fp.intensity)
        for fp, path in routed if path is not None
    ]


# Machine types that represent the factory boundary as a SOURCE: they hold no
# recipe and stand in for "the outside world" delivering raw material.
# depot_loader is deliberately NOT here -- it is a SINK (finished goods leave
# through it), so it must never inject material. It was listed here by mistake;
# harmless in practice because derive_flow_matrix never makes a loader a flow
# source, but wrong if the graph ever changed. stash is likewise sink-only.
BOUNDARY_SUPPLIERS = {"main_unit", "depot_unloader"}


def build_injection_map(placed_machines, routed) -> dict:
    """{instance_id: {materials injected from outside}}.

    A machine's input is injected for free ONLY when the routed belt carrying
    that material originates at a boundary supplier (main_unit / depot). Every
    other ingredient has to arrive down a real belt from a real producer.

    This replaces find_dag_roots(), which returned "machines with no incoming
    flow that have a recipe". In the v4 pipeline main_unit supplies every raw
    material, so essentially no machine is a DAG root and that set came back
    empty -- nothing was ever injected and the simulation produced nothing at
    all. It also injected ALL of a machine's inputs whenever it qualified, which
    would hand free material to a half-connected machine and hide exactly the
    starvation the Gantt chart exists to show.
    """
    lookup = {pm.instance_id: pm for pm in placed_machines}
    injection: dict[str, set] = {}
    for fp, path in routed:
        if path is None:
            continue
        src = lookup.get(fp.src_id)
        if src is None or src.machine_type.id not in BOUNDARY_SUPPLIERS:
            continue
        injection.setdefault(fp.dst_id, set()).add(fp.material)

    # A machine with a recipe but no incoming routed belt at all is a genuine
    # chain head (nothing upstream exists to feed it); treat its inputs as
    # boundary-supplied so the chain can start, and let the Gantt show whether
    # everything downstream of it actually keeps up.
    fed = {fp.dst_id for fp, p in routed if p is not None}
    for pm in placed_machines:
        if pm.instance_id in fed or pm.instance_id in injection:
            continue
        if pm.active_recipe is None or not pm.machine_type.recipes:
            continue
        if pm.active_recipe >= len(pm.machine_type.recipes):
            continue
        rec = pm.machine_type.recipes[pm.active_recipe]
        mats = {m for m, _ in rec.inputs} | {m for m, _ in rec.common_extra}
        if mats:
            injection[pm.instance_id] = mats
    return injection


print("build_belt_paths_from_routed() and build_injection_map() defined.")


### 14.3  Simulation engine

In [ ]:
class SimEngine:
    """Discrete-time factory simulation.

    One tick = one simulation time unit.
    Belt speed = 1 cell / tick.
    Machine speed = recipe.process_time ticks per run.

    Designed to run AFTER evaluate_fitness() has placed machines and belts.
    """

    def __init__(
        self,
        placed_machines: list,
        belt_paths:      list,
        flow_matrix:     list,
        revenue_table:   dict,
        injection_map:   dict = None,
        raw_supply_ids:  set  = None,
        inject_every:    int  = 1,
        enforce_power:   bool = False,
        power_grace_ticks: int = 0,
        min_live_fraction: float = 0.0,
    ):
        self.belt_paths     = belt_paths
        self.revenue_table  = revenue_table
        self.inject_every   = inject_every
        # injection_map: {instance_id: {materials supplied from outside the
        # factory}}. Per-material, so a machine that gets one ingredient from a
        # depot and another from a real producer only gets the depot one for
        # free -- it still starves if the real belt never lands.
        if injection_map is not None:
            self.injection_map = {k: set(v) for k, v in injection_map.items()}
        else:
            self.injection_map = {iid: None for iid in (raw_supply_ids or set())}

        # One state per placed machine instance
        self.states: dict = {}
        for pm in placed_machines:
            self.states[pm.instance_id] = MachineSimState(
                instance_id=pm.instance_id,
                placed=pm,
            )

        # ── power economy ────────────────────────────────────────────────
        # Demand is summed over every PLACED machine, running or not. That is
        # deliberate: charging only machines that happen to be running would
        # reward starving them, which is exactly the loophole this model must
        # not have.
        self.enforce_power       = enforce_power
        self.power_grace_ticks   = power_grace_ticks
        self.min_live_fraction   = min_live_fraction
        self.power_demand_per_tick = sum(pm.machine_type.power_cost for pm in placed_machines)
        self.power_pool          = 0.0
        self.power_history:  list = []
        self.unpowered_ticks: int = 0
        self.powered:        bool = True

        self.in_transit:      list = []
        self.production_log:  list = []   # [(tick, instance_id, material, qty)]
        self.total_produced:  dict = {}
        self.tick_count:      int  = 0
        self.snapshots:       list = []
        self._snapshot_every: int  = 5

        # Fractional accumulator for sub-tick supply rates
        self._accum: dict[tuple[str, str], float] = {}
        # Supply rates per tick: 250 ferri/min=4.17/tick, 60 ori/min=1.0/tick, 30 ame/min=0.5/tick
        self.supply_rates: dict[str, float] = {
            'ferri_ore': 3.0 * TICK_SECONDS, 'ori_ore': 5.0 * TICK_SECONDS, 'ame_ore': 3.0 * TICK_SECONDS,
            'buckflower': 1.0 * TICK_SECONDS, 'sandleaf': 0.5 * TICK_SECONDS, 'citrome': 1.0 * TICK_SECONDS, 'aketine': 0.5 * TICK_SECONDS,
            # Batteries are IMPORTED fuel for the thermal banks (the BOM never
            # manufactures them -- banks are appended after the expansion loop,
            # so their demand never re-enters the queue). Without an entry here
            # the injector fell back to qty/process_time = 1/100 = 0.01/tick,
            # so a bank waited 100 ticks for one battery and then burned it over
            # another 100 -- measured 99 ticks starved before its first run. A
            # bank consumes 0.01 batteries/tick, so 0.05 keeps it fed.
            'HC_bat': 0.05, 'SC_bat': 0.05, 'LC_bat': 0.05,
        }

    # ── Core tick ─────────────────────────────────────────────────────────────

    def tick(self) -> None:
        t = self.tick_count = self.tick_count + 1

        # Step 1: advance belt items
        still_moving = []
        for item in self.in_transit:
            item.ticks_elapsed += 1
            if item.is_delivered(self.belt_paths):
                bp    = self.belt_paths[item.path_idx]
                state = self.states.get(bp.dst_id)
                if state is not None:
                    buf = state.buffer_in
                    buf[item.material] = buf.get(item.material, 0) + item.qty
            else:
                still_moving.append(item)
        self.in_transit = still_moving

        # Step 2: inject raw materials into DAG root machines (fractional accumulator)
        # Supply rates (per tick): ferri=4.17, ori=1.0, ame=0.5
        # Uses fractional accumulators so partial-tick supply is never lost
        for iid, allowed in self.injection_map.items():
            state = self.states.get(iid)
            if state is None:
                continue
            pm  = state.placed
            if pm.active_recipe is None or not pm.machine_type.recipes:
                continue
            ridx = min(pm.active_recipe, len(pm.machine_type.recipes) - 1)
            _rec = pm.machine_type.recipes[ridx]
            for mat, qty in list(_rec.inputs) + list(_rec.common_extra):
                if allowed is not None and mat not in allowed:
                    continue          # this ingredient must arrive on a real belt
                rate = self.supply_rates.get(mat, 0.0)
                if rate <= 0:
                    rate = float(qty) / max(1.0, float(_rec.process_time))
                key = (iid, mat)
                self._accum[key] = self._accum.get(key, 0.0) + rate
                inject_qty = int(self._accum[key])
                if inject_qty > 0:
                    state.buffer_in[mat] = state.buffer_in.get(mat, 0) + inject_qty
                    self._accum[key] -= inject_qty

        # Step 3: complete running recipes
        for state in self.states.values():
            if state.status != 'running':
                continue
            state.ticks_remaining -= 1
            if state.ticks_remaining <= 0:
                pm   = state.placed
                ridx = state.active_recipe_idx
                rec  = pm.machine_type.recipes[ridx]
                for out_mat, out_qty in rec.outputs:
                    if out_mat in NON_CONVEYED_MATERIALS:
                        # Electricity has no belt. It went into buffer_out and
                        # sat there forever -- measured: 3,300 units generated,
                        # 3,300 still in the buffer, 0 belts carrying it.
                        #
                        # out_qty is a RATED power, sustained for the whole
                        # process_time of the burn, so the energy delivered is
                        # rating x duration. Crediting the bare rating would
                        # make one HC_bat cover only 1100 machine-ticks instead
                        # of 110,000, i.e. it would silently reimpose the
                        # energy-batch reading that _power_output rejects.
                        self.power_pool += out_qty * max(1, rec.process_time)
                    else:
                        state.buffer_out[out_mat] = (
                            state.buffer_out.get(out_mat, 0) + out_qty)
                    self.production_log.append(
                        (t, state.instance_id, out_mat, out_qty))
                    self.total_produced[out_mat] = (
                        self.total_produced.get(out_mat, 0) + out_qty)
                state.runs_completed    += 1
                state.active_recipe_idx  = None
                state.ticks_remaining    = 0
                state.status             = 'idle'

        # Step 3.5: settle the power bill for this tick.
        # Grace is a fixed tick count, never "until equilibrium" -- see the note
        # on STARTUP_FREE. Already-running recipes are allowed to finish; only
        # STARTING new work requires power.
        if (not self.enforce_power) or t <= self.power_grace_ticks:
            self.powered = True
        elif self.power_pool >= self.power_demand_per_tick:
            self.power_pool -= self.power_demand_per_tick
            self.powered = True
        else:
            self.powered = False
            self.unpowered_ticks += 1
        self.power_history.append(self.power_pool)

        # Step 4: start new recipes (idle, starved or unpowered machines)
        for state in self.states.values():
            if state.status not in ('idle', 'starved', 'unpowered'):
                continue
            # Generators are exempt. Gating them on grid power deadlocks the
            # factory: a brownout stops the thermal banks from burning fuel,
            # which is the only way the pool ever refills. Verified -- without
            # this, all 19 machines sat 'unpowered' from tick 61 to 400.
            if not self.powered and not state.placed.machine_type.is_power_src:
                state.status = 'unpowered'
                continue
            pm = state.placed
            if pm.active_recipe is None or not pm.machine_type.recipes:
                state.status = 'idle'
                continue
            ridx = min(pm.active_recipe, len(pm.machine_type.recipes) - 1)
            rec  = pm.machine_type.recipes[ridx]
            # common_extra is a real consumed ingredient (e.g. grinding r0 needs
            # sandleaf_pow on top of buck_pow); it was silently ignored here, so
            # such machines ran without one of their inputs.
            required = list(rec.inputs) + list(rec.common_extra)
            if all(state.buffer_in.get(mat, 0) >= qty for mat, qty in required):
                for mat, qty in required:
                    state.buffer_in[mat] -= qty
                state.active_recipe_idx = ridx
                # process_time is already in TICKS (see Recipe). Dividing by
                # TICK_SECONDS here halved every duration and put the simulation
                # on a different clock from _chain_time, which uses it directly.
                state.ticks_remaining   = max(1, int(rec.process_time))
                state.status            = 'running'
            else:
                state.status = 'starved'

        # Step 5: dispatch output to belt paths
        for state in self.states.values():
            if not state.buffer_out:
                continue
            for path_idx, bp in enumerate(self.belt_paths):
                if bp.src_id != state.instance_id:
                    continue
                qty_avail = state.buffer_out.get(bp.material, 0)
                if qty_avail <= 0:
                    continue
                qty_send = min(1.0, qty_avail)
                state.buffer_out[bp.material] -= qty_send
                self.in_transit.append(InTransitItem(
                    material=bp.material, qty=qty_send,
                    path_idx=path_idx, ticks_elapsed=0))

        # Step 6: record status history
        for state in self.states.values():
            state.status_history.append(state.status)

        # Step 7: snapshot
        if t <= 20 or t % self._snapshot_every == 0:
            self.snapshots.append({
                'tick'          : t,
                'in_transit'    : [(it.path_idx, it.ticks_elapsed,
                                    it.material, it.qty)
                                   for it in self.in_transit],
                'statuses'      : {iid: s.status
                                   for iid, s in self.states.items()},
                'total_produced': dict(self.total_produced),
                'buffer_in'     : {iid: dict(s.buffer_in)
                                   for iid, s in self.states.items()},
            })

    # ── Equilibrium ───────────────────────────────────────────────────────────

    def _rate_over(self, lo_tick: int, hi_tick: int) -> dict:
        """{material: units produced} for ticks in [lo_tick, hi_tick]."""
        counts = {}
        for tick, _iid, mat, qty in self.production_log:
            if lo_tick <= tick <= hi_tick:
                counts[mat] = counts.get(mat, 0) + qty
        return counts

    def live_fraction(self, window: int = 30) -> float:
        """Fraction of recipe-bearing machines that produced inside the window."""
        recipe_ids = {
            iid for iid, st in self.states.items()
            if st.placed.active_recipe is not None and st.placed.machine_type.recipes
        }
        if not recipe_ids:
            return 1.0
        lo = max(1, self.tick_count - window + 1)
        produced = {e[1] for e in self.production_log if e[0] >= lo}
        return len(recipe_ids & produced) / len(recipe_ids)

    def is_equilibrium(self, window: int = 30, threshold: float = 0.08,
                       min_live_fraction: float = None) -> bool:
        """True when production is stable AND the factory is actually alive.

        Two things were wrong here.

        1. `production_log[-window:]` slices the last `window` LOG ENTRIES, not
           the last `window` TICKS. Measured on a real run, the last 30 entries
           spanned 8 ticks. The comparison window therefore narrowed as
           throughput rose, so a factory that sped up looked *more* stable.
           Both windows are now selected by tick.

        2. Only production_log was consulted, and a machine that never completes
           a recipe writes nothing to it. A layout could leave most of its
           machines permanently starved and still be declared converged --
           stably broken. `min_live_fraction` now requires a share of the
           recipe-bearing machines to have actually produced.
        """
        if self.tick_count < window * 2:
            return False
        if min_live_fraction is None:
            min_live_fraction = self.min_live_fraction
        if self.live_fraction(window) < min_live_fraction:
            return False

        t  = self.tick_count
        r1 = self._rate_over(t - window + 1, t)
        r2 = self._rate_over(t - 2 * window + 1, t - window)
        if not r1 or not r2:
            return False
        for mat in set(r1) | set(r2):
            v1, v2 = r1.get(mat, 0.0), r2.get(mat, 0.0)
            if v2 == 0:
                if v1 > 0: return False
                continue
            if abs(v1 - v2) / v2 > threshold:
                return False
        return True

    def run(self, max_ticks: int = 400, window: int = 30,
            threshold: float = 0.08, verbose: bool = True) -> object:
        """Run until equilibrium or max_ticks. Returns equilibrium tick or None."""
        eq_tick = None
        for _ in range(max_ticks):
            self.tick()
            if eq_tick is None and self.is_equilibrium(window, threshold):
                eq_tick = self.tick_count
        if verbose:
            if eq_tick:
                print(f"  Equilibrium at tick {eq_tick} / {max_ticks}")
            else:
                print(f"  No equilibrium within {max_ticks} ticks")
        return eq_tick

    # ── Metrics ───────────────────────────────────────────────────────────────

    def throughput_at_equilibrium(self, window: int = 30) -> dict:
        """Mean units/tick per material over the last `window` TICKS.

        Was `production_log[-window:]`, i.e. the last `window` log ENTRIES,
        divided by `window` as though those entries were ticks. That is what
        made simulated revenue come out as implausibly round numbers.
        """
        lo   = max(1, self.tick_count - window + 1)
        span = max(1, self.tick_count - lo + 1)
        return {mat: qty / span for mat, qty in self._rate_over(lo, self.tick_count).items()}

    def simulated_revenue_per_tick(self, window: int = 30) -> float:
        """Revenue/tick at equilibrium — more accurate than analytical estimate."""
        thr = self.throughput_at_equilibrium(window)
        return sum(
            self.revenue_table.get(mat, 0) * rate
            for mat, rate in thr.items()
        )

    def pipeline_latencies(self) -> list:
        """Transit time per belt path, sorted by descending latency."""
        return sorted(
            [{'src': bp.src_id, 'dst': bp.dst_id,
              'material': bp.material,
              'path_len': len(bp.path),
              'transit_ticks': bp.transit_ticks,
              'intensity': bp.intensity}
             for bp in self.belt_paths],
            key=lambda x: -x['transit_ticks']
        )


def simulate_layout(result, label="layout", max_ticks=None, eq_window=None,
                    eq_threshold=None, verbose=True):
    """Run the discrete-time simulation on a v4 result dict.

    This is the bridge that was missing: SimEngine, BeltPath, plot_gantt and
    plot_throughput were all fully implemented but nothing in the v4 pipeline
    ever constructed an engine, so Gate 2 was only ever the static
    _chain_time() DP. Returns (engine, eq_tick).
    """
    max_ticks    = CFG.SIM_MAX_TICKS    if max_ticks    is None else max_ticks
    eq_window    = CFG.SIM_EQ_WINDOW    if eq_window    is None else eq_window
    eq_threshold = CFG.SIM_EQ_THRESHOLD if eq_threshold is None else eq_threshold

    belts     = build_belt_paths_from_routed(result["routed"])
    injection = build_injection_map(result["placed"], result["routed"])

    engine = SimEngine(
        placed_machines   = result["placed"],
        belt_paths        = belts,
        flow_matrix       = result["flows"],
        revenue_table     = REVENUE_TABLE,
        injection_map     = injection,
        enforce_power     = CFG.SIM_ENFORCE_POWER,
        power_grace_ticks = CFG.POWER_GRACE_TICKS if STARTUP_FREE else 0,
        min_live_fraction = CFG.SIM_MIN_LIVE_FRACTION,
    )
    if verbose:
        print(f"  [SIM] {label}: {len(result['placed'])} machines, {len(belts)} belts, "
              f"{len(injection)} boundary-fed machines")
    eq_tick = engine.run(max_ticks=max_ticks, window=eq_window,
                         threshold=eq_threshold, verbose=verbose)

    if verbose:
        rev = engine.simulated_revenue_per_tick(window=eq_window)
        busy = sum(1 for st in engine.states.values()
                   if st.status_history and 'running' in st.status_history)
        starved = sum(1 for st in engine.states.values()
                      if st.status_history and st.status_history[-1] == 'starved')
        print(f"  [SIM] {label}: ran={busy}/{len(engine.states)} machines, "
              f"starved at end={starved}, sim revenue/tick={rev:.3f}, eq_tick={eq_tick}")
        print(f"  [SIM] {label}: live_fraction={engine.live_fraction(eq_window):.0%} "
              f"(needs >= {CFG.SIM_MIN_LIVE_FRACTION:.0%} for equilibrium)")
        # Analytic ledger beside the SIMULATED per-item throughput. A static
        # rate the simulation does not reproduce shows up here as a specific
        # material at 0%, rather than as two totals that disagree.
        try:
            _tp = engine.throughput_at_equilibrium(window=eq_window)
            production_report(result["placed"], result["routed"],
                              title=f"{label}: analytic vs simulated",
                              indent="  ", simulated=_tp, sim_ticks=eq_window)
        except Exception as _e:
            print(f"  [SIM] {label}: per-item comparison unavailable ({_e})")
        if engine.enforce_power:
            print(f"  [SIM] {label}: power demand={engine.power_demand_per_tick}/tick, "
                  f"grace={engine.power_grace_ticks}t, "
                  f"brownout ticks={engine.unpowered_ticks}/{engine.tick_count}, "
                  f"pool at end={engine.power_pool:.0f}")
    return engine, eq_tick


def diagnose_simulation(engine, eq_tick, analytic_rate=None, window=None):
    """Why a simulated layout produced what it produced.

    Phase 5's agreement check only ever compared EQUILIBRIUM TIMING: the static
    Gate 2 (`chain_time <= SIM_EQ_MAX_TICKS`) against the simulated eq_tick. A
    layout can satisfy that and still produce nothing at all. Observed on a real
    run: two layouts reported analytic rates of 7.0000 and 6.2000 against 0.000
    simulated revenue/tick, and Phase 5 printed "[OK] ... agree on every
    simulated layout".

    Returns the evidence plus a single `cause`:

      "none"        the simulation reproduces the analytical claim
      "power"       brownouts dominate -- the factory cannot pay its own draw
      "no-supply"   nothing is injected at the boundary; the chain has no root
      "starvation"  machines are fed too slowly, or never run at all
      "throughput"  it produces, but far below what the analytic model claims

    `brownout_fraction` is measured against POST-GRACE ticks. Power is free for
    the first POWER_GRACE_TICKS by construction, so including them understates
    the failure: a layout browned out on every tick it is actually charged for
    reads 1.00 here, not 0.63.
    """
    window = CFG.SIM_EQ_WINDOW if window is None else window
    n = max(1, len(engine.states))
    sim_rev = engine.simulated_revenue_per_tick(window=window)

    ran = sum(1 for st in engine.states.values()
              if st.status_history and 'running' in st.status_history)
    starved_end = sum(1 for st in engine.states.values()
                      if st.status_history and st.status_history[-1] == 'starved')
    live = engine.live_fraction(window)

    post_grace = max(1, engine.tick_count - (engine.power_grace_ticks
                                             if engine.enforce_power else 0))
    brownout_fraction = (engine.unpowered_ticks / post_grace
                         if engine.enforce_power else 0.0)

    shortfall = (sim_rev <= 1e-9 or
                 (analytic_rate is not None and analytic_rate > 1e-9 and
                  sim_rev < CFG.SIM_AGREE_FRACTION * analytic_rate))

    cause = "none"
    if shortfall:
        if engine.enforce_power and brownout_fraction >= CFG.SIM_BROWNOUT_FRACTION:
            cause = "power"
        elif not engine.injection_map:
            cause = "no-supply"
        elif sim_rev <= 1e-9 or starved_end or ran < n:
            cause = "starvation"
        else:
            cause = "throughput"

    return dict(sim_rev=sim_rev, produced=sim_rev > 1e-9, cause=cause,
                ran=ran, machines=n, never_ran=n - ran, starved_end=starved_end,
                live_fraction=live, eq_tick=eq_tick,
                brownout_ticks=engine.unpowered_ticks, ticks=engine.tick_count,
                post_grace_ticks=post_grace, brownout_fraction=brownout_fraction,
                power_pool=engine.power_pool,
                power_demand=engine.power_demand_per_tick,
                injected=len(engine.injection_map),
                analytic_rate=analytic_rate)


def explain_simulation(diag, label=""):
    """One-line human-readable verdict for a diagnose_simulation() dict."""
    if diag["cause"] == "none":
        return f"{label}consistent (sim {diag['sim_rev']:.3f}/tick)"
    if diag["cause"] == "power":
        return (f"{label}POWER COLLAPSE -- browned out "
                f"{diag['brownout_ticks']}/{diag['post_grace_ticks']} post-grace ticks "
                f"({diag['brownout_fraction']:.0%}), demand {diag['power_demand']:.0f}/tick, "
                f"pool {diag['power_pool']:.0f} at end")
    if diag["cause"] == "no-supply":
        return (f"{label}NO BOUNDARY SUPPLY -- 0 machines are injected, so the "
                f"production chain has no root")
    if diag["cause"] == "starvation":
        return (f"{label}STARVATION -- {diag['never_ran']}/{diag['machines']} machines "
                f"never ran, {diag['starved_end']} starved at the end, "
                f"live_fraction {diag['live_fraction']:.0%}")
    return (f"{label}UNDER-PRODUCTION -- sim {diag['sim_rev']:.3f}/tick against an "
            f"analytic {diag['analytic_rate']:.3f}")


print("SimEngine defined.")
print("simulate_layout() defined -- bridges a v4 result dict into the simulator.")
print("Methods: tick(), run(), is_equilibrium(), throughput_at_equilibrium(),")
print("         simulated_revenue_per_tick(), pipeline_latencies()")






In [ ]:
# ─── NSGA-II SETUP — recipe index and reproducible per-genotype seeds ──────────
# (all tunable parameters now live in the Control Panel, section 1.5)
# Colab/Jupyter compatibility. The comment must not share the line with
# the magic -- pip parses it as part of the requirement and every run
# printed `ERROR: Invalid requirement: '#'`.
!pip install deap
import math
import random
from collections import defaultdict, deque
from dataclasses import dataclass
from deap import base, creator, tools, algorithms
import numpy as np


def individual_repair_seed(individual) -> int:
    """Deterministic seed derived from an individual's genes.

    The ZPF disruption-recovery search is stochastic (destroy-operator choice,
    swap indices, SA acceptance). Without this, re-running it on the *same*
    genotype at a later point (e.g. re-checking the Pareto front after the GA
    has finished) can land on a different, possibly worse, recovery purely by
    chance, even though nothing about the individual changed. Seeding a
    private RNG from a hash of the genotype makes recovery a deterministic
    function of the genes.
    """
    key = tuple(round(v, 6) if isinstance(v, float) else v for v in individual)
    return hash((CFG.RANDOM_SEED, key)) & 0xFFFFFFFF


def _chromosome_repair_seed(chromosome) -> int:
    """Deterministic seed derived from a chromosome's own gene content (see
    individual_repair_seed above for why this matters). evaluate_layout() is
    called more than once on an identical chromosome object in the common
    case — once directly in evaluate_nsga2, and again inside zpf_recovery as
    its disruption-free baseline unless a precomputed `base` is passed in —
    so keying the seed off the gene content itself, rather than plumbing an
    rng through every caller, keeps every one of those calls reproducible
    without changing evaluate_layout's external signature.
    """
    # zlib.crc32 over a stable string, NOT hash(). Python salts string
    # hashing per process (PYTHONHASHSEED), so hash() on a tuple containing
    # machine_id strings returned a DIFFERENT value in every interpreter --
    # the seed was deterministic within one run and different across runs,
    # which is exactly what this function exists to prevent. Verified: the
    # same chromosome produced seeds 3419549971 / 3674244194 / 340575188 and
    # three different layouts in three consecutive processes.
    import zlib
    key = "|".join(f"{g.machine_id},{g.recipe_idx},{g.rotation.value}"
                   for g in chromosome.genes)
    return zlib.crc32(f"{CFG.RANDOM_SEED}|{key}".encode()) & 0xFFFFFFFF


# We extract all recipes from MACHINE_CATALOGUE as (machine_id, recipe_idx)
ALL_RECIPES = []
for m_id, m_type in MACHINE_CATALOGUE.items():
    if m_type.recipes:
        for r_idx in range(len(m_type.recipes)):
            ALL_RECIPES.append((m_id, r_idx))
TARGET_PRODUCTS = [
    "buck_cap_A", "can_cit_A", "buck_cap_B", "can_cit_B",
    "HC_bat", "SC_bat", "LC_bat"
]
print("DEAP NSGA-II initialized (v4, standalone).")


In [ ]:
def _strongly_connected(nodes, succ):
    """Tarjan's SCC, iterative so deep chains cannot blow the Python stack.

    Returns a list of components, each a list of node ids. A node with no cycle
    through it comes back as a component of one.
    """
    index, low, on_stack = {}, {}, set()
    stack, out, counter = [], [], [0]

    for root in nodes:
        if root in index:
            continue
        index[root] = low[root] = counter[0]
        counter[0] += 1
        stack.append(root)
        on_stack.add(root)
        work = [(root, iter(succ.get(root, ())))]
        while work:
            v, it = work[-1]
            descended = False
            for w in it:
                if w not in index:
                    index[w] = low[w] = counter[0]
                    counter[0] += 1
                    stack.append(w)
                    on_stack.add(w)
                    work.append((w, iter(succ.get(w, ()))))
                    descended = True
                    break
                if w in on_stack:
                    low[v] = min(low[v], index[w])
            if descended:
                continue
            work.pop()
            if work:
                low[work[-1][0]] = min(low[work[-1][0]], low[v])
            if low[v] == index[v]:
                comp = []
                while True:
                    w = stack.pop()
                    on_stack.discard(w)
                    comp.append(w)
                    if w == v:
                        break
                out.append(comp)
    return out


def _chain_stats(placed, routed):
    """Critical path through the production graph, and its irreducible part.

    Returns dict(time, ideal, cyclic):

      time   -- longest path, counting each machine's process_time and each
                belt's travel (len(path)-1 cells at BELT_SPEED).
      ideal  -- the SAME path with every belt treated as instantaneous. This is
                the part of the critical path the layout cannot do anything
                about: it is set by the recipe chain, not by where the machines
                ended up. `time - ideal` is the layout's own contribution.
      cyclic -- whether any strongly connected component has more than one
                machine in it.

    CYCLES. The previous version ran Kahn's algorithm, then "resolved" whatever
    was left over (i.e. every node on a cycle) by relaxing it
    `min(len(remaining), 10)` times with no convergence test. `arrival` grows
    monotonically along a cycle and never reaches a fixed point, so the reported
    chain_time for any cyclic layout was a function of that iteration count and
    nothing else -- change the 10 and the sustain gate changes.

    A longest path is not defined on a cyclic graph, so the cycle is contracted
    instead: each strongly connected component becomes one node whose cost is
    the sum of its members' process times plus the belt travel of its internal
    edges -- one lap of the loop -- and the longest path is taken over the
    condensation, which is a DAG by construction. That is well-defined,
    iteration-count-independent, and reduces to the old answer on acyclic
    layouts.
    """
    proc = {p.instance_id: (p.machine_type.recipes[p.active_recipe].process_time
            if p.active_recipe is not None and p.machine_type.recipes
            and p.active_recipe < len(p.machine_type.recipes) else 0)
            for p in placed}
    nodes = [p.instance_id for p in placed]
    known = set(nodes)

    edges = []
    succ = defaultdict(list)
    for fp, path in routed:
        if not path:
            continue
        if fp.src_id not in known or fp.dst_id not in known:
            continue
        edges.append((fp.src_id, fp.dst_id, (len(path) - 1) * BELT_SPEED))
        succ[fp.src_id].append(fp.dst_id)

    comps = _strongly_connected(nodes, succ)
    comp_of = {n: i for i, c in enumerate(comps) for n in c}

    weight = [float(sum(proc.get(n, 0) for n in c)) for c in comps]
    weight_ideal = list(weight)
    dag = {}                                   # (a, b) -> longest belt on it
    for s, d, w in edges:
        a, b = comp_of[s], comp_of[d]
        if a == b:
            weight[a] += w                     # internal to the loop
        elif dag.get((a, b), -1) < w:
            dag[(a, b)] = w

    preds = defaultdict(list)
    indeg = [0] * len(comps)
    for (a, b), w in dag.items():
        preds[b].append((a, w))
        indeg[b] += 1

    order, q = [], deque(i for i in range(len(comps)) if indeg[i] == 0)
    while q:
        c = q.popleft()
        order.append(c)
        for (a, b) in dag:
            if a == c:
                indeg[b] -= 1
                if indeg[b] == 0:
                    q.append(b)

    arrival, arrival_ideal = [0.0] * len(comps), [0.0] * len(comps)
    for c in order:
        base = max((arrival[a] + w for a, w in preds[c]), default=0.0)
        base_i = max((arrival_ideal[a] for a, _ in preds[c]), default=0.0)
        arrival[c] = weight[c] + base
        arrival_ideal[c] = weight_ideal[c] + base_i

    return dict(time=max(arrival, default=0.0),
                ideal=max(arrival_ideal, default=0.0),
                cyclic=any(len(c) > 1 for c in comps))


def _chain_time(placed, routed):
    """Critical-path length in ticks. Thin wrapper over _chain_stats."""
    return _chain_stats(placed, routed)["time"]


def _sustain_budget(chain_ideal):
    """Tick budget the sustain gate scores a layout against.

    Flat SIM_EQ_MAX_TICKS is what made the gate reward destruction: the critical
    path grows with the machines on it, so removing machines always improved the
    ratio. Scaling the budget with the layout's own irreducible processing time
    makes the gate measure the layout's contribution -- belt travel, detours,
    congestion -- which is the thing recovery can actually change.
    See CFG.SUSTAIN_NORMALISE.
    """
    if CFG.SUSTAIN_NORMALISE and chain_ideal > 0:
        return CFG.SUSTAIN_SLACK * chain_ideal
    return float(SIM_EQ_MAX_TICKS)

def _power_output(placed, routed=None):
    """Total power GENERATION of a layout, in the same units as sum(power_cost).

    A thermal bank is a GENERATOR RATED at its recipe's power output while it is
    burning fuel -- CFG.BATTERY_CAPACITY is documented as "power supplied per
    Thermal Bank", and one HC_bat sustains 1100 for the 100 ticks of the burn.
    power_cost is likewise a rated draw, so `output > demand` compares two
    ratings and is dimensionally correct as originally written.

    An earlier revision of this notebook divided the output by process_time on
    the assumption that 1100 was an energy batch (as an output quantity means
    for every other recipe). That was wrong -- power is not an item and never
    rides a belt -- and it turned one correctly-sized bank into seventeen: they
    could not all be fuelled through main_unit's six output ports, so 16 of 17
    sat unconnected and overall routing fell from 14/27 to 15/43. Set
    CFG.POWER_RATE_CORRECT=True to restore that behaviour for comparison.

    An unfuelled bank contributes NOTHING. `routed` supplies the evidence: a
    bank counts only if some successfully routed flow delivers one of its
    recipe inputs. find_unconnected_ther_banks() has always been able to spot
    these, but no gate ever consulted it, so a bank with no belt at all still
    "supplied" its full rating -- which is the main reason Gate 1 could not
    fail. Pass routed=None to skip the check (rated capacity only).
    """
    fuelled = None
    if routed is not None:
        fuelled = {(fp.dst_id, fp.material) for fp, path in routed if path is not None}

    total = 0.0
    for p in placed:
        if p.machine_type.id != "ther_bank" or p.active_recipe is None:
            continue
        if p.active_recipe >= len(p.machine_type.recipes):
            continue
        rec = p.machine_type.recipes[p.active_recipe]
        if fuelled is not None:
            needs = [m for m, _ in rec.inputs]
            if needs and not any((p.instance_id, m) in fuelled for m in needs):
                continue                      # generator with no fuel belt
        gen = sum(q for m, q in rec.outputs if m == "power")
        total += gen / max(1, rec.process_time) if CFG.POWER_RATE_CORRECT else gen
    return total


def _calculate_revenue(placed, routed, breakdown=False):
    """Revenue per tick, discounted by each machine's input-supply efficiency.

    With `breakdown=True` returns `(revenue, ledger)` instead of a bare float.
    The ledger is the same set of accumulators this function already builds, so
    it sums to `revenue` by construction -- it is not a second calculation that
    could disagree with the first. See production_report() for the printer.

    Efficiency is evaluated against the machine's RECIPE REQUIREMENTS, not
    against whichever input flows happen to survive. Previously it took min()
    over the surviving flows, so destroying a supplier REMOVED that input from
    the minimum and the consumer's efficiency went UP -- which is why bombing a
    layout could raise its reported rate and drive regret to 0.0000. A required
    input that no routed flow supplies now always scores CFG.STARVED_INPUT_EFF,
    whether the flow is unrouted or absent entirely.
    """
    in_nodes, out_nodes = defaultdict(list), defaultdict(list)
    all_nodes = set()
    incoming = defaultdict(lambda: defaultdict(list))     # dst -> material -> [(src, routed)]

    for fp, path in routed:
        in_nodes[fp.dst_id].append(fp.src_id)
        out_nodes[fp.src_id].append(fp.dst_id)
        all_nodes.update([fp.src_id, fp.dst_id])
        incoming[fp.dst_id][fp.material].append((fp.src_id, path is not None))

    for pm in placed:
        all_nodes.add(pm.instance_id)

    indeg = {n: len(in_nodes[n]) for n in all_nodes}
    queue = deque(n for n in all_nodes if indeg[n] == 0)
    topo = []
    while queue:
        node = queue.popleft()
        topo.append(node)
        for dst in out_nodes[node]:
            indeg[dst] -= 1
            if indeg[dst] == 0:
                queue.append(dst)
    topo += [n for n in all_nodes if n not in set(topo)]   # cyclic remainder

    lookup = {pm.instance_id: pm for pm in placed}
    efficiency = {pm.instance_id: 1.0 for pm in placed}
    for node in topo:
        pm = lookup.get(node)
        if pm is None or pm.active_recipe is None or not pm.machine_type.recipes:
            continue
        if pm.active_recipe >= len(pm.machine_type.recipes):
            continue
        rec = pm.machine_type.recipes[pm.active_recipe]
        needs = {m for m, _ in list(rec.inputs) + list(rec.common_extra)}
        if not needs:
            continue
        min_eff = 1.0
        for material in needs:
            best = CFG.STARVED_INPUT_EFF    # nothing supplies it at all
            for src, ok in incoming[node].get(material, []):
                cand = efficiency.get(src, 1.0) * (1.0 if ok else CFG.STARVED_INPUT_EFF)
                if cand > best:
                    best = cand
            if best < min_eff:
                min_eff = best
        efficiency[node] = min_eff

    # P2: how output is credited. See CFG.REVENUE_MODE for why "net" is the
    # default and what the alternatives cost.
    mode = getattr(CFG, "REVENUE_MODE", "net")
    shipped = None
    if mode == "sink":
        _sink_ids = {pm.instance_id for pm in placed
                     if pm.machine_type.id in IO_SINK_IDS}
        shipped = {(fp.src_id, fp.material) for fp, path in routed
                   if path is not None and fp.dst_id in _sink_ids}

    produced, consumed = defaultdict(float), defaultdict(float)
    credited = defaultdict(float)          # revenue actually booked, per material
    makers   = defaultdict(set)            # material -> machine types producing it
    revenue = 0.0
    for pm in placed:
        if pm.active_recipe is not None and pm.machine_type.recipes:
            if pm.active_recipe >= len(pm.machine_type.recipes):
                continue
            r = pm.machine_type.recipes[pm.active_recipe]
            eff = efficiency.get(pm.instance_id, 1.0)
            ticks = max(1.0, r.process_time)
            # Accumulated for EVERY mode now, not just "net". Under gross/sink
            # they do not feed the revenue sum -- they exist so the ledger can
            # still show what the factory makes and eats.
            for out_name, out_qty in r.outputs:
                produced[out_name] += (out_qty / ticks) * eff
                makers[out_name].add(pm.machine_type.id)
            for in_name, in_qty in list(r.inputs) + list(r.common_extra):
                consumed[in_name] += (in_qty / ticks) * eff
            if mode == "net":
                # Discounted by the same efficiency on both sides: a starved
                # machine neither produces nor draws at its rated rate.
                continue
            for out_name, out_qty in r.outputs:
                if shipped is not None and (pm.instance_id, out_name) not in shipped:
                    continue
                if out_name in REVENUE_TABLE:
                    _v = REVENUE_TABLE[out_name] * (out_qty / ticks) * eff
                    credited[out_name] += _v
                    revenue += _v

    if mode == "net":
        for material, rate in produced.items():
            if material in REVENUE_TABLE:
                _v = REVENUE_TABLE[material] * max(0.0, rate - consumed[material])
                credited[material] += _v
                revenue += _v

    if not breakdown:
        return revenue

    _recipe_ids = [pm.instance_id for pm in placed
                   if pm.active_recipe is not None and pm.machine_type.recipes]
    _effs = [efficiency.get(i, 1.0) for i in _recipe_ids]
    ledger = dict(
        mode=mode, revenue=revenue,
        produced=dict(produced), consumed=dict(consumed), credited=dict(credited),
        makers={k: sorted(v) for k, v in makers.items()},
        efficiency=dict(efficiency),
        n_recipe_machines=len(_recipe_ids),
        n_full=sum(1 for e in _effs if e > 0.999),
        n_starved=sum(1 for e in _effs if e <= CFG.STARVED_INPUT_EFF * 10),
        mean_efficiency=(sum(_effs) / len(_effs)) if _effs else 0.0,
        worst_machines=sorted(
            ((efficiency.get(pm.instance_id, 1.0), pm.instance_id) for pm in placed
             if pm.active_recipe is not None and pm.machine_type.recipes),
            key=lambda t: t[0])[:6],
    )
    return revenue, ledger

def _decode_instances(instances, rng=None):
    """Shared BLF-decode-and-route helper used by evaluate_layout and both
    repair passes below, so they all measure a candidate layout identically.

    `rng` is threaded into blf_decode so the decode is reproducible; previously
    the decoder drew from the global random module regardless of what seeded
    rng the caller had constructed."""
    grid = FactoryGrid()
    chromo = Chromosome([Gene(m.id, Rotation.R0, r) for m, r, _, _ in instances])
    placed = blf_decode(chromo, grid, rng=rng)
    flows = derive_flow_matrix(placed)
    routed = route_belts(grid, {p.instance_id: p for p in placed}, flows)
    
    # MAGIC DEPOTS: Apply repair_feasibility to spawn depots/stashes for any starved
    # connections so the main GA loop can actually find feasible layouts.
    rep_count, rep_machines, rep_belts = repair_feasibility(
        grid, placed, flows, routed,
        battery_item=None, verbose=False)
        
    if rep_machines:
        # repair_feasibility no longer appends to `placed` itself, so this is
        # now the single place each magic depot enters the list (it used to be
        # added in both places, double-counting every depot).
        placed.extend(rep_machines)
        
    if rep_belts:
        flows, routed = apply_repair_belts(placed, flows, routed, rep_belts)
            
    return grid, placed, flows, routed


def alns_macro_repair(sorted_instances, max_iters=None, destroy_prob=None, rng=None):
    """Destroy-and-reinsert repair that reduces the unplaced-machine count
    AND actively favours orderings that route more flows, not just ones that
    place more machines. Ported from the v3-architecture notebook (already
    fixed there: private seeded rng instead of the bare global `random`
    module). Returns (instances, is_feasible, grid, placed, flows, routed)
    where is_feasible means "fully placed and fully routed" — a
    placement/routing-only check, not the full 4-gate power/sustain
    feasibility evaluate_layout computes.
    """
    max_iters = CFG.ALNS_MAX_ITERS if max_iters is None else max_iters
    destroy_prob = CFG.ALNS_DESTROY_PROB if destroy_prob is None else destroy_prob
    rng = rng if rng is not None else random

    if not sorted_instances:
        return sorted_instances, True, FactoryGrid(), [], [], []

    best_instances = sorted_instances[:]
    best_grid, best_placed, best_fm, best_routes = _decode_instances(best_instances, rng=rng)
    best_paths = sum(1 for _, p in best_routes if p is not None)
    best_unplaced = len(best_instances) - len(best_placed)

    if best_unplaced == 0 and best_paths == len(best_fm):
        return best_instances, True, best_grid, best_placed, best_fm, best_routes

    for _ in range(max_iters):
        destroyed, kept = [], []
        for inst in best_instances:
            (destroyed if rng.random() < destroy_prob else kept).append(inst)
        if not destroyed:
            continue
        candidate = kept[:]
        for d_inst in destroyed:
            candidate.insert(rng.randint(0, len(candidate)), d_inst)

        cand_grid, cand_placed, cand_fm, cand_routes = _decode_instances(candidate, rng=rng)
        cand_paths = sum(1 for _, p in cand_routes if p is not None)
        cand_unplaced = len(candidate) - len(cand_placed)

        # Placement takes priority (unplaced machines fail the hardest gate),
        # but once placement is at least as good, ANY improvement in routed
        # paths is kept — not just when placement is *exactly* tied at its
        # best-ever value. Routing at this scale is the harder, more common
        # bottleneck, so it needs its own gradient rather than riding along
        # as a tiebreaker that rarely triggers.
        better = (cand_unplaced < best_unplaced or
                  (cand_unplaced <= best_unplaced and cand_paths > best_paths))
        if better:
            best_instances, best_unplaced, best_paths = candidate, cand_unplaced, cand_paths
            best_grid, best_placed, best_fm, best_routes = cand_grid, cand_placed, cand_fm, cand_routes

        if best_unplaced == 0 and best_paths == len(best_fm):
            return best_instances, True, best_grid, best_placed, best_fm, best_routes

    is_feasible = (best_unplaced == 0 and best_paths == len(best_fm) and len(best_fm) > 0)
    return best_instances, is_feasible, best_grid, best_placed, best_fm, best_routes


def tabu_micro_polish(sorted_instances, max_iters=None, tabu_list_size=None, rng=None):
    """MHC-improving swap search, run only once placement/routing are already
    complete. Ported from the v3-architecture notebook."""
    max_iters = CFG.TABU_MAX_ITERS if max_iters is None else max_iters
    tabu_list_size = CFG.TABU_LIST_SIZE if tabu_list_size is None else tabu_list_size
    rng = rng if rng is not None else random

    best_instances = sorted_instances[:]
    best_grid, best_placed, best_fm, best_routes = _decode_instances(best_instances, rng=rng)
    if len(best_instances) < 2:
        return best_instances, best_grid, best_placed, best_fm, best_routes

    best_mhc = sum(len(p) for _, p in best_routes if p is not None)
    tabu_list = []

    for _ in range(max_iters):
        idx1, idx2 = rng.randrange(len(best_instances)), rng.randrange(len(best_instances))
        if idx1 == idx2:
            continue
        swap_pair = tuple(sorted([idx1, idx2]))
        if swap_pair in tabu_list:
            continue

        candidate = best_instances[:]
        candidate[idx1], candidate[idx2] = candidate[idx2], candidate[idx1]
        cand_grid, cand_placed, cand_fm, cand_routes = _decode_instances(candidate, rng=rng)

        if len(cand_placed) < len(candidate):
            tabu_list.append(swap_pair); continue
        cand_paths = sum(1 for _, p in cand_routes if p is not None)
        if cand_paths < len(cand_fm):
            tabu_list.append(swap_pair); continue

        cand_mhc = sum(len(p) for _, p in cand_routes if p is not None)
        if cand_mhc < best_mhc:
            best_instances, best_mhc = candidate, cand_mhc
            best_grid, best_placed, best_fm, best_routes = cand_grid, cand_placed, cand_fm, cand_routes
            tabu_list.append(swap_pair)
            if len(tabu_list) > tabu_list_size:
                tabu_list.pop(0)

    return best_instances, best_grid, best_placed, best_fm, best_routes


def connect_or_remove_unlinked(grid, placed, flows, routed, rng=None, max_attempts=None, max_passes=None):
    """Final targeted cleanup for machines ALNS/Tabu couldn't fully connect.

    Stage 1 (connect): for every unrouted flow, retries A* across multiple
    (output-port, input-port) combinations on the SAME two machines, not
    just the one derive_flow_matrix originally assigned -- a different port
    pair often opens up a path the assigned one couldn't find, especially in
    congested layouts.

    Stage 2 (remove): any machine that STILL has zero successful connections
    afterward is removed outright -- same convention cull_unlinked_machines
    uses elsewhere in this project (main_unit and power sources exempt).
    This is a machine-level check: a machine with *some* working connections
    survives even if one of its other flows never routes -- it removes
    fully-isolated dead weight, it does not guarantee every surviving
    machine is 100% routed.

    IMPORTANT: this never calls derive_flow_matrix again. Re-deriving the
    supply/demand matching from scratch on a reduced machine set reshuffles
    ALL pairings (it's a greedy, order-dependent match), which can orphan
    machines that were working fine and cascade into removing far more than
    intended. Instead, flows touching a removed machine are simply dropped
    from the existing, already-computed flow/routed lists -- every other
    pairing is left exactly as ALNS/Tabu found it.

    Repeats a few passes since removing one machine can orphan another that
    depended on it. Returns (grid, placed, flows, routed, n_removed).
    """
    rng = rng if rng is not None else random
    max_attempts = CFG.RECONNECT_MAX_ATTEMPTS if max_attempts is None else max_attempts
    max_passes = CFG.RECONNECT_MAX_PASSES if max_passes is None else max_passes

    total_removed = 0
    current_routed = routed

    for _ in range(max_passes):
        placed_lookup = {p.instance_id: p for p in placed}

        # Stage 1: retry unrouted flows across several port combinations.
        patched_routed = []
        for fp, path in current_routed:
            if path is not None:
                patched_routed.append((fp, path))
                continue
            src_pm, dst_pm = placed_lookup.get(fp.src_id), placed_lookup.get(fp.dst_id)
            found_path = None
            if src_pm and dst_pm:
                outs, ins = src_pm.output_belt_points(), dst_pm.input_belt_points()
                port_pairs = [(o, i) for o in range(len(outs)) for i in range(len(ins))]
                rng.shuffle(port_pairs)
                for oi, ii in port_pairs[:max_attempts]:
                    candidate = astar(grid, outs[oi], ins[ii])
                    if candidate and grid.place_belt(candidate, f"{fp.src_id}->{fp.dst_id}"):
                        found_path = candidate
                        break
            patched_routed.append((fp, found_path))

        # Stage 2: any machine with zero successful connections is removed.
        connected_ids = set()
        for fp, path in patched_routed:
            if path is not None:
                connected_ids.add(fp.src_id)
                connected_ids.add(fp.dst_id)

        to_remove = [pm for pm in placed
                     if pm.machine_type.id != "main_unit" and not pm.machine_type.is_power_src
                     and pm.instance_id not in connected_ids]

        if not to_remove:
            return grid, placed, [fp for fp, _ in patched_routed], patched_routed, total_removed

        total_removed += len(to_remove)
        remove_ids = {pm.instance_id for pm in to_remove}
        for pm in to_remove:
            grid.remove_machine(pm)
        placed = [pm for pm in placed if pm.instance_id not in remove_ids]
        # Drop only flows that touched a removed machine; every surviving
        # pairing -- routed or still unrouted -- carries over unchanged.
        current_routed = [(fp, path) for fp, path in patched_routed
                           if fp.src_id not in remove_ids and fp.dst_id not in remove_ids]

    flows = [fp for fp, _ in current_routed]
    return grid, placed, flows, current_routed, total_removed


def evaluate_layout(chromosome, rng=None, repair=False, alns_iters=None, tabu_iters=None):
    # Reconstruct an ALNS-compatible instance list straight from the
    # chromosome's genes, in their existing (priority-sorted) order — ALNS
    # only ever reads the machine type and recipe index out of each tuple,
    # so the placeholder priority/orig_idx fields here are never read back.
    instances = [(MACHINE_CATALOGUE[g.machine_id], g.recipe_idx, 0.0, i)
                 for i, g in enumerate(chromosome.genes)]

    # Seeded deterministically from the chromosome's own gene content (see
    # _chromosome_repair_seed): makes repair a pure function of the layout,
    # so calling evaluate_layout more than once on an identical chromosome
    # always reproduces the same repair instead of drifting across calls.
    rng = rng if rng is not None else random.Random(_chromosome_repair_seed(chromosome))

    # Previously a single un-repaired blf_decode() — anything that didn't
    # fit on the first greedy pass was just dropped, with nothing retrying
    # to rescue it. ALNS actively destroys-and-reinserts to close that gap,
    # then Tabu polishes MHC once placement/routing are complete.
    if repair:
        alns_iters = alns_iters or CFG.ALNS_MAX_ITERS
        tabu_iters = tabu_iters or CFG.TABU_MAX_ITERS
        repaired_instances, is_feasible, grid, placed, flows, routed = alns_macro_repair(
            instances, max_iters=alns_iters, destroy_prob=CFG.ALNS_DESTROY_PROB, rng=rng)
        if is_feasible:
            repaired_instances, grid, placed, flows, routed = tabu_micro_polish(
                repaired_instances, max_iters=tabu_iters, tabu_list_size=CFG.TABU_LIST_SIZE, rng=rng)
    else:
        # blf_decode() takes (chromosome, grid) and returns just `placed` --
        # it doesn't accept a raw instances list or return this 4-tuple.
        # _decode_instances (defined above, already used by alns_macro_repair
        # and tabu_micro_polish) does exactly this decode+derive+route
        # sequence and returns the right shape; this fast path just needed
        # to call it instead. As written, this branch raised a TypeError on
        # the very first evaluation -- i.e. every call with the (default)
        # repair=False, which is what the main GA loop uses.
        grid, placed, flows, routed = _decode_instances(instances, rng=rng)

    # all_placed tracks raw ALNS/BLF placement success -- did every gene make
    # it onto the grid at all -- BEFORE any deliberate pruning below. That
    # stays a hard requirement: it's what ALNS is specifically fighting for.
    all_placed = len(placed) == len(chromosome.genes)

    # Final cleanup: anything ALNS placed but never got connected to
    # anything gets one more targeted reconnect attempt, then removed if
    # that still fails. This can only IMPROVE routing/power/sustain (fewer,
    # better-connected machines) at the cost of building less than the
    # original gene count -- tracked explicitly via machines_removed rather
    # than silently changing what "feasible" means.
    # Flow demand as derived, BEFORE connect_or_remove_unlinked deletes any of
    # it. Pruning a machine also drops its flows, which made an unroutable
    # connection disappear from the routing denominator and reappear as a
    # PLACEMENT failure -- measured: all_placed=True while the placement gate
    # read 0.96, with blf_decode having placed 100% of machines.
    flows_demanded = len(flows)

    machines_removed = 0
    if repair or CFG.PRUNE_UNCONNECTABLE:
        grid, placed, flows, routed, machines_removed = connect_or_remove_unlinked(
            grid, placed, flows, routed, rng=rng, max_passes=10 if repair else None)

    paths_found = sum(path is not None for _, path in routed)

    # MHC: routed flows count normally. An UNROUTED flow is NOT ignored --
    # it costs a distance-based penalty (2x Manhattan distance between the
    # two machines, or a flat grid.width+grid.height if either machine is
    # somehow missing) instead. This mirrors the v3-architecture notebook's
    # actual evaluate_fitness exactly, and matters a lot: it's what gives
    # the GA a real gradient to minimise unrouted flows even now that
    # routing completeness is no longer a hard feasibility gate (below) --
    # without it, an unrouted flow would be free, and nothing would push
    # the search toward closing the gap.
    placed_lookup = {p.instance_id: p for p in placed}
    mhc = 0.0
    for fp, path in routed:
        if path is not None:
            mhc += fp.intensity * (len(path) - 1)
        else:
            src_pm, dst_pm = placed_lookup.get(fp.src_id), placed_lookup.get(fp.dst_id)
            if src_pm and dst_pm:
                dist = (abs(src_pm.x + src_pm.w // 2 - dst_pm.x - dst_pm.w // 2) +
                        abs(src_pm.y + src_pm.h // 2 - dst_pm.y - dst_pm.h // 2))
                mhc += fp.intensity * dist * 2
            else:
                mhc += fp.intensity * (GRID_W + GRID_H)

    build_cost = sum(sum(q for _, q in p.machine_type.build_cost) for p in placed)
    demand = sum(p.machine_type.power_cost for p in placed)
    output = _power_output(placed, routed)

    power_ok = output > demand
    _cs = _chain_stats(placed, routed)
    chain_time, chain_ideal = _cs["time"], _cs["ideal"]
    sustain_ok = chain_time <= _sustain_budget(chain_ideal)

    # Feasibility now matches what the v3-architecture notebook's
    # evaluate_fitness ACTUALLY checks (verified directly in its source):
    # `feasible = power_ok and sustain_ok and all_placed` -- routing
    # completeness was never part of its gate either. Unrouted flows are
    # costed via the MHC penalty above instead of blocking feasibility
    # outright, which is exactly why v3 reports "feasible" layouts readily
    # even when routing isn't 100% complete. Set CFG.REQUIRE_FULL_ROUTING
    # to restore the previous, stricter bar this notebook used before.
    feasible = all_placed and power_ok and sustain_ok
    if CFG.REQUIRE_FULL_ROUTING:
        feasible = feasible and paths_found == len(flows)

    revenue = _calculate_revenue(placed, routed)
    return dict(grid=grid, placed=placed, flows=flows, routed=routed, mhc=mhc,
                build_cost=build_cost, power_ok=power_ok, chain_time=chain_time,
                chain_ideal=chain_ideal, chain_cyclic=_cs["cyclic"],
                feasible=feasible, paths_found=paths_found, flows_total=len(flows),
                flows_demanded=flows_demanded,
                demand=demand, output=output, all_placed=all_placed, revenue=revenue,
                machines_removed=machines_removed)




print("alns_macro_repair() and tabu_micro_polish() defined; evaluate_layout() now repairs before scoring.")


# ─── Where the rate comes from ───────────────────────────────────────────────
def production_report(placed, routed, title="", feasibility=None, indent="    ",
                      simulated=None, sim_ticks=None, min_rate=1e-9):
    """Print the per-item ledger behind a reported revenue figure.

    `terminal_rate = revenue x feasibility` is what gets quoted, and a bare
    float cannot be defended. This prints, per material: how much the factory
    makes per tick, how much it consumes internally, the net that leaves, the
    unit price, and the revenue actually credited -- which sums to `revenue`
    exactly, because it comes from the same accumulators.

    Pass `simulated` (a {material: units} dict from
    SimEngine.throughput_at_equilibrium) and `sim_ticks` to get the simulated
    per-tick rate in the same table. A large analytic figure sitting next to a
    simulated zero is the single most useful thing this prints: it localises the
    disagreement to a material instead of leaving it as two contradictory
    totals.
    """
    revenue, L = _calculate_revenue(placed, routed, breakdown=True)
    mats = sorted(set(L["produced"]) | set(L["consumed"]),
                  key=lambda m: (-L["credited"].get(m, 0.0), m))
    sim_rate = {}
    if simulated and sim_ticks:
        sim_rate = {m: v / max(1, sim_ticks) for m, v in simulated.items()}

    head = f"{indent}PRODUCTION" + (f" — {title}" if title else "")
    print(f"{head}   [mode={L['mode']}]")
    cols = (f"{indent}  {'material':<16}{'made/t':>9}{'used/t':>9}{'net/t':>9}"
            f"{'unit':>6}{'rev/t':>9}{'share':>8}")
    if sim_rate:
        cols += f"{'sim/t':>9}{'ratio':>8}"
    print(cols)

    shown = 0
    for m in mats:
        made = L["produced"].get(m, 0.0)
        used = L["consumed"].get(m, 0.0)
        cred = L["credited"].get(m, 0.0)
        if made < min_rate and used < min_rate:
            continue
        price = REVENUE_TABLE.get(m, 0)
        share = cred / revenue if revenue > min_rate else 0.0
        line = (f"{indent}  {m:<16}{made:>9.4f}{used:>9.4f}{made - used:>9.4f}"
                f"{price:>6}{cred:>9.4f}{share:>7.1%}")
        if sim_rate:
            sv = sim_rate.get(m, 0.0)
            # Ratio of simulated to analytic PRODUCTION, not to revenue: the
            # question is whether the factory physically makes what the static
            # model says it makes.
            line += f"{sv:>9.4f}" + (f"{sv / made:>7.0%}" if made > min_rate else f"{'—':>8}")
        print(line)
        shown += 1
    if not shown:
        print(f"{indent}  (nothing produced)")

    tail = (f"{indent}  {'TOTAL':<16}{'':>9}{'':>9}{'':>9}{'':>6}"
            f"{revenue:>9.4f}{1.0 if revenue > min_rate else 0.0:>7.1%}")
    print(tail)
    if feasibility is not None:
        print(f"{indent}  revenue {revenue:.4f} x feasibility {feasibility:.1%} "
              f"= terminal_rate {revenue * feasibility:.4f}")
    print(f"{indent}  supply: {L['n_full']}/{L['n_recipe_machines']} machines at full rate, "
          f"{L['n_starved']} starved, mean efficiency {L['mean_efficiency']:.2f}")
    if L["n_starved"]:
        worst = ", ".join(f"{i}={e:.2f}" for e, i in L["worst_machines"] if e < 0.999)
        if worst:
            print(f"{indent}  worst-supplied: {worst}")
    return revenue, L


print("production_report() defined -- per-item ledger behind every reported rate.")


In [ ]:
class ZPF:
    """Four-zone Euclidean disruption model from Thesis Chapter 3.
    
    - Crater: the w x h rectangle at (x,y). NOTHING can be placed here.
    - Free zone: circle of radius r1 from crater centre. Machines can be freely reconfigured.
    - Penalty zone: circle of radius r2 (r2 > r1). Fixed penalty per square placed here.
    - Stable zone: everything outside r2. Only belts can change; machines are frozen.
    """
    def __init__(self, x, y, w, h, r1=5.0, r2=12.0):
        self.x, self.y, self.w, self.h, self.r1, self.r2 = x, y, w, h, r1, r2
    def distance(self, x, y):
        dx = max(self.x - x, 0, x - (self.x + self.w - 1))
        dy = max(self.y - y, 0, y - (self.y + self.h - 1))
        return math.hypot(dx, dy)
    def zone(self, x, y):
        d = self.distance(x, y)
        return "unusable" if d == 0 else "flexible" if d <= self.r1 else "penalty" if d <= self.r2 else "stable"
    def machine_zone(self, pm):
        """Zone of a whole machine, decided by a MAJORITY OF ITS CELLS.

        Previously the zone of the single centre cell, so a machine straddling
        a boundary was classified on one arbitrary square and a large machine
        could be called "stable" (frozen) while most of its footprint sat
        inside the blast circle. A machine does not have to lie strictly
        inside a circle to belong to it -- it belongs when strictly more than
        half its cells are inside. For a 3x3 that is at least 5 of 9.
        """
        cells = pm.cells()
        n = len(cells)
        if n == 0:
            return "stable"
        if any(self.distance(x, y) == 0 for x, y in cells):
            return "unusable"            # any part of it sits in the crater
        if sum(1 for x, y in cells if self.distance(x, y) <= self.r1) * 2 > n:
            return "flexible"            # majority in the free circle -- no penalty
        if sum(1 for x, y in cells if self.distance(x, y) <= self.r2) * 2 > n:
            return "penalty"             # majority in the ring -- costed
        return "stable"                  # everything else is frozen


# ─── Shared post-routing repair for BOTH recovery variants ───────────────────
def _finalise_recovery(grid, placed, flows, routed, rng, max_port_pairs=20):
    """Reconnection polish + feasibility repair, applied identically by the
    NSGA-II and the SA recovery.

    This used to live inside the GA's _evaluate_individual only. The SA's
    recover() routed once and reported, so the two variants were not solving the
    same problem and the GA's feasibility advantage was partly just this code.
    A comparison of two search strategies has to hold the repair constant.

    Two steps:

    1. RECONNECTION POLISH. For every flow A*-routing failed on, retry across
       randomised (output port, input port) pairs. The default route uses one
       fixed pair, so a flow can fail purely because that particular port faced
       a wall while another port on the same machine was clear. Capped at
       `max_port_pairs` pairs -- the full product is |outs| x |ins| per flow and
       this runs inside the innermost loop of the search.

    2. FEASIBILITY REPAIR. repair_feasibility() spawns depot machines to feed
       connections nothing else supplies. Every repair belt is appended to BOTH
       `flows` and `routed`: appending only to `routed` inflates paths_found
       without moving len(flows), which drives `unrouted` negative and rewards
       the search for spawning depots instead of routing real flows.

    Returns (placed, flows, routed) -- all three are rebound, not mutated in
    place, so callers must use the return value.
    """
    placed_lookup = {p.instance_id: p for p in placed}
    patched = []
    for fp, path in routed:
        if path is not None:
            patched.append((fp, path))
            continue
        src_pm, dst_pm = placed_lookup.get(fp.src_id), placed_lookup.get(fp.dst_id)
        found = None
        if src_pm and dst_pm:
            outs, ins = src_pm.output_belt_points(), dst_pm.input_belt_points()
            port_pairs = [(o, i) for o in range(len(outs)) for i in range(len(ins))]
            rng.shuffle(port_pairs)
            for oi, ii in port_pairs[:max_port_pairs]:
                cand = astar(grid, outs[oi], ins[ii])
                if cand and grid.place_belt(cand, f"{fp.src_id}->{fp.dst_id}"):
                    found = cand
                    break
        patched.append((fp, found))
    routed = patched

    _n, rep_machines, rep_belts = repair_feasibility(
        grid, placed, flows, routed, battery_item=None, verbose=False)
    if rep_machines:
        placed.extend(rep_machines)
    flows, routed = apply_repair_belts(placed, flows, routed, rep_belts)

    return placed, flows, routed


# ─── R4: position-based recovery neighbourhood, shared by BOTH variants ──────
def _zone_of(disruption, pm):
    """machine_zone, tolerating `disruption is None` (the no-damage reference
    state, where nothing is frozen and nothing is destroyed)."""
    return "flexible" if disruption is None else disruption.machine_zone(pm)


def _zpf_place(disruption, stable, order, anchors, rotations=None):
    """Rebuild the factory after the blast.

    Stable machines go back exactly where they were. Each movable machine is
    placed at its ANCHOR if the solution gave it one, otherwise back at its
    original coordinates; either way, if the target cell is taken the search
    spirals outward from that target rather than from the crater corner, so a
    displaced machine lands as close to its intended spot as it can.

    `rotations` overrides a machine's orientation, which decides which face
    carries its I/O ports. Every chromosome in this notebook is built at
    Rotation.R0 and nothing ever varied it, so an entire dimension of the layout
    problem was fixed at zero -- see CFG.ROTATION_POLISH_TOP and the `rotate`
    operator in ZPFNeighbourhood.

    Returns (grid, placed, unplaced_ids).
    """
    rotations = rotations or {}
    grid = FactoryGrid()
    if disruption is not None:
        for y in range(disruption.y, disruption.y + disruption.h):
            for x in range(disruption.x, disruption.x + disruption.w):
                if grid.in_bounds(x, y):
                    grid.grid[y][x] = (TileState.BLOCKED, None)

    placed, unplaced = [], []
    for pm in stable:
        if grid.place_machine(pm):
            placed.append(pm)
        else:
            unplaced.append(pm.instance_id)

    # Hub first: a lost main_unit costs the factory every raw import and every
    # product sink at once, so it gets first claim on space. singleton==True
    # marks exactly this class of machine.
    order = sorted(order, key=lambda pm: not pm.machine_type.singleton)

    for old in order:
        mtype = old.machine_type
        was_destroyed = disruption is not None and disruption.machine_zone(old) == "unusable"
        anchor = anchors.get(old.instance_id)
        rot = rotations.get(old.instance_id, old.rotation)
        pw, ph = mtype.size(rot)
        home = (old.x, old.y)
        found = None

        def _mk(nx, ny):
            return type(old)(mtype, nx, ny, rot,
                             old.instance_id, old.active_recipe)

        def _admissible(cand, clearance):
            nx, ny = cand.x, cand.y
            if nx < 0 or ny < 0 or nx + pw > grid.width or ny + ph > grid.height:
                return False
            if mtype.edge_only:
                if not grid.is_on_edge(nx, ny, pw, ph):
                    return False
                # A destroyed depot loader/unloader has nowhere to go: no grid
                # edge exists inside a crater in the middle of the factory, so
                # under the r2 rule it was permanently lost. It is allowed back
                # onto any edge instead. See CFG.ZPF_EDGE_RELOCATION.
                if not CFG.ZPF_EDGE_RELOCATION and _zone_of(disruption, cand) == "stable":
                    return False
            else:
                if _zone_of(disruption, cand) == "stable":
                    return False
                # Belt connect points must be on the grid or nothing can ever
                # attach to this machine (blf_decode checks this at initial
                # placement; recovery did not).
                if not all(grid.in_bounds(px, py) for px, py in
                           cand.input_belt_points() + cand.output_belt_points()):
                    return False
            if clearance and not grid.check_clearance(nx, ny, pw, ph, clearance):
                return False
            return True

        # 1. An UNDAMAGED machine that has been neither relocated NOR rotated
        #    goes straight back where it was. This is the deterministic repair,
        #    tried first so the search can never do worse than the heuristic it
        #    used to be a wrapper around.
        if anchor is None and not was_destroyed and rot is old.rotation:
            cand = _mk(*home)
            if grid.place_machine(cand):
                found = cand

        # 2. Otherwise spiral outward from the intended position.
        if found is None:
            seed = anchor if anchor is not None else home
            r2 = disruption.r2 if disruption is not None else max(GRID_W, GRID_H)
            max_r = int(2 * r2) + max(pw, ph) + 2
            if mtype.edge_only and CFG.ZPF_EDGE_RELOCATION:
                max_r = max(GRID_W, GRID_H)      # any edge is a legal home
            for clearance in (MIN_MACHINE_SPACING, 0):
                for nx, ny in generate_spiral(seed[0], seed[1], max_r):
                    cand = _mk(nx, ny)
                    if not _admissible(cand, clearance):
                        continue
                    if grid.place_machine(cand):
                        found = cand
                        break
                if found:
                    break

        if found:
            placed.append(found)
        else:
            unplaced.append(old.instance_id)

    return grid, placed, unplaced


class ZPFNeighbourhood:
    """Destroy-and-reinsert over COORDINATES -- the standard ALNS-for-FLP
    operator set -- replacing the permutation neighbourhood both recoveries
    used to search.

    WHY IT WAS REPLACED. A solution used to be an ordering of the movable
    machines, handed to a repair whose first step puts every UNDAMAGED machine
    straight back at its original coordinates. Those coordinates came from a
    layout that was already collision-free, so they always succeed, in any
    order. The ordering therefore only mattered for machines that were actually
    destroyed -- with a 3x3 crater, typically one -- and permuting a
    ten-element list changes nothing unless two elements contend for the same
    cell, which they almost never do. Measured on the quick profile, 20
    evaluations produced 1 to 4 DISTINCT costs and the best was usually found on
    evaluation 1, i.e. by the deterministic repair before the metaheuristic did
    anything. Both variants were wrappers around the same repair, so a
    GA-vs-SA recovery comparison could not show a difference and a null result
    would have been the expected outcome rather than evidence about the methods.

    A SOLUTION here is (order, anchors):

      order    the sequence machines claim space in. Still meaningful: it
               decides who wins a contested cell.
      anchors  instance_id -> (x, y) or None. None means "put it back where it
               was". A coordinate means "this machine has been reinserted
               HERE", and _zpf_place tries that cell first, then spirals
               outward from it instead of from home.

    The operators move ANCHORS, so the search space is the set of layouts
    reachable inside the blast circle -- which is what the thesis claims it is.

    OPERATORS
      random    destroy k machines, reinsert each at a uniformly-sampled cell
                inside r2
      greedy    destroy k, reinsert each at whichever of ZPF_GREEDY_SAMPLES
                sampled cells minimises the Manhattan distance to its flow
                partners -- the greedy insertion ALNS is normally run with
      shift     jitter k anchors by a few cells (local refinement)
      restore   return k machines to their original coordinates. The undo move:
                without it the search cannot walk back a bad relocation, and
                the avoidable-relocation penalty makes walking back valuable.
      swap      exchange two machines' anchors
      order     re-order the placement sequence -- the ONLY move the previous
                neighbourhood had

    Set CFG.ZPF_POSITION_SEARCH = False to restrict it to `order` alone, which
    reproduces the old behaviour exactly and is how the "the metaheuristic
    contributes nothing at this crater size" claim can be measured rather than
    asserted.
    """

    ALL_OPERATORS = ("random", "greedy", "shift", "restore", "swap", "rotate",
                     "order")

    def __init__(self, disruption, movable, base, rng, k_max=None):
        self.d = disruption
        self.rng = rng
        self.movable = list(movable)
        self.ids = [p.instance_id for p in self.movable]
        self.by_id = {p.instance_id: p for p in self.movable}
        self.home = {p.instance_id: (p.x, p.y) for p in base["placed"]}
        self.k_max = max(1, CFG.ZPF_DESTROY_MAX if k_max is None else k_max)
        self.operators = (self.ALL_OPERATORS if CFG.ZPF_POSITION_SEARCH
                          else ("order",))

        self.partners = defaultdict(list)
        for fp in base.get("flows", ()):
            self.partners[fp.src_id].append(fp.dst_id)
            self.partners[fp.dst_id].append(fp.src_id)

        self.cx = disruption.x + disruption.w / 2.0
        self.cy = disruption.y + disruption.h / 2.0
        self.counts = defaultdict(int)
        # P7: rotation decides which face carries the I/O ports and was fixed at
        # R0 everywhere in this notebook. Here it is part of the solution and
        # the `rotate` operator searches it.
        self.rotations = {p.instance_id: p.rotation for p in self.movable}

    # ── candidate coordinates ────────────────────────────────────────────────
    def _candidate(self, pm):
        """One admissible top-left corner for `pm`, or None."""
        if pm.machine_type.edge_only and CFG.ZPF_EDGE_RELOCATION:
            return self._edge_cell(pm)
        for _ in range(20):
            ang = self.rng.random() * 2.0 * math.pi
            rad = self.d.r2 * math.sqrt(self.rng.random())   # uniform on the disc
            x = int(round(self.cx + rad * math.cos(ang))) - pm.w // 2
            y = int(round(self.cy + rad * math.sin(ang))) - pm.h // 2
            x = max(0, min(GRID_W - pm.w, x))
            y = max(0, min(GRID_H - pm.h, y))
            probe = type(pm)(pm.machine_type, x, y, pm.rotation,
                             pm.instance_id, pm.active_recipe)
            if self.d.machine_zone(probe) != "stable":
                return (x, y)
        return None

    def _edge_cell(self, pm):
        """Edge-only machines can only stand on the grid boundary, and no
        boundary cell exists inside a crater in the middle of the factory."""
        w, h = pm.w, pm.h
        side = self.rng.randrange(4)
        if side == 0:
            return (self.rng.randrange(max(1, GRID_W - w)), 0)
        if side == 1:
            return (self.rng.randrange(max(1, GRID_W - w)), max(0, GRID_H - h))
        if side == 2:
            return (0, self.rng.randrange(max(1, GRID_H - h)))
        return (max(0, GRID_W - w), self.rng.randrange(max(1, GRID_H - h)))

    def _effective(self, iid, anchors):
        a = anchors.get(iid)
        return a if a is not None else self.home.get(iid)

    def _greedy_cell(self, pm, anchors):
        """Cheapest of a few sampled cells, scored by distance to flow partners.

        A surrogate, deliberately: the real objective needs a full decode and
        re-route, and this runs inside the innermost loop of the search.
        """
        targets = [self._effective(q, anchors)
                   for q in self.partners.get(pm.instance_id, ())]
        targets = [t for t in targets if t]
        best, best_cost = None, None
        for _ in range(max(1, CFG.ZPF_GREEDY_SAMPLES)):
            cell = self._candidate(pm)
            if cell is None:
                continue
            if not targets:
                return cell
            cost = sum(abs(cell[0] - tx) + abs(cell[1] - ty) for tx, ty in targets)
            if best_cost is None or cost < best_cost:
                best, best_cost = cell, cost
        return best

    # ── solutions ────────────────────────────────────────────────────────────
    def initial(self):
        """The deterministic repair: everything back where it was."""
        return (self.movable[:], {}, {})

    def perturb(self, sol):
        order, anchors, rotations = sol
        order, anchors, rotations = list(order), dict(anchors), dict(rotations)
        if not self.ids:
            return (order, anchors, rotations)

        op = self.rng.choice(self.operators)
        self.counts[op] += 1
        k = self.rng.randint(1, min(self.k_max, len(self.ids)))
        picks = self.rng.sample(self.ids, k)

        if op == "random":
            for iid in picks:
                cell = self._candidate(self.by_id[iid])
                if cell:
                    anchors[iid] = cell
        elif op == "greedy":
            for iid in picks:
                cell = self._greedy_cell(self.by_id[iid], anchors)
                if cell:
                    anchors[iid] = cell
        elif op == "shift":
            for iid in picks:
                origin = self._effective(iid, anchors)
                if not origin:
                    continue
                pm = self.by_id[iid]
                anchors[iid] = (
                    max(0, min(GRID_W - pm.w, origin[0] + self.rng.randint(-3, 3))),
                    max(0, min(GRID_H - pm.h, origin[1] + self.rng.randint(-3, 3))))
        elif op == "restore":
            # The undo move: put the machine back exactly as it was, ORIENTATION
            # INCLUDED. Dropping only the anchor would make a rotation
            # irreversible, and the avoidable-relocation penalty means walking a
            # bad move back is often the improving step.
            for iid in picks:
                anchors.pop(iid, None)
                rotations.pop(iid, None)
        elif op == "swap" and len(self.ids) >= 2:
            a, b = self.rng.sample(self.ids, 2)
            av, bv = anchors.get(a), anchors.get(b)
            for iid, val in ((a, bv), (b, av)):
                if val is None:
                    anchors.pop(iid, None)
                else:
                    anchors[iid] = val
        elif op == "rotate":
            # P7: rotation decides which face carries the I/O ports. It was
            # fixed at R0 everywhere in this notebook, so this whole dimension
            # of the layout problem went unsearched.
            for iid in picks:
                choices = [r for r in Rotation
                           if r is not rotations.get(iid, self.rotations.get(iid))]
                if choices:
                    rotations[iid] = self.rng.choice(choices)
        elif len(order) >= 2:                       # "order"
            i, j = self.rng.randrange(len(order)), self.rng.randrange(len(order))
            order[i], order[j] = order[j], order[i]

        return (order, anchors, rotations)

    def crossover(self, s1, s2):
        """OX on the placement sequence, uniform on the anchors and rotations."""
        o1, a1, r1 = s1
        o2, a2, r2 = s2
        c1_order, c2_order = self._ordered_crossover(o1, o2)
        c1_anch, c2_anch, c1_rot, c2_rot = {}, {}, {}, {}
        for iid in self.ids:
            swap = self.rng.random() < 0.5
            first_a, second_a = (a1, a2) if swap else (a2, a1)
            first_r, second_r = (r1, r2) if swap else (r2, r1)
            if first_a.get(iid) is not None:
                c1_anch[iid] = first_a[iid]
            if second_a.get(iid) is not None:
                c2_anch[iid] = second_a[iid]
            if first_r.get(iid) is not None:
                c1_rot[iid] = first_r[iid]
            if second_r.get(iid) is not None:
                c2_rot[iid] = second_r[iid]
        return (c1_order, c1_anch, c1_rot), (c2_order, c2_anch, c2_rot)

    def _ordered_crossover(self, p1, p2):
        n = len(p1)
        if n < 3:
            return p1[:], p2[:]
        a, b = sorted(self.rng.sample(range(n), 2))
        out = []
        for src, other in ((p1, p2), (p2, p1)):
            child = [None] * n
            child[a:b + 1] = src[a:b + 1]
            used = set(id(x) for x in child if x is not None)
            fill = [x for x in other if id(x) not in used]
            j = 0
            for i in range(n):
                if child[i] is None:
                    child[i] = fill[j]
                    j += 1
            out.append(child)
        return out[0], out[1]

    def summary(self):
        if not self.counts:
            return "operators: none applied"
        total = sum(self.counts.values())
        return "operators: " + "  ".join(
            f"{k}={v}({v / total:.0%})" for k, v in sorted(self.counts.items()))


print("ZPFNeighbourhood defined -- destroy-and-reinsert over coordinates "
      f"(position search {'ON' if CFG.ZPF_POSITION_SEARCH else 'OFF'}).")


def zpf_recovery(chromosome, disruption, iterations=None, rng=None, base=None,
                 pop_size=None, ngen=None):
    """NSGA-II + ALNS recovery restricted to flexible/penalty machines.

    Objective 1: minimise unrouted paths.
    Objective 2: minimise MHC + zone-occupancy penalty + avoidable-relocation
                 penalty.

    The crater is permanently blocked, stable machines are frozen, and belts are
    freely reconfigured everywhere. What the search MOVES is a set of machine
    coordinates inside the blast circle -- see ZPFNeighbourhood for why it used
    to move a permutation instead and why that could not work.
    """
    # `iterations` is the TOTAL evaluation budget for the recovery search.
    # Budget = pop_size (initial population) + pop_size * ngen (offspring).
    iterations = CFG.ZPF_ITERATIONS if iterations is None else iterations
    pop_size = CFG.ZPF_POP_SIZE if pop_size is None else pop_size
    if ngen is None:
        ngen = max(1, int(round((iterations - pop_size) / max(1, pop_size))))
    rng = rng if rng is not None else random
    print(f"  [ZPF-NSGA2] budget={iterations} evals -> pop_size={pop_size}, ngen={ngen} "
          f"({pop_size + pop_size * ngen} evaluations)")

    if base is None:
        base = evaluate_layout(chromosome)

    movable = [p for p in base["placed"] if disruption.machine_zone(p) != "stable"]
    stable  = [p for p in base["placed"] if disruption.machine_zone(p) == "stable"]
    if not movable:
        return base, 0.0

    base_ids = {p.instance_id for p in base["placed"]}
    # ── cost of AVOIDABLE disturbance ────────────────────────────────────────
    # A machine sitting in the crater had no choice but to relocate. Every other
    # machine COULD have been left exactly where it was -- recovery tries the
    # original position first -- so if it ended up somewhere else, that was the
    # search's decision and it is charged for it.
    #
    # Without this the search was rewarded for rearranging healthy neighbours:
    # the only cost of a move was the incidental change in the zone-occupancy
    # penalty (5 x distance), and since that term SHRINKS toward the blast,
    # hauling an untouched machine inward -- or into the free circle, where it
    # is zero -- actively improved the objective.
    #
    # Scaled by the machine's ORIGINAL distance from the blast, so disturbing
    # something at the edge of the ring costs about twice what disturbing
    # something beside the crater does: a small disruption must not justify
    # rearranging the far side of the factory. Independent of how far the
    # machine was dragged -- reach is what is being priced, not travel.
    _orig_pos = {p.instance_id: (p.x, p.y) for p in base["placed"]}
    _destroyed_ids = {p.instance_id for p in base["placed"]
                      if disruption.machine_zone(p) == "unusable"}

    def _relocation_penalty(placed_list):
        """(total cost, number of healthy machines disturbed)."""
        total, n = 0.0, 0
        for p in placed_list:
            o = _orig_pos.get(p.instance_id)
            if o is None or (p.x, p.y) == o:
                continue                       # newly spawned, or never moved
            if p.instance_id in _destroyed_ids:
                continue                       # forced move -- exempt
            d = disruption.distance(o[0] + p.w / 2.0, o[1] + p.h / 2.0)
            total += CFG.PENALTY_PER_MOVED_MACHINE * (1.0 + d / max(1e-9, disruption.r2))
            n += 1
        return total, n

    hood = ZPFNeighbourhood(disruption, movable, base, rng)

    # Evaluations are the common currency for comparing recovery methods: an SA
    # iteration and an NSGA-II individual both cost one decode + reroute, which
    # dominates everything else here. Counted in a list so the nested function
    # can write to it without a nonlocal declaration.
    _eval_count = [0]
    _recovery_trace = []
    _seen_costs = set()

    def _evaluate_individual(sol):
        """Evaluate one (order, anchors) solution."""
        _eval_count[0] += 1
        order, anchors, rotations = sol
        grid, placed, unplaced = _zpf_place(disruption, stable, order, anchors,
                                            rotations)

        # Keep the pre-bomb supply plan; re-route belts only. See
        # CFG.ZPF_PRESERVE_FLOWS for why re-deriving here is not safe.
        if CFG.ZPF_PRESERVE_FLOWS:
            _alive = {p.instance_id for p in placed}
            flows = [fp for fp in base["flows"]
                     if fp.src_id in _alive and fp.dst_id in _alive]
        else:
            flows = derive_flow_matrix(placed)
        routed = route_belts(grid, {p.instance_id: p for p in placed}, flows)

        # Reconnection polish + feasibility repair. Shared with the SA variant
        # so both recoveries are handed the same repair budget.
        placed, flows, routed = _finalise_recovery(grid, placed, flows, routed, rng)

        mhc = sum(f.intensity * (len(p) - 1) for f, p in routed if p)
        paths_found = sum(1 for _, p in routed if p is not None)
        unrouted = len(flows) - paths_found

        # Zone-occupancy penalty: cost of standing in the ring at all.
        penalty = sum(5.0 * disruption.distance(p.x + p.w/2.0, p.y + p.h/2.0)
                       for p in placed if disruption.machine_zone(p) == "penalty")
        # Relocation penalty: cost of having disturbed a machine that could
        # have stayed put. See _relocation_penalty above.
        reloc, n_disturbed = _relocation_penalty(placed)

        demand = sum(p.machine_type.power_cost for p in placed)
        output = _power_output(placed, routed)
        power_ok = output > demand
        _cs = _chain_stats(placed, routed)
        chain_time, chain_ideal = _cs["time"], _cs["ideal"]
        sustain_ok = chain_time <= _sustain_budget(chain_ideal)
        revenue = _calculate_revenue(placed, routed)

        # Count only ORIGINAL machines: spawned depots must not be able to
        # substitute for a machine the recovery failed to re-place.
        n_original = sum(1 for p in placed if p.instance_id in base_ids)
        feasible = (n_original == len(base["placed"]) and
                    paths_found == len(flows) and power_ok and sustain_ok)

        result = dict(grid=grid, placed=placed, flows=flows, routed=routed,
                      # Recovery starts from base["placed"], which is ALREADY pruned,
                      # so it can never contain the machines evaluate_layout removed.
                      # Carry the same accounting across so the placement gate means
                      # the same thing pre- and post-bomb (otherwise the bomb appears
                      # to lose machines it never touched).
                      machines_removed=base.get("machines_removed", 0),
                      unplaced_ids=unplaced,
                      zone_penalty=penalty, relocation_penalty=reloc,
                      machines_disturbed=n_disturbed,
                      mhc=mhc, power_ok=power_ok, chain_time=chain_time,
                      chain_ideal=chain_ideal, chain_cyclic=_cs["cyclic"],
                      # Score routing against the PRE-BOMB demand and placement
                      # against the ORIGINAL machines only -- see
                      # _zpf_flow_demand and _feasibility_breakdown.
                      flows_demanded=_zpf_flow_demand(base),
                      placed_count=n_original,
                      paths_found=paths_found, flows_total=len(flows),
                      demand=demand, output=output, feasible=feasible,
                      revenue=revenue)

        cost = mhc + penalty + reloc + (0 if feasible else CFG.INFEASIBLE_PENALTY)
        _seen_costs.add(round(cost, 6))
        return result, (unrouted, mhc + penalty + reloc)

    # --- NSGA-II Implementation ---
    def _dominates(a, b):
        """True if a dominates b (all objectives <= and at least one <)."""
        return all(ai <= bi for ai, bi in zip(a, b)) and any(ai < bi for ai, bi in zip(a, b))

    def _fast_non_dominated_sort(population):
        """Returns list of fronts (each front is a list of indices)."""
        n = len(population)
        domination_count = [0] * n
        dominated_set = [[] for _ in range(n)]
        fronts = [[]]

        for i in range(n):
            for j in range(i + 1, n):
                fi, fj = population[i][1], population[j][1]
                if _dominates(fi, fj):
                    dominated_set[i].append(j)
                    domination_count[j] += 1
                elif _dominates(fj, fi):
                    dominated_set[j].append(i)
                    domination_count[i] += 1
            if domination_count[i] == 0:
                fronts[0].append(i)

        k = 0
        while fronts[k]:
            next_front = []
            for i in fronts[k]:
                for j in dominated_set[i]:
                    domination_count[j] -= 1
                    if domination_count[j] == 0:
                        next_front.append(j)
            k += 1
            fronts.append(next_front)

        return [f for f in fronts if f]

    def _crowding_distance(front, population):
        """Compute crowding distance for a front."""
        n = len(front)
        if n <= 2:
            return [float('inf')] * n

        distances = [0.0] * n
        n_obj = len(population[front[0]][1])

        for m in range(n_obj):
            sorted_indices = sorted(range(n), key=lambda i: population[front[i]][1][m])
            distances[sorted_indices[0]] = float('inf')
            distances[sorted_indices[-1]] = float('inf')

            obj_range = (population[front[sorted_indices[-1]]][1][m] -
                        population[front[sorted_indices[0]]][1][m])
            if obj_range < 1e-9:
                continue

            for k in range(1, n - 1):
                distances[sorted_indices[k]] += (
                    population[front[sorted_indices[k+1]]][1][m] -
                    population[front[sorted_indices[k-1]]][1][m]
                ) / obj_range

        return distances

    # Initialize population. Individual 0 is the deterministic repair --
    # everything back where it was -- so the search starts from the heuristic it
    # is meant to improve on and can never return something worse.
    print(f"  [ZPF-NSGA2] Initializing population of {pop_size}...")
    population = []  # List of (solution, objectives, result)

    for p_idx in range(pop_size):
        sol = hood.initial()
        if p_idx > 0:
            for _ in range(1 + p_idx % 3):
                sol = hood.perturb(sol)
        result, objectives = _evaluate_individual(sol)
        population.append((sol, objectives, result))

    # Evolution loop
    _stagnant_evals, _prev_best, stopped_early = 0, math.inf, False
    for gen in range(ngen):
        # Stagnation is counted in EVALUATIONS, not generations, so the
        # threshold means the same thing here as it does in the SA variant.
        if CFG.ZPF_EARLY_STOP and _stagnant_evals >= CFG.ZPF_EARLY_STOP:
            stopped_early = True
            print(f"    Stopped at gen {gen}/{ngen}: {_stagnant_evals} evaluations "
                  f"without improving the record.")
            break
        # Generate offspring via crossover + mutation
        offspring = []
        indices = list(range(len(population)))
        rng.shuffle(indices)

        for k in range(0, len(indices) - 1, 2):
            p1 = population[indices[k]][0]
            p2 = population[indices[k+1]][0]

            c1, c2 = hood.crossover(p1, p2)

            if rng.random() < 0.8:
                c1 = hood.perturb(c1)
            if rng.random() < 0.8:
                c2 = hood.perturb(c2)

            r1, o1 = _evaluate_individual(c1)
            offspring.append((c1, o1, r1))
            r2, o2 = _evaluate_individual(c2)
            offspring.append((c2, o2, r2))

        # Combine parent + offspring
        combined = population + offspring

        # NSGA-II selection
        fronts = _fast_non_dominated_sort(combined)
        new_pop = []

        for front in fronts:
            if len(new_pop) + len(front) <= pop_size:
                new_pop.extend(front)
            else:
                remaining = pop_size - len(new_pop)
                distances = _crowding_distance(front, combined)
                sorted_front = sorted(range(len(front)), key=lambda i: distances[i], reverse=True)
                new_pop.extend(front[sorted_front[k]] for k in range(remaining))
                break

        population = [combined[i] for i in new_pop]

        # Progress logging
        best_unrouted = min(p[1][0] for p in population)
        best_cost = min(p[1][1] for p in population)
        n_feasible = sum(1 for p in population if p[2]["feasible"])
        _gen_evals = _eval_count[0] - (_recovery_trace[-1]["evaluations"]
                                       if _recovery_trace else pop_size)
        if best_cost < _prev_best - 1e-9:
            _prev_best, _stagnant_evals = best_cost, 0
        else:
            _stagnant_evals += max(0, _gen_evals)
        _recovery_trace.append(dict(step=gen + 1, cost=best_cost, best_cost=best_cost,
                                    best_unrouted=best_unrouted, n_feasible=n_feasible,
                                    evaluations=_eval_count[0]))
        if gen % 5 == 0 or gen == ngen - 1:
            print(f"    Gen {gen+1:3d}/{ngen}: best_unrouted={best_unrouted}, "
                  f"best_cost={best_cost:.0f}, feasible={n_feasible}/{len(population)}")

    # Select best result: prefer feasible, then lowest unrouted, then lowest cost
    population.sort(key=lambda p: (not p[2]["feasible"], p[1][0], p[1][1]))
    best_sol, best_obj, best = population[0]

    moved_penalty = sum(5.0 * disruption.distance(p.x + p.w/2.0, p.y + p.h/2.0)
                        for p in best["placed"] if disruption.machine_zone(p) == "penalty")
    reloc_cost, n_disturbed = _relocation_penalty(best["placed"])
    best_cost = (best["mhc"] + moved_penalty + reloc_cost
                 + (0 if best["feasible"] else CFG.INFEASIBLE_PENALTY))

    print(f"  [ZPF-NSGA2] Recovery complete. Feasible={best['feasible']}, "
          f"Paths={best['paths_found']}/{best['flows_total']}, MHC={best['mhc']:.0f}")
    print(f"  [ZPF-NSGA2] Disturbed {n_disturbed} healthy machine(s) "
          f"(avoidable-move cost {reloc_cost:.0f}); "
          f"zone penalty {moved_penalty:.0f}; "
          f"{len(best.get('unplaced_ids', []))} machine(s) could not be re-placed")
    print(f"  [ZPF-NSGA2] {len(_seen_costs)} distinct cost(s) over {_eval_count[0]} "
          f"evaluations; {hood.summary()}")

    # First generation whose best cost equals the final best cost -- i.e. the
    # point after which the remaining budget bought nothing.
    _final = _recovery_trace[-1]["best_cost"] if _recovery_trace else best_cost
    _evals_to_best = next((t["evaluations"] for t in _recovery_trace
                           if t["best_cost"] <= _final + 1e-9), _eval_count[0])
    if best is base:
        best = dict(base)
    best["recovery_log"] = dict(
        method="GA", evaluations=_eval_count[0], generations=ngen,
        stopped_early=stopped_early, budget=iterations,
        early_stop_after=CFG.ZPF_EARLY_STOP,
        pop_size=pop_size, evals_to_best=_evals_to_best,
        wasted_evaluations=_eval_count[0] - _evals_to_best,
        distinct_costs=len(_seen_costs),
        operator_counts=dict(hood.counts),
        best_cost=best_cost, trace=_recovery_trace,
    )
    return best, best_cost

print("ZPF disruption model + zpf_recovery() defined (NSGA-II + ALNS).")


# ─── The un-recovered post-bomb state (R1's missing baseline) ─────────────────
def crater_violations(state, disruption):
    """(machines, belts) that intersect the crater. Must always be empty.

    _zpf_place paints the crater BLOCKED before placing anything, can_place
    accepts only EMPTY, and A*'s traversable set is (EMPTY, BELT) -- so a
    violation should be structurally impossible. It is asserted anyway, on every
    scenario, because it is the one property every Phase 4 number and every
    Phase 6 figure silently assumes, and a regression in the placement or
    routing path would not otherwise surface as an error. It is cheap: one pass
    over the footprints and the belt paths.
    """
    if disruption is None:
        return [], []
    bad_machines = [p.instance_id for p in state["placed"]
                    if any(disruption.distance(x, y) == 0 for x, y in p.cells())]
    bad_belts = [(fp.src_id, fp.dst_id) for fp, path in state["routed"]
                 if path and any(disruption.distance(x, y) == 0 for x, y in path)]
    return bad_machines, bad_belts


def post_bomb_state(base, disruption):
    """The factory immediately after the blast, with NO recovery attempted.

    Machines whose footprint intersects the crater are gone. Everything else
    stays exactly where it was: the flexible and penalty rings are only
    PERMITTED to move, and nothing has moved them yet. Belts are re-routed,
    because the crater is genuinely in the way -- but the SUPPLY PLAN is carried
    over unchanged (CFG.ZPF_PRESERVE_FLOWS). Phase 6 used to call
    derive_flow_matrix() from scratch here, which hands the damaged layout a
    different greedy matching: a different factory, not a damaged one.

    Deliberately does NOT call _finalise_recovery -- the reconnection polish and
    the depot spawning ARE recovery, and this is the state before any of it.

    Why this exists. `regret = pre - post_recovery` mixes how bad the bomb was
    with how well the algorithm recovered, so a layout the crater barely touched
    posts low regret no matter what the recovery did. With this baseline:

        recovery_gain     = post_recovery - post_bomb
        recovery_fraction = (post_recovery - post_bomb) / (pre - post_bomb)

    separate the two, and recovery_fraction is comparable across layouts of
    different sizes and damage levels. It is the quantity a claim like "SA
    recovered 62% of lost throughput where GA recovered 71%" actually needs.

    THE REFERENCE STATE. Pass `disruption=None` and nothing is destroyed: the
    result is the undamaged layout put through the SAME fresh-grid re-route the
    recovery pipeline uses. That matters because the pre-bomb layout's belts
    were laid incrementally -- blf_decode, then ALNS, then Tabu, then
    connect_or_remove, then repair_feasibility, each into whatever space the
    last one left -- while everything downstream of the bomb re-routes from
    scratch. Comparing the two mixes the bomb with the re-route, and not
    slightly: on the pool fork the un-recovered post-bomb layout scored 6.20
    against a pre-bomb 3.17, i.e. destroying two machines appeared to double the
    factory's revenue, purely because a less congested grid routed flows the
    incremental process never managed. With the reference state the damage
    decomposes:

        reroute = ref  - pre      nothing to do with the bomb
        damage  = post - ref      what the crater cost
        gain    = rec  - post     what the recovery contributed
    """
    grid = FactoryGrid()
    if disruption is not None:
        for y in range(disruption.y, disruption.y + disruption.h):
            for x in range(disruption.x, disruption.x + disruption.w):
                if grid.in_bounds(x, y):
                    grid.grid[y][x] = (TileState.BLOCKED, None)

    placed, lost = [], []
    for pm in base["placed"]:
        if _zone_of(disruption, pm) == "unusable" or not grid.place_machine(pm):
            lost.append(pm.instance_id)
        else:
            placed.append(pm)

    alive = {p.instance_id for p in placed}
    if CFG.ZPF_PRESERVE_FLOWS:
        flows = [fp for fp in base["flows"]
                 if fp.src_id in alive and fp.dst_id in alive]
    else:
        flows = derive_flow_matrix(placed)
    routed = route_belts(grid, {p.instance_id: p for p in placed}, flows)

    paths_found = sum(1 for _, p in routed if p is not None)
    mhc = sum(f.intensity * (len(p) - 1) for f, p in routed if p)
    demand = sum(p.machine_type.power_cost for p in placed)
    output = _power_output(placed, routed)
    cs = _chain_stats(placed, routed)
    return dict(grid=grid, placed=placed, flows=flows, routed=routed, mhc=mhc,
                paths_found=paths_found, flows_total=len(flows),
                flows_demanded=_zpf_flow_demand(base),
                machines_removed=base.get("machines_removed", 0),
                placed_count=len(placed), destroyed_ids=lost,
                demand=demand, output=output, power_ok=output > demand,
                chain_time=cs["time"], chain_ideal=cs["ideal"],
                feasible=False, revenue=_calculate_revenue(placed, routed))


def _zpf_flow_demand(base):
    """Flow demand every post-bomb state is scored against.

    The routing gate is `paths_found / demand`. Pre-bomb the denominator is the
    flow matrix as originally DERIVED; post-bomb it used to be whatever flows
    survived, so destroying a machine deleted its unroutable flows from the
    denominator and the routing gate went UP. Measured on the quick profile: a
    crater that destroyed 3 machines moved feasibility 96.6% -> 96.8% and regret
    came out 0.0000.

    A flow whose endpoint was destroyed is not "no longer required" -- the
    material genuinely stops moving. Holding the denominator at the pre-bomb
    demand is what makes pre and post comparable at all.
    """
    return base.get("flows_demanded") or len(base.get("flows") or ())


def recovery_metrics(pre_rate, post_bomb_rate, post_recovery_rate, ref_rate=None):
    """(regret, gain, fraction) -- see post_bomb_state for why all three.

    `regret` keeps its original definition against the REPORTED pre-bomb rate,
    because that is the number the thesis quotes. `fraction` is measured against
    `ref_rate`, the same layout re-routed on a fresh grid, so the denominator is
    what the crater actually cost rather than the crater plus the difference
    between two belt-laying procedures. Falls back to `pre_rate` when no
    reference was computed.
    """
    regret = max(0.0, pre_rate - post_recovery_rate)
    gain = post_recovery_rate - post_bomb_rate
    base = pre_rate if ref_rate is None else ref_rate
    lost = base - post_bomb_rate
    # A fraction of a loss that rounds to nothing is not a percentage, it is a
    # division by noise: an observed run reported "recovered 251050.7%" because
    # the crater cost 0.0006 of a rate of 3.7. Anything under 1% of the
    # reference is reported as n/a rather than as a number nobody can use.
    fraction = (gain / lost) if lost > max(1e-9, 0.01 * abs(base)) else float("nan")
    return regret, gain, fraction

print("post_bomb_state() defined -- un-recovered baseline for regret / recovery_gain / recovery_fraction.")

# ─── Bombing scenario generation ─────────────────────────────────────────────
def make_bomb_scenarios(placed, n=None, seed=0, require_damage=None, attempts=None):
    """N crater positions for one layout, sampled uniformly over the grid.

    With `require_damage` a position is rejected unless it destroys at least one
    machine. That conditions the sample on a non-null event without distorting
    WHERE damage lands: a uniform draw naturally favours dense regions and large
    footprints, which is what an accident does. Centring on a uniformly chosen
    machine would instead over-sample isolated ones.

    Returns a list of ZPF objects. Deterministic in `seed`, so the same layout
    yields the same scenario set for every recovery algorithm.
    """
    n               = CFG.ZPF_SCENARIOS       if n              is None else n
    require_damage  = CFG.ZPF_REQUIRE_DAMAGE  if require_damage is None else require_damage
    attempts        = CFG.ZPF_SAMPLE_ATTEMPTS if attempts       is None else attempts
    rng = random.Random(seed)

    cw, ch, r1, r2 = CFG.ZPF_W, CFG.ZPF_H, CFG.ZPF_R1, CFG.ZPF_R2
    frac = getattr(CFG, "ZPF_SCALE_TO_LAYOUT", 0.0)
    if frac > 0 and placed:
        # Size the crater against the LAYOUT, not the grid. See
        # CFG.ZPF_SCALE_TO_LAYOUT for the measured effect.
        xs = [p.x for p in placed] + [p.x + p.w for p in placed]
        ys = [p.y for p in placed] + [p.y + p.h for p in placed]
        extent = max(max(xs) - min(xs), max(ys) - min(ys))
        scale = max(1.0, (frac * extent) / max(1, CFG.ZPF_W))
        cw = max(1, int(round(CFG.ZPF_W * scale)))
        ch = max(1, int(round(CFG.ZPF_H * scale)))
        r1, r2 = CFG.ZPF_R1 * scale, CFG.ZPF_R2 * scale

    out, tries = [], 0
    while len(out) < n and tries < attempts:
        tries += 1
        x = rng.randrange(0, max(1, GRID_W - cw))
        y = rng.randrange(0, max(1, GRID_H - ch))
        z = ZPF(x, y, cw, ch, r1, r2)
        if require_damage and not any(z.machine_zone(p) == "unusable" for p in placed):
            continue
        out.append(z)
    if not out:                                # degenerate layout: fall back
        out = [ZPF(CFG.ZPF_X, CFG.ZPF_Y, CFG.ZPF_W, CFG.ZPF_H, CFG.ZPF_R1, CFG.ZPF_R2)]
    return out


def scenario_damage(placed, z):
    """(destroyed, movable) machine counts for one crater, before any recovery."""
    zones = [z.machine_zone(p) for p in placed]
    return sum(1 for v in zones if v == "unusable"), sum(1 for v in zones if v != "stable")


print("make_bomb_scenarios() defined -- rejection-sampled craters, shared per layout.")


In [ ]:
# ─── V4 FITNESS EVALUATION ───────────────────────────────────────────────────────
def _expand_bom(seed_demand, multipliers):
    """One backward expansion pass. Returns {(machine_id, recipe_idx): count}."""
    demand, expanded, required = defaultdict(float), defaultdict(float), defaultdict(int)
    for name, rate in seed_demand.items():
        if rate > 0:
            demand[name] += rate
    queue = deque(demand)
    while queue:
        item = queue.popleft()
        delta = demand[item] - expanded[item]
        if delta <= 1e-9:
            continue
        producers = [(mid, i, r, q) for mid, mt in MACHINE_CATALOGUE.items()
                     for i, r in enumerate(mt.recipes)
                     for out, q in r.outputs if out == item]
        if not producers:
            expanded[item] = demand[item]        # raw import; nothing makes it
            continue
        mid, idx, recipe, out = max(producers, key=lambda p: multipliers.get((p[0], p[1]), .5))
        target_count = math.ceil(demand[item] / (out / recipe.process_time))
        increment = max(0, target_count - required[(mid, idx)])
        required[(mid, idx)] += increment
        expanded[item] = demand[item]
        for material, qty in recipe.inputs + recipe.common_extra:
            demand[material] += increment * qty / recipe.process_time
            queue.append(material)
    return required


# ─── Which battery should the thermal banks burn? ────────────────────────────
_CHAIN_SIZE_CACHE: dict = {}

def _chain_machine_count(material, _seen=frozenset()):
    """Machines standing behind one unit of `material`. 0 means raw.

    A crude but sufficient proxy for "how much factory does this fuel cost me":
    a raw material arrives through a depot and costs nothing to make, while a
    two-stage battery drags a whole sub-chain in with it. Cycles (planting <->
    seed-picking) terminate at 0 rather than recursing.
    """
    if not _seen and material in _CHAIN_SIZE_CACHE:
        return _CHAIN_SIZE_CACHE[material]
    if material in _seen:
        return 0
    producers = [(mid, i, r) for mid, mt in MACHINE_CATALOGUE.items()
                 for i, r in enumerate(mt.recipes)
                 for out, _ in r.outputs if out == material]
    if not producers:
        # Two very different cases that used to be conflated. An ore arrives on
        # its own at a fixed rate, so it costs no machines. Anything else with
        # no producer cannot be obtained at all, and must never be scored as
        # the cheapest option -- which is what put the two W-battery recipes at
        # the top of the fuel table.
        if material in RAW_MATERIALS:
            return 0
        return math.inf
    _, _, rec = producers[0]
    n = 1 + sum(_chain_machine_count(m, _seen | {material})
                for m, _ in list(rec.inputs) + list(rec.common_extra))
    if not _seen:
        _CHAIN_SIZE_CACHE[material] = n
    return n


def fuel_options():
    """Every ther_bank recipe, with the cost of supplying it."""
    mt, out = MACHINE_CATALOGUE["ther_bank"], []
    for i, rec in enumerate(mt.recipes):
        power = sum(q for m, q in rec.outputs if m == "power")
        fuels = [m for m, _ in list(rec.inputs) + list(rec.common_extra)]
        if power <= 0 or len(fuels) != 1:
            continue
        chain = _chain_machine_count(fuels[0])
        if chain == math.inf or fuels[0] in UNOBTAINABLE:
            continue                     # dead recipe -- nothing can supply it
        out.append(dict(idx=i, fuel=fuels[0], power=power, chain=chain,
                        raw=(fuels[0] in RAW_MATERIALS),
                        score=power / (1.0 + chain)))
    return out


def choose_power_recipe(verbose=False):
    """(recipe_index, power_per_bank) for the fuel the banks will burn.

    Scored as power per machine of supporting chain, so a fuel that costs three
    machines to keep supplied has to deliver three times the power to win. This
    is a BOM-time decision, not a searched one -- see F1 for why putting it in
    the genome is the better fix.
    """
    opts = fuel_options()
    if not opts:
        return 2, max(1e-9, CFG.BATTERY_CAPACITY)

    pin = getattr(CFG, "POWER_FUEL", "auto")
    if pin and pin != "auto":
        for o in opts:
            if o["fuel"] == pin:
                if verbose:
                    print(f"  Power fuel PINNED to {o['fuel']} "
                          f"({o['power']} power, chain {o['chain']} machines)")
                return o["idx"], o["power"]
        print(f"  [WARN] CFG.POWER_FUEL={pin!r} is not a ther_bank fuel; using auto.")

    pool = opts if getattr(CFG, "POWER_FUEL_ALLOW_RAW", False) else \
           [o for o in opts if not o["raw"]] or opts
    best = max(pool, key=lambda o: (o["score"], o["power"]))
    if verbose:
        print("  Power fuel options (power per machine of supporting chain):")
        for o in sorted(opts, key=lambda o: -o["score"]):
            mark = "<-- chosen" if o["idx"] == best["idx"] else ""
            flag = " RAW" if o["raw"] else "    "
            print(f"    recipe {o['idx']}  {o['fuel']:<10}{flag} "
                  f"power={o['power']:>5}  chain={o['chain']:>2} machines  "
                  f"score={o['score']:>7.1f}  {mark}")
        if not getattr(CFG, "POWER_FUEL_ALLOW_RAW", False) and any(o["raw"] for o in opts):
            print("    (RAW fuels excluded: CFG.POWER_FUEL_ALLOW_RAW=False -- they "
                  "have no chain, so they would make power free)")
    return best["idx"], best["power"]


def _bank_fuel(pm, default="SC_bat"):
    """The battery a specific placed thermal bank actually burns."""
    mt = pm.machine_type
    if pm.active_recipe is None or not mt.recipes or pm.active_recipe >= len(mt.recipes):
        return default
    rec = mt.recipes[pm.active_recipe]
    for m, _ in list(rec.inputs) + list(rec.common_extra):
        if m in BATTERY_POWER:
            return m
    return default




def check_raw_dependencies(placed, routed, verbose=True, indent="    "):
    """Materials a layout imports across the boundary that are not ores.

    A layout may legitimately draw ore from the boundary. Anything else it takes
    from main_unit or a depot without producing is an assumption, not a supply
    chain -- and in the `_fixed` fork the thermal banks' own battery fuel is
    exactly that: banks are appended to the BOM after the expansion loop, so
    their demand never re-enters the queue and the batteries come from nowhere.
    `_pool` fixes this with SELF_FUEL_POWER; here it is at least visible.
    """
    made = set()
    for pm in placed:
        mt = pm.machine_type
        if pm.active_recipe is None or not mt.recipes:
            continue
        if pm.active_recipe >= len(mt.recipes):
            continue
        made.update(o for o, _ in mt.recipes[pm.active_recipe].outputs)

    imported = {}
    for pm in placed:
        mt = pm.machine_type
        if pm.active_recipe is None or not mt.recipes:
            continue
        if pm.active_recipe >= len(mt.recipes):
            continue
        rec = mt.recipes[pm.active_recipe]
        for m, _ in list(rec.inputs) + list(rec.common_extra):
            if m in made or m in RAW_MATERIALS or m in NON_CONVEYED_MATERIALS:
                continue
            imported.setdefault(m, []).append(pm.instance_id)

    if verbose and imported:
        print(f"{indent}[RAW] {len(imported)} non-ore material(s) imported with no "
              f"producer in this layout:")
        for m, users in sorted(imported.items()):
            tag = " UNOBTAINABLE" if m in UNOBTAINABLE else ""
            print(f"{indent}  {m}{tag} -> {len(users)} machine(s): "
                  f"{', '.join(users[:4])}")
    return imported


print("choose_power_recipe() defined -- thermal-bank fuel is no longer hardcoded.")
choose_power_recipe(verbose=True)


def derive_rate_balanced_bom(product_rates, multipliers):
    """Incremental backward expansion; shared demand is never discarded.

    With CFG.SELF_FUEL_POWER the thermal banks' own fuel demand is fed back
    through the expansion as a fixed point: more machines -> more power -> more
    banks -> more battery demand -> more machines, iterated until the count
    settles. Previously banks were appended after the expansion finished, so
    their fuel demand never entered the queue and no battery was ever produced --
    the banks ran on material conjured out of main_unit.
    """
    if not hasattr(multipliers, "get"):
        multipliers = dict(zip(ALL_RECIPES, multipliers))

    tb        = MACHINE_CATALOGUE["ther_bank"]
    # CFG.THER_BANK_RECIPE (default 2 = SC_bat) is honoured only when
    # CFG.POWER_FUEL pins a fuel; otherwise the chooser decides.
    _tb_idx, _ = choose_power_recipe()
    tb_recipe = tb.recipes[_tb_idx]
    fuel_mat  = tb_recipe.inputs[0][0]
    per_bank  = sum(q for m, q in tb_recipe.outputs if m == "power")
    if CFG.POWER_RATE_CORRECT:
        per_bank /= max(1, tb_recipe.process_time)
    per_bank  = max(per_bank, 1e-9)
    burn_rate = 1.0 / max(1, tb_recipe.process_time)      # batteries/tick/bank

    base_seed = {name: rate for name, rate in zip(TARGET_PRODUCTS, product_rates) if rate > 0}

    fuel_demand, required, n_banks = 0.0, {}, 0
    for _ in range(max(1, CFG.BOM_POWER_PASSES)):
        seed = dict(base_seed)
        if CFG.SELF_FUEL_POWER and fuel_demand > 0:
            seed[fuel_mat] = seed.get(fuel_mat, 0.0) + fuel_demand
        required = _expand_bom(seed, multipliers)

        power = sum(MACHINE_CATALOGUE[mid].power_cost * cnt
                    for (mid, _), cnt in required.items())
        power += MACHINE_CATALOGUE["main_unit"].power_cost
        new_banks = (math.floor(power / per_bank) + 1) if power else 0
        new_fuel  = new_banks * burn_rate

        if not CFG.SELF_FUEL_POWER or abs(new_fuel - fuel_demand) < 1e-9:
            n_banks = new_banks
            break
        fuel_demand, n_banks = new_fuel, new_banks

    instances = [(MACHINE_CATALOGUE["main_unit"], None)]
    for (mid, idx), count in required.items():
        instances.extend((MACHINE_CATALOGUE[mid], idx) for _ in range(count))

    # Chain terminators, added before the bank count is finalised so their own
    # power_cost is included. See CFG.IMPORT_DEPOTS / EXPORT_STASHES.
    def _n_term(val, n_base):
        if val is None or val <= 0:
            return 0
        return int(val) if val >= 1 else max(1, math.ceil(val * n_base))

    _base_n = len(instances)
    for _mid, _knob in (("depot_unloader", CFG.IMPORT_DEPOTS),
                        ("stash",          CFG.EXPORT_STASHES),
                        ("depot_loader",   CFG.EXPORT_LOADERS)):
        for _ in range(_n_term(_knob, _base_n)):
            instances.append((MACHINE_CATALOGUE[_mid], None))

    power = sum(m.power_cost for m, _ in instances)
    if power:
        n_banks = max(n_banks, math.floor(power / per_bank) + 1)
    instances.extend((tb, _tb_idx) for _ in range(n_banks))
    return instances


def _feasibility_breakdown(res, chromosome_len):
    """The four sub-ratios that _feasibility_ratio averages.

    Reported everywhere a feasibility percentage is printed, because the
    headline number on its own is close to unreadable: the four ratios carry
    equal weight, so ANY single gate hitting 0 costs exactly 25 percentage
    points however good the other three are. Observed on a real run, the same
    chromosome through Phase 1 then Phase 2:

        phase placed removed flows routed | place route power sust | FEAS
           P1     16       2    23     14 |  0.89  0.61  1.00  0.76 |  81%
           P2     18       0    27     23 |  1.00  0.85  0.00  0.22 |  52%

    Phase 2 placed MORE machines, pruned FEWER, and routed a higher fraction --
    and still scored 29 points lower, purely because the power and sustain gates
    fell. Without the breakdown that reads as the layout getting worse.

    Note `power` is close to binary when the BOM contains a single thermal
    bank: the bank is either fuelled (1.00) or it is not (0.00).
    """
    # PLACEMENT means "did the decoder find a spot for every machine". Machines
    # deleted afterwards by connect_or_remove_unlinked were placed fine -- they
    # were removed for being unconnectable, which is a ROUTING failure. Adding
    # machines_removed back keeps this gate measuring placement only.
    # `placed_count`, when present, is the count of ORIGINAL machines -- the
    # recovery result's `placed` list also holds any depots repair_feasibility
    # spawned, and without this a spawned depot substitutes for a machine the
    # recovery failed to re-place.
    placed_ok = res.get("placed_count", len(res["placed"])) + res.get("machines_removed", 0)
    placement = min(1.0, placed_ok / max(1, chromosome_len))
    # ROUTING is scored against the flow demand as originally derived, so a
    # connection that was "solved" by deleting one of its machines still counts
    # as unrouted instead of quietly leaving the denominator.
    _demand = res.get("flows_demanded") or res["flows_total"]
    routing   = min(1.0, (res["paths_found"] / _demand) if _demand else 1.0)
    power     = min(1.0, res["output"] / res["demand"]) if res["demand"] > 0 else 1.0
    # SUSTAIN is scored against a budget that scales with the layout's own
    # irreducible processing time, not a flat tick count -- otherwise the gate
    # rewards having fewer machines and a bombed layout scores BETTER on it than
    # the layout it was made from. See CFG.SUSTAIN_NORMALISE and _sustain_budget.
    ct        = res["chain_time"]
    budget    = _sustain_budget(res.get("chain_ideal", 0.0))
    sustain   = (min(1.0, budget / ct) if 0 < ct < math.inf
                 else (1.0 if ct == 0 else 0.0))
    return dict(placement=placement, routing=routing, power=power, sustain=sustain)


def _feas_str(res, chromosome_len):
    """One-line 'place/route/power/sustain' summary for logging."""
    b = _feasibility_breakdown(res, chromosome_len)
    return (f"place {b['placement']:.2f} route {b['routing']:.2f} "
            f"power {b['power']:.2f} sust {b['sustain']:.2f}")


def _feasibility_ratio(res, chromosome_len):
    """Headline feasibility. Gates with weight 0 are dropped, not zero-weighted
    into a still-4-way average, so switching one off renormalises the rest.

    CFG.FEAS_AGGREGATION selects the rule -- see the note there. The default is
    still the weighted mean, but W_POWER now defaults to 0.0, which removes the
    25-point cliff a single fuel belt used to cause.
    """
    b = _feasibility_breakdown(res, chromosome_len)
    pairs = [(CFG.W_PLACEMENT, b["placement"]), (CFG.W_ROUTING, b["routing"]),
             (CFG.W_POWER, b["power"]), (CFG.W_SUSTAIN, b["sustain"])]
    pairs = [(w, v) for w, v in pairs if w > 0.0]
    if not pairs:
        return 1.0
    mode = getattr(CFG, "FEAS_AGGREGATION", "mean")
    if mode == "min":
        return min(v for _, v in pairs)
    total = sum(w for w, _ in pairs)
    if mode == "geometric":
        out = 1.0
        for w, v in pairs:
            out *= max(v, 1e-6) ** (w / total)
        return out
    return sum(w * v for w, v in pairs) / total


def _calculate_weighted_score(feasibility, revenue):
    """Combined ranking score for Phase 2/3 candidate sorting. Was called
    from both refined.sort()/polished.sort() but never actually defined --
    guaranteed NameError the moment Phase 2 ran its sort. WEIGHT_FEASIBILITY
    (100.0) vs WEIGHT_REVENUE (0.2) makes the intent clear: rank primarily
    by how close to fully feasible a candidate is, with revenue only
    distinguishing between otherwise similarly-feasible candidates, not
    trading feasibility away for it.
    """
    return feasibility * CFG.WEIGHT_FEASIBILITY + revenue * CFG.WEIGHT_REVENUE



def evaluate_nsga2(individual):
    n_mix, n_rec = len(TARGET_PRODUCTS), len(ALL_RECIPES)
    rates = individual[:n_mix]
    instances, chromo = chromosome_for(individual)

    raw_terminal_rate = sum(max(0.0, rate) * REVENUE_TABLE[name] for name, rate in zip(TARGET_PRODUCTS, rates)
                             if name in REVENUE_TABLE)

    if len(instances) > MAX_MACHINES:
        # Graded over-budget penalty — replaces a flat (1e9, 0.0, 1e9, 1e9)
        # for ANY overage, which made a BOM 1 machine over the cap and one
        # 1000 machines over the cap look identical to the optimizer: both a
        # dead end, so there was no gradient pulling the population back
        # DOWN toward the boundary from above, only pressure to stay well
        # clear of it entirely. Scaling the penalty with the overage, and
        # still reporting a (discounted) terminal_rate instead of zero, lets
        # NSGA-II tell "barely over, ambitious" apart from "wildly oversized"
        # and approach MAX_MACHINES as a real boundary rather than a cliff to
        # avoid.
        overage_ratio = (len(instances) - MAX_MACHINES) / MAX_MACHINES
        penalty = 1e5 * (1.0 + overage_ratio)
        return (penalty, raw_terminal_rate * 0.1, penalty)

    result = evaluate_layout(chromo, repair=False)

    total_genes = len(instances)
    main_feas = _feasibility_ratio(result, total_genes)
    terminal_rate = result["revenue"] * main_feas

    # No regret objective here: this phase builds layouts, it does not bomb
    # them. Regret is produced in Phase 4 and, against simulated throughput,
    # in Phase 5. See the note on CFG.OBJ_WEIGHTS.
    #
    # NOTE: this attribute does NOT survive CFG.MULTIPROCESSING -- joblib
    # evaluates in a worker process and only the returned tuple comes back, so
    # the parent's individuals never see it. Phase 2 recomputes everything it
    # needs, and the dashboard no longer reads it. Kept only for single-process
    # debugging.
    individual._flp_cache = dict(result=result, zpf=None, terminal_rate=terminal_rate,
                                  main_feas=main_feas, zpf_feas=0.0, total_genes=total_genes)

    return (result["build_cost"], terminal_rate, result["mhc"])

print("derive_rate_balanced_bom() and evaluate_nsga2() defined (v4, 3 objectives: cost / rate / MHC).")

# ─── Deterministic BOM ordering (replaces the GA's priority tail) ─────────────
_DEPTH_CACHE: dict = {}

def _recipe_depth(mid, idx, _stack=None):
    """Longest chain from a raw material to this (machine, recipe).

    Used to place upstream machines before downstream ones, so blf_decode's
    adjacency spiral has a placed neighbour to cluster around. Cyclic chains
    (planting <-> seed-picking) return 0 rather than recursing forever.
    """
    key = (mid, idx)
    if key in _DEPTH_CACHE:
        return _DEPTH_CACHE[key]
    _stack = _stack or set()
    if key in _stack:
        return 0
    mt = MACHINE_CATALOGUE[mid]
    if idx is None or not mt.recipes or idx >= len(mt.recipes):
        return 0
    rec = mt.recipes[idx]
    best = 0
    for mat, _ in list(rec.inputs) + list(rec.common_extra):
        for pid, pmt in MACHINE_CATALOGUE.items():
            for pidx, prec in enumerate(pmt.recipes):
                if any(o == mat for o, _ in prec.outputs):
                    best = max(best, 1 + _recipe_depth(pid, pidx, _stack | {key}))
    _DEPTH_CACHE[key] = best
    return best


def _order_instances(instances):
    """Deterministic build order for a BOM.

    The GA used to carry MAX_MACHINES float 'priority' genes and sort by them.
    Measured, one uniform crossover preserved only 5% of the resulting order at
    MAX_MACHINES=160 -- the placement order was effectively re-randomised every
    generation, so nothing about it could ever be inherited, and 70% of the
    genome was noise. Ordering is now derived from the production graph and the
    ARRANGEMENT is left to ALNS/Tabu in Phase 2, which is what they exist for.

    Order: main_unit, then import depots, then production machines shallowest
    first, then thermal banks, then export sinks.
    """
    def key(item):
        m, r = item
        mid = m.id
        if mid == "main_unit":       band = -3
        elif mid == "depot_unloader": band = -2
        elif mid == "ther_bank":      band = 90
        elif mid in ("stash", "depot_loader"): band = 99
        else:                         band = _recipe_depth(mid, r)
        return (band, mid, -1 if r is None else r)
    return sorted(instances, key=key)


def chromosome_for(individual):
    """Individual -> (instances, Chromosome). Single definition, used by the GA
    and by Phase 2, so the two can never build different layouts."""
    n_mix, n_rec = len(TARGET_PRODUCTS), len(ALL_RECIPES)
    rates = individual[:n_mix]
    mult  = dict(zip(ALL_RECIPES, individual[n_mix:n_mix + n_rec]))
    instances = derive_rate_balanced_bom(rates, mult)
    ordered = _order_instances(instances)
    return instances, Chromosome([Gene(m.id, Rotation.R0, r) for m, r in ordered])


def genotype_key(individual):
    """Identity of the LAYOUT an individual encodes, for de-duplication.

    The population collapses hard: a real 100x30 run produced 20 Phase-2
    candidates with only 9 distinct (rate, cost, MHC) triples, and bombed 10
    layouts of which only 5 were distinct. Half the Phase 2/4 budget was spent
    re-doing identical work.
    """
    _, chromo = chromosome_for(individual)
    return tuple((g.machine_id, g.recipe_idx) for g in chromo.genes)


print("Deterministic BOM ordering enabled; placement priorities dropped from the genome.")


# ─── P7: rotation search on the layout side ──────────────────────────────────
def rotation_polish(chromo, result, rng=None):
    """Budgeted greedy rotation pass over the machines that cost the most.

    Every chromosome in this notebook is built `Gene(m.id, Rotation.R0, r)` --
    in evaluate_nsga2, in Phase 2 and in the recovery -- so rotation, which
    decides which face carries a machine's I/O ports, was fixed at zero
    everywhere and an entire dimension of the layout problem was never searched.
    greedy_rotation_search() in the legacy appendix does this exhaustively and is
    called by nothing; at 4 x n_genes x passes full evaluations it is
    unaffordable at MAX_MACHINES=160.

    This is the same idea with a budget. Only the CFG.ROTATION_POLISH_TOP genes
    whose machines carry the most handling cost in `result` are considered, and
    only a rotation that strictly improves (feasibility, then revenue, then MHC)
    is kept. Returns (chromosome, result) -- unchanged if nothing helped, so it
    can never make a candidate worse.

    Set CFG.ROTATION_POLISH_TOP = 0 to skip it entirely.
    """
    top_n = getattr(CFG, "ROTATION_POLISH_TOP", 0)
    if not top_n or len(chromo.genes) > getattr(CFG, "ROTATION_POLISH_MAX_BOM", 0):
        return chromo, result

    n_genes = len(chromo.genes)

    def _score(res):
        # Scored by the SAME criterion Phase 2 ranks candidates on -- feasibility
        # heavily weighted, revenue only separating similarly-feasible ones --
        # with MHC breaking ties. An earlier version ranked on feasibility alone
        # first and accepted +386 MHC for +2.4 points of feasibility; MHC is one
        # of the three NSGA-II objectives, so it cannot be a free variable here.
        f = _feasibility_ratio(res, n_genes)
        return (_calculate_weighted_score(f, res["revenue"]), -res["mhc"])

    def _affordable(res):
        """A rotation may improve the ranking score, but not by spending
        unlimited MHC to do it. See CFG.ROTATION_POLISH_MHC_TOLERANCE."""
        tol = getattr(CFG, "ROTATION_POLISH_MHC_TOLERANCE", 0.05)
        return res["mhc"] <= best_result["mhc"] * (1.0 + tol) + 1e-9

    best_chromo, best_result, best_score = chromo, result, _score(result)

    # Rank by the handling cost the machine carries, so the budget is spent
    # where a different port face can actually pay for itself. A gene index does
    # not map back to an instance id (blf_decode assigns those), so the cost is
    # aggregated per machine TYPE and each gene inherits its type's rank.
    load = defaultdict(float)
    for fp, path in result["routed"]:
        cost = fp.intensity * ((len(path) - 1) if path else GRID_W + GRID_H)
        load[fp.src_id] += cost
        load[fp.dst_id] += cost
    by_type = defaultdict(float)
    for pm in result["placed"]:
        by_type[pm.machine_type.id] += load.get(pm.instance_id, 0.0)
    order = sorted(range(n_genes),
                   key=lambda i: -by_type.get(chromo.genes[i].machine_id, 0.0))

    for _ in range(max(1, getattr(CFG, "ROTATION_POLISH_PASSES", 1))):
        improved = False
        for i in order[:top_n]:
            for rot in Rotation:
                if rot is best_chromo.genes[i].rotation:
                    continue
                candidate = best_chromo.copy()
                candidate.genes[i].rotation = rot
                cand_result = evaluate_layout(candidate, rng=rng, repair=False)
                cand_score = _score(cand_result)
                if cand_score > best_score and _affordable(cand_result):
                    best_chromo, best_result, best_score = candidate, cand_result, cand_score
                    improved = True
        if not improved:
            break
    return best_chromo, best_result


print("rotation_polish() defined -- budgeted per-gene rotation search "
      f"(top {getattr(CFG, 'ROTATION_POLISH_TOP', 0)} machines).")


In [ ]:
# ─── GA CONFIGURATION ───────────────────────────────────────────────────────────
if "FitnessV4" in creator.__dict__: del creator.FitnessV4
if "IndividualV4" in creator.__dict__: del creator.IndividualV4
creator.create("FitnessV4", base.Fitness, weights=CFG.OBJ_WEIGHTS)
creator.create("IndividualV4", list, fitness=creator.FitnessV4)
toolbox = base.Toolbox()

# --- Multiprocessing Toggle ---
if getattr(CFG, 'MULTIPROCESSING', True):
    from joblib import Parallel, delayed
    def joblib_map(func, iterable):
        return Parallel(n_jobs=getattr(CFG, 'N_JOBS', -1), backend="loky")(delayed(func)(item) for item in iterable)
    toolbox.register("map", joblib_map)
    print(f"DEAP Multiprocessing ENABLED (Cores: {getattr(CFG, 'N_JOBS', 'All')})")
else:
    toolbox.register("map", map)
    print("DEAP Multiprocessing DISABLED")

n_mix, n_rec = len(TARGET_PRODUCTS), len(ALL_RECIPES)

def init_individual():
    # Terminal rates are per tick. Activate a random SUBSET of products
    # (each independently, at CFG.PRODUCT_ACTIVATION_PROB) instead of always
    # exactly one — the single-product-only initialization was the main
    # reason layouts stayed small and simple: with pwx previously never
    # touching this segment either (see below), a lineage's product mix was
    # frozen at whatever init_individual() drew for its entire evolutionary
    # history. At least one product is guaranteed active so the BOM is never
    # trivially empty.
    rates = [0.0] * n_mix
    active = [i for i in range(n_mix) if random.random() < CFG.PRODUCT_ACTIVATION_PROB]
    if not active:
        active = [random.randrange(n_mix)]
    for i in active:
        rates[i] = random.uniform(CFG.MIN_TERMINAL_RATE, CFG.MAX_TERMINAL_RATE)
    # No placement-priority tail: ordering is derived from the production graph
    # (see _order_instances). The genome is now product rates + recipe
    # multipliers only -- what to build, not how to arrange it.
    return creator.IndividualV4(rates +
        [random.uniform(CFG.MULT_INIT_MIN, CFG.MULT_INIT_MAX) for _ in range(n_rec)])

def pwx(a, b):
    # Uniform crossover across the WHOLE individual — product-mix rates and
    # recipe multipliers included, not just the layout-priority tail. Before,
    # "what to build and how much" could only change within a lineage via
    # slow Gaussian mutation; two individuals with different, independently-
    # successful product combinations could never recombine into a denser
    # offspring that builds both.
    # range(len(a)), not range(len(a) - 1): the old bound silently excluded
    # the final gene from crossover forever.
    for i in range(len(a)):
        if random.random() < .5: a[i], b[i] = b[i], a[i]
    return a, b

def custom_mutate(ind):
    for i in range(n_mix):
        ind[i] = min(CFG.MAX_TERMINAL_RATE, max(0., ind[i] + random.gauss(0, CFG.MUT_RATE_SIGMA)))
    for i in range(n_mix, len(ind)):
        ind[i] = min(1., max(0., ind[i] + random.gauss(0, CFG.MUT_GENE_SIGMA)))
    return ind,

toolbox.register("individual", init_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_nsga2)
toolbox.register("mate", pwx)
toolbox.register("mutate", custom_mutate)
toolbox.register("select", tools.selNSGA2)
print("GA configured: FitnessV4 weights", CFG.OBJ_WEIGHTS)


# ─── Selection-pressure helpers (see CFG.GA_MATING_SELECTION / GA_MAX_CLONES) ──
def _objective_key(ind):
    """Identity of an individual in OBJECTIVE space.

    Individuals sharing this key cannot dominate one another, so they all land
    on the same front. derive_rate_balanced_bom is a step function of the genome
    (ceil() rounding), so this happens a lot: a real 200-individual run carried
    about 9 distinct keys.
    """
    return tuple(round(v, 6) for v in ind.fitness.values)


def _mating_pool(pop, k):
    """NSGA-II's mating selection: binary tournament on (rank, crowding).

    This was missing entirely -- algorithms.varAnd() was handed the population
    directly, so every individual reproduced regardless of quality and the only
    selection pressure anywhere in the loop was the environmental step.
    selTournamentDCD needs crowding distances, which selNSGA2 assigns; the run
    loop therefore ranks the initial population before the first generation.
    """
    if len(pop) < 4:
        return [toolbox.clone(random.choice(pop)) for _ in range(max(2, k))]
    k = 4 * max(1, min(k, len(pop)) // 4)      # DCD consumes parents in fours
    return tools.selTournamentDCD(pop, k)


def _cap_duplicates(ranked, max_copies):
    """Split an NSGA-II-RANKED list into (kept, overflow), allowing at most
    `max_copies` individuals per objective vector in `kept`.

    Order is preserved, so `kept` holds the best representative of each distinct
    objective vector first. This is what stops a handful of designs from filling
    the whole first front and reducing environmental selection to crowding
    distance -- which rewards spread, not quality, and is why the best rate
    froze while the average collapsed toward the cheap-and-useless corner.
    """
    counts, kept, overflow = {}, [], []
    for ind in ranked:
        key = _objective_key(ind)
        seen = counts.get(key, 0)
        if seen < max_copies:
            counts[key] = seen + 1
            kept.append(ind)
        else:
            overflow.append(ind)
    return kept, overflow


def _fresh_individuals(n):
    """n newly sampled, evaluated individuals."""
    if n <= 0:
        return []
    newcomers = [toolbox.individual() for _ in range(n)]
    for ind, fit in zip(newcomers, list(toolbox.map(toolbox.evaluate, newcomers))):
        ind.fitness.values = fit
    return newcomers



In [ ]:
# ─── RUN NSGA-II ────────────────────────────────────────────────────────────────
def run_nsga2():
    """NSGA-II with the two pieces that were missing: mating selection, and a
    cap on how many identical designs may occupy the first front.

    Symptom this fixes, from a real 200x50 run: best rate frozen from generation
    4 to generation 50, average rate collapsing and then freezing, and all 200
    individuals on the first Pareto front -- 46 of 50 generations bought
    nothing, so the layouts Phase 4 went on to bomb were effectively the
    generation-4 population. See CFG.GA_MATING_SELECTION for the diagnosis.

    The per-generation line now reports `front1` and `distinct`, so a saturated
    front is visible while the run is happening instead of being inferred from a
    flat convergence plot afterwards.
    """
    print(f"Starting NSGA-II: Pop={POP_SIZE}, Gen={NGEN}...")
    pop = toolbox.population(n=POP_SIZE)

    fitnesses = list(toolbox.map(toolbox.evaluate, pop))
    for ind, fit in zip(pop, fitnesses):
        ind.fitness.values = fit

    # Rank the initial population so it carries the crowding distances
    # selTournamentDCD needs before the first mating round.
    pop = toolbox.select(pop, k=POP_SIZE)

    history = {'gen': [], 'best_cost': [], 'best_rate': [], 'best_mhc': [],
               'avg_rate': [], 'front1': [], 'distinct': [], 'fresh': []}

    stagnation_counter = 0
    best_rate_ever = -1.0
    improved_gens = 0
    champion = None
    champions_restored = 0
    fresh_budget = max(1, int(POP_SIZE * CFG.GA_FRESH_BLOOD))

    for gen in range(1, NGEN + 1):
        parents = (_mating_pool(pop, POP_SIZE) if CFG.GA_MATING_SELECTION else pop)
        offspring = algorithms.varAnd(parents, toolbox, cxpb=CXPB, mutpb=MUTPB)
        fits = list(toolbox.map(toolbox.evaluate, offspring))
        for ind, fit in zip(offspring, fits):
            ind.fitness.values = fit

        # Environmental selection. Rank everything first, then cap duplicates so
        # a handful of designs cannot occupy the whole front.
        ranked = toolbox.select(pop + offspring, k=len(pop) + len(offspring))
        if CFG.GA_MAX_CLONES > 0:
            kept, overflow = _cap_duplicates(ranked, CFG.GA_MAX_CLONES)
        else:
            kept, overflow = ranked, []

        new_pop = kept[:POP_SIZE]
        # Fresh blood for the slots the cap freed, up to a per-generation budget.
        # Bounded on purpose: refilling every freed slot at random would be a
        # restart each generation, not a fix.
        fresh = _fresh_individuals(min(fresh_budget, POP_SIZE - len(new_pop)))
        new_pop += fresh
        # Anything still short comes back from the ranked overflow, best first,
        # so population size and general character are preserved.
        new_pop += overflow[:max(0, POP_SIZE - len(new_pop))]
        pop = toolbox.select(new_pop, k=min(POP_SIZE, len(new_pop)))

        # Best-rate elitism. See CFG.GA_RATE_ELITISM: Pareto-elitism does not
        # protect any single objective once the front outgrows the population.
        if CFG.GA_RATE_ELITISM:
            gen_best = max(pop, key=lambda x: x.fitness.values[1])
            if champion is None or gen_best.fitness.values[1] > champion.fitness.values[1]:
                champion = toolbox.clone(gen_best)
            elif champion.fitness.values[1] > gen_best.fitness.values[1] + 1e-12:
                # The champion fell out of the population -- put it back over
                # the worst individual by rate, then re-rank so the crowding
                # distances the next mating round reads are consistent.
                worst_i = min(range(len(pop)), key=lambda i: pop[i].fitness.values[1])
                pop[worst_i] = toolbox.clone(champion)
                champions_restored += 1
                pop = toolbox.select(pop, k=len(pop))

        best = max(pop, key=lambda x: x.fitness.values[1])
        current_best_rate = best.fitness.values[1]

        if current_best_rate > best_rate_ever + 1e-4:
            best_rate_ever = current_best_rate
            stagnation_counter = 0
            improved_gens += 1
        else:
            stagnation_counter += 1

        if stagnation_counter >= CFG.GA_STAGNATION_GENS:
            print(f"  [AIS] Stagnation ({CFG.GA_STAGNATION_GENS} gens). "
                  f"Clonal hypermutation + fresh blood.")
            num_elites = max(5, int(POP_SIZE * 0.05))
            elites = tools.selNSGA2(pop, k=min(num_elites, len(pop)))

            clones = [toolbox.clone(ind) for ind in elites for _ in range(3)]
            for clone in clones:
                for _ in range(3):
                    toolbox.mutate(clone)
                del clone.fitness.values
            clone_fits = list(toolbox.map(toolbox.evaluate, clones))
            for ind, fit in zip(clones, clone_fits):
                ind.fitness.values = fit

            # Fresh randoms as well as elite clones. When the whole population
            # already sits on one front, a mutated copy of an elite is unlikely
            # to dominate anything either -- previous runs triggered this four
            # times and it changed nothing.
            newcomers = clones + _fresh_individuals(fresh_budget)

            ranked_pop = tools.selNSGA2(pop, k=len(pop))
            keep = max(0, len(pop) - len(newcomers))
            pop = toolbox.select(ranked_pop[:keep] + newcomers,
                                 k=min(POP_SIZE, keep + len(newcomers)))

            best = max(pop, key=lambda x: x.fitness.values[1])
            stagnation_counter = 0

        avg_rate = sum(x.fitness.values[1] for x in pop) / len(pop)
        front1 = tools.sortNondominated(pop, len(pop), first_front_only=True)[0]
        n_distinct = len({_objective_key(i) for i in pop})

        history['gen'].append(gen)
        history['best_cost'].append(best.fitness.values[0])
        history['best_rate'].append(best.fitness.values[1])
        history['best_mhc'].append(best.fitness.values[2])
        history['avg_rate'].append(avg_rate)
        history['front1'].append(len(front1))
        history['distinct'].append(n_distinct)
        history['fresh'].append(len(fresh))

        print(f"Gen {gen:2d}: Best Rate={best.fitness.values[1]:.4f}, "
              f"Cost={best.fitness.values[0]:.0f}, MHC={best.fitness.values[2]:.0f}"
              f"  |  Avg Rate={avg_rate:.4f}"
              f"  |  front1={len(front1)}/{len(pop)} distinct={n_distinct} "
              f"fresh=+{len(fresh)}")

    print("NSGA-II Complete!")
    # The number that says whether the budget was worth spending.
    print(f"  {improved_gens}/{NGEN} generation(s) improved the best rate; "
          f"final front1={history['front1'][-1]}/{POP_SIZE}, "
          f"distinct objectives={history['distinct'][-1]}")
    if champions_restored:
        print(f"  [elitism] the best-rate individual was dropped by crowding-distance "
              f"selection and restored {champions_restored} time(s). Without "
              f"CFG.GA_RATE_ELITISM those layouts would simply have been lost.")
    if history['front1'][-1] >= POP_SIZE:
        print("  [!] The first front still fills the population. Selection is then")
        print("      pure crowding distance, which rewards spread rather than")
        print("      quality. Raise CFG.GA_FRESH_BLOOD, lower CFG.GA_MAX_CLONES,")
        print("      or reconsider whether three correlated objectives (cost,")
        print("      rate, MHC) can be separated at this population size.")
    return pop, history

final_pop, conv_history = run_nsga2()

# selNSGA2(pop, k=len(pop)) returns the WHOLE population, NSGA-II-sorted -- it
# is a ranking, not a Pareto front. Keep both, named for what they are.
ranked_pop   = tools.selNSGA2(final_pop, k=len(final_pop))
pareto_front = tools.sortNondominated(final_pop, len(final_pop), first_front_only=True)[0]
print(f"Population={len(final_pop)}  |  true first Pareto front={len(pareto_front)}")


In [ ]:
# ─── NSGA-II CONVERGENCE & PARETO FRONT VISUALIZATION ───────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('NSGA-II Optimization Dashboard (v4)', fontsize=14, fontweight='bold')

# 1. Terminal-rate convergence (already folds the graded feasibility score in
#    — see evaluate_nsga2 — so this is the closest v4 analogue of the old
#    "Feasibility Convergence" + "Revenue Convergence" panels combined)
ax = axes[0, 0]
ax.plot(conv_history['gen'], conv_history['best_rate'], 'o-',
        color='#2980B9', linewidth=2, markersize=4, label='Best')
ax.plot(conv_history['gen'], conv_history['avg_rate'], 's--',
        color='#95A5A6', linewidth=1.5, markersize=3, label='Avg')
ax.set_xlabel('Generation')
ax.set_ylabel('Terminal Rate (feasibility-weighted)')
ax.set_title('Terminal Rate Convergence')
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# 2. Build cost convergence
ax = axes[0, 1]
ax.plot(conv_history['gen'], conv_history['best_cost'], 'o-',
        color='#8E44AD', linewidth=2, markersize=4)
ax.set_xlabel('Generation')
ax.set_ylabel('Build Cost')
ax.set_title('Build Cost Convergence (Best)')
ax.grid(alpha=0.3)

# 3. MHC convergence
ax = axes[1, 0]
ax.plot(conv_history['gen'], conv_history['best_mhc'], 'o-',
        color='#E67E22', linewidth=2, markersize=4)
ax.set_xlabel('Generation')
ax.set_ylabel('Material Handling Cost')
ax.set_title('MHC Convergence (Best)')
ax.grid(alpha=0.3)

# 4. True first Pareto front (Terminal Rate vs MHC, coloured by build cost).
#    This used to colour by ind._flp_cache['main_feas'], which is set inside
#    evaluate_nsga2 -- i.e. inside a joblib WORKER process. Only the fitness
#    tuple is returned to the parent, so the attribute never existed here and
#    getattr(..., {}) silently painted every point 0.0. Build cost is objective
#    0, so it is available on the fitness tuple itself and is always correct.
ax = axes[1, 1]
rate_vals = [ind.fitness.values[1] for ind in pareto_front]
mhc_vals  = [ind.fitness.values[2] for ind in pareto_front]
cost_vals = [ind.fitness.values[0] for ind in pareto_front]
sc = ax.scatter(rate_vals, mhc_vals, c=cost_vals, cmap='viridis_r',
                s=60, edgecolors='#2C3E50', linewidth=0.8)
plt.colorbar(sc, ax=ax, label='Build cost')
ax.set_xlabel('Terminal Rate')
ax.set_ylabel('MHC (intensity x belt cells)')
ax.set_title(f'First Pareto Front (n={len(pareto_front)})')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('v4_nsga2_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: v4_nsga2_dashboard.png")

In [ ]:
# ─── PHASE 2: RIGOROUS FEASIBILITY POLISH ───────────────────────────────────────
from types import SimpleNamespace
import traceback

print("=" * 70)
print("PHASE 2: Rigorous Feasibility Polish — Re-evaluating top candidates")
print("=" * 70)

# Drawn from the NSGA-II-RANKED population, not the first front alone -- the
# first front can be smaller than 20 individuals.
# De-duplicated by the LAYOUT each individual encodes, walking further down the
# ranked list to fill the quota. A real 100x30 run produced 20 candidates with
# only 9 distinct layouts and bombed 10 of which 5 were distinct -- half the
# Phase 2/4 budget was spent re-doing identical work.
#
# OVER-CAP FILTER. evaluate_nsga2 never evaluates an individual whose BOM
# exceeds MAX_MACHINES -- it returns a graded penalty and `raw_terminal_rate *
# 0.1` instead of a measured rate. That discounted figure is still large for an
# ambitious BOM, so over-cap individuals outrank properly evaluated ones on
# objective 1, which is the key this loop sorts by. See CFG.PHASE2_ENFORCE_CAP.
_seen_geno, candidates = set(), []
_over_cap = []                       # (bom_size, individual), rejected by the cap
for _ind in sorted(ranked_pop, key=lambda x: x.fitness.values[1], reverse=True):
    if len(candidates) >= CFG.PHASE2_CANDIDATES:
        break
    # chromosome_for() is called once here and its result reused for the
    # de-duplication key; genotype_key() would re-derive the same BOM.
    _instances, _chromo = chromosome_for(_ind)
    if CFG.PHASE2_ENFORCE_CAP and len(_instances) > MAX_MACHINES:
        _over_cap.append((len(_instances), _ind))
        continue
    _k = tuple((g.machine_id, g.recipe_idx) for g in _chromo.genes)
    if _k in _seen_geno:
        continue
    _seen_geno.add(_k)
    candidates.append(_ind)

if _over_cap:
    _sizes = sorted(n for n, _ in _over_cap)
    print(f"  {len(_over_cap)} individual(s) skipped: BOM over MAX_MACHINES="
          f"{MAX_MACHINES} (sizes {_sizes[0]}-{_sizes[-1]}). They were never")
    print( "    evaluated by the GA either -- their fitness is a penalty, not a"
           " measurement.")

if not candidates and _over_cap:
    # An empty Phase 2 breaks every phase after it, so fall back to the smallest
    # BOMs -- loudly. A population entirely over the cap means MAX_MACHINES or
    # the terminal-rate band is mis-set, not that these layouts are acceptable.
    _over_cap.sort(key=lambda item: item[0])
    candidates = [_i for _n, _i in _over_cap[:CFG.PHASE2_CANDIDATES]]
    print(f"  [!] EVERY ranked individual exceeds MAX_MACHINES={MAX_MACHINES}.")
    print(f"      Falling back to the {len(candidates)} smallest BOM(s) "
          f"({_over_cap[0][0]} genes and up) so the pipeline can continue.")
    print( "      These layouts are OUT OF SPEC: raise MAX_MACHINES, or lower")
    print( "      MIN/MAX_TERMINAL_RATE so the BOMs fit the cap.")

print(f"  {len(candidates)} distinct layouts selected from {len(ranked_pop)} individuals")
refined = []

for i, ind in enumerate(candidates):
    instances, chromo = chromosome_for(ind)

    # HEAVY REPAIR
    result = evaluate_layout(chromo, repair=True, alns_iters=CFG.ALNS_MAX_ITERS * 3, tabu_iters=CFG.TABU_MAX_ITERS * 2)

    # P7: rotation was fixed at R0 for every gene everywhere in this notebook.
    # A budgeted greedy pass over the machines carrying the most handling cost;
    # it keeps a rotation only if it strictly improves, so it cannot make a
    # candidate worse. CFG.ROTATION_POLISH_TOP = 0 disables it.
    _rot_before = _feasibility_ratio(result, len(instances)), result["mhc"]
    chromo, result = rotation_polish(chromo, result)
    if (_feasibility_ratio(result, len(instances)), result["mhc"]) != _rot_before:
        print(f"    rotation polish: feas {_rot_before[0]:.3f} -> "
              f"{_feasibility_ratio(result, len(instances)):.3f}, "
              f"MHC {_rot_before[1]:.0f} -> {result['mhc']:.0f}")

    # Pool conservation. Transfer through the logistics pool is free and
    # instant, so the only real constraint is that an INTERNAL material cannot
    # be withdrawn unless something deposited it. Nothing gated this and nothing
    # reported it, and a real run showed `utility_supply: deposited 0.0,
    # withdrawn 1.0` -- material appearing out of nowhere.
    _pbal = pool_balance(result["placed"], result["flows"], result["routed"])
    _pdef = {m: v for m, v in _pbal.items() if v["deficit"] > 1e-6}
    if _pdef:
        print("    [POOL] conservation deficit (withdrawn but never deposited): "
              + ", ".join(f"{m} {v['deficit']:.3f}/tick" for m, v in sorted(_pdef.items())))

    total_genes = len(instances)
    main_feas = _feasibility_ratio(result, total_genes)
    terminal_rate = result["revenue"] * main_feas

    refined.append((ind, result, terminal_rate, total_genes, chromo, main_feas))

    feas_tag = "FEASIBLE" if result["feasible"] else "PARTIAL"
    print(f"  Candidate {i+1:2d}: Rate={terminal_rate:.4f}, Cost={result['build_cost']:.0f}, "
          f"MHC={result['mhc']:.0f}, Feasibility={main_feas:.1%} [{feas_tag}]")
    print(f"                 gates: {_feas_str(result, total_genes)}  "
          f"(placed {len(result['placed'])}/{total_genes}, "
          f"routed {result['paths_found']}/{result['flows_total']})")

# Sort by feasible first, then by terminal rate
refined.sort(key=lambda x: _calculate_weighted_score(_feasibility_ratio(x[1], x[3]), x[2]), reverse=True)
n_feasible = sum(1 for r in refined if r[1]["feasible"])
print(f"\nRefinement complete: {n_feasible}/{len(candidates)} candidates are fully feasible.")



In [ ]:
# ─── connect_deep_polish() ───────────────────────────────────────────────────
def connect_deep_polish(grid, placed, flows, routed, rng=None, max_attempts=None, max_passes=None):
    """Deep connection pass for machines ALNS/Tabu couldn't fully connect.
    Never removes a machine -- the BOM is already sized to the minimum
    machines needed for the target product mix, so an unconnected machine is
    a missing link to fix, not dead weight to prune (that's what
    evaluate_layout's own machines_removed cleanup is for, separately).

    Retries A* across multiple (output-port, input-port) combinations on the
    SAME two machines for every unrouted flow, not just the one
    derive_flow_matrix originally assigned -- a different port pair often
    opens up a path the assigned one couldn't find, especially in congested
    layouts. Repeated across several passes, since a belt drawn in one pass
    can open up grid space that unblocks a different flow's path in a later
    pass, even with no machines ever being removed to cascade.

    This used to have a second stage: any revenue-producing machine with no
    routed outgoing flow got a nearby "stash" utility placed and belted to
    it as a generic sink, on the assumption that a sellable output needs an
    explicit outgoing flow to count. That assumption doesn't hold here --
    _calculate_revenue credits a machine's output purely from its own
    upstream supply efficiency, with no outgoing flow required at all. So
    that stage was "fixing" machines that were never broken, and the added
    stash machines/flows could themselves fail to route, dragging otherwise-
    fully-feasible layouts down (confirmed on a real run: Phase 3 turned a
    5/20-feasible population from Phase 2 into 0/20). Removed.

    IMPORTANT: this never calls derive_flow_matrix again -- re-deriving the
    supply/demand matching from scratch reshuffles ALL pairings (it's a
    greedy, order-dependent match), which could change which machines feed
    which and cascade into a very different flow graph than the one ALNS/
    Tabu actually placed for. Only the flows/routes this function itself
    discovers (retried connections) are added on top of what was already
    there -- every existing pairing is left exactly as ALNS/Tabu found it.

    Returns (grid, placed, flows, routed, n_stash_added) -- n_stash_added is
    kept in the signature for compatibility with existing call sites but is
    now always 0.
    """
    rng = rng if rng is not None else random
    max_attempts = CFG.RECONNECT_MAX_ATTEMPTS if max_attempts is None else max_attempts
    max_passes = CFG.RECONNECT_MAX_PASSES if max_passes is None else max_passes

    n_stash_added = 0
    current_routed = routed

    for _ in range(max_passes):
        placed_lookup = {p.instance_id: p for p in placed}
        made_progress = False

        patched_routed = []
        for fp, path in current_routed:
            if path is not None:
                patched_routed.append((fp, path))
                continue
            src_pm, dst_pm = placed_lookup.get(fp.src_id), placed_lookup.get(fp.dst_id)
            found_path = None
            if src_pm and dst_pm:
                outs, ins = src_pm.output_belt_points(), dst_pm.input_belt_points()
                port_pairs = [(o, i) for o in range(len(outs)) for i in range(len(ins))]
                rng.shuffle(port_pairs)
                for oi, ii in port_pairs[:max_attempts]:
                    candidate = astar(grid, outs[oi], ins[ii])
                    if candidate and grid.place_belt(candidate, f"{fp.src_id}->{fp.dst_id}"):
                        found_path = candidate
                        made_progress = True
                        break
            patched_routed.append((fp, found_path))

        current_routed = patched_routed
        if not made_progress:
            break

    flows = [fp for fp, _ in current_routed]
    return grid, placed, flows, current_routed, n_stash_added





# ─── Rip-up and reroute ───────────────────────────────────────────────────────
def _best_route(grid, src_pm, dst_pm, max_pairs=12):
    """Shortest A* path over up to `max_pairs` (out-port, in-port) combinations."""
    outs, ins = src_pm.output_belt_points(), dst_pm.input_belt_points()
    best, tried = None, 0
    for o in outs:
        for i in ins:
            p = astar(grid, o, i)
            if p is not None and (best is None or len(p) < len(best)):
                best = p
            tried += 1
            if tried >= max_pairs:
                return best
    return best


def ripup_and_reroute(grid, placed, routed, rng=None,
                      rounds=None, max_evict=None, max_pairs=12, verbose=False):
    """Let a blocked flow evict belts that are in its way, then re-lay them.

    Standard rip-up-and-reroute from detailed routing. Belts here are placed
    greedily and never removed, so the first flow to claim a corridor keeps it
    forever and a later flow with no alternative simply fails -- even when a
    small rearrangement would satisfy both.

    For each unrouted flow this evicts up to `max_evict` belts crossing the
    bounding box between its two machines, routes the blocked flow into the
    freed space, then re-lays the evicted ones. The move is kept ONLY if the
    total number of routed flows strictly increases, so the pass is monotone:
    it can never make routing worse.

    Returns (routed, n_gained).
    """
    rng = rng if rng is not None else random
    rounds    = CFG.RIPUP_ROUNDS    if rounds    is None else rounds
    max_evict = CFG.RIPUP_MAX_EVICT if max_evict is None else max_evict
    if rounds <= 0:
        return routed, 0

    lookup = {p.instance_id: p for p in placed}
    routed = list(routed)
    label  = lambda fp: f"{fp.src_id}->{fp.dst_id}"
    live   = lambda: [(label(f), p) for f, p in routed if p is not None]
    gained = 0

    for _ in range(rounds):
        progress = False
        for i, (fp, path) in enumerate(routed):
            if path is not None:
                continue
            src_pm, dst_pm = lookup.get(fp.src_id), lookup.get(fp.dst_id)
            if src_pm is None or dst_pm is None:
                continue

            x0, x1 = sorted((src_pm.x, dst_pm.x))
            y0, y1 = sorted((src_pm.y, dst_pm.y))
            evict = [j for j, (f2, p2) in enumerate(routed)
                     if p2 is not None and j != i
                     and any(x0 - 2 <= x <= x1 + 2 and y0 - 2 <= y <= y1 + 2 for x, y in p2)]
            if not evict:
                continue
            rng.shuffle(evict)
            evict = evict[:max_evict]

            keep = [(label(routed[j][0]), routed[j][1])
                    for j, (f2, p2) in enumerate(routed) if p2 is not None and j not in evict]
            grid.rebuild_belts(keep)

            new_path = _best_route(grid, src_pm, dst_pm, max_pairs)
            if new_path is None:
                grid.rebuild_belts(live())
                continue

            keep.append((label(fp), new_path))
            grid.rebuild_belts(keep)

            restored = []
            for j in evict:
                f2 = routed[j][0]
                s2, d2 = lookup.get(f2.src_id), lookup.get(f2.dst_id)
                p2 = _best_route(grid, s2, d2, max_pairs) if (s2 and d2) else None
                if p2 is not None:
                    keep.append((label(f2), p2))
                    grid.rebuild_belts(keep)
                    restored.append((j, p2))

            if len(restored) + 1 > len(evict):          # strictly better, or revert
                routed[i] = (fp, new_path)
                for j in evict:
                    routed[j] = (routed[j][0], None)
                for j, p2 in restored:
                    routed[j] = (routed[j][0], p2)
                gained += len(restored) + 1 - len(evict)
                progress = True
            else:
                grid.rebuild_belts(live())

        if not progress:
            break

    grid.rebuild_belts(live())
    if verbose and gained:
        print(f"  [RIPUP] recovered {gained} connection(s)")
    return routed, gained


print("ripup_and_reroute() defined -- blocked flows may evict and re-lay belts.")


def ensure_full_connectivity(grid, placed, flows, routed, rng=None, max_attempts=None):
    """Final connectivity sweep: guarantees every production machine has
    at least one incoming AND one outgoing belt.

    Skips utility-only machines (main_unit, depot_loader) that don't need
    bidirectional flow.  For every stranded machine it finds, it tries:
      1. Re-route an existing unrouted flow touching that machine
      2. For missing INPUT only: spawn a depot_unloader adjacent and route
         from it. (A missing OUTPUT does NOT get a spawned sink -- see the
         comment above Strategy B's old block below for why: revenue here
         doesn't require an explicit outgoing flow, so there was nothing to
         fix.)

    Returns (grid, placed, flows, routed, n_added).
    """
    import random as _rnd
    rng = rng if rng is not None else _rnd
    max_attempts = max_attempts or 20
    n_added = 0

    # Machines that don't need bidirectional flow
    EXEMPT_TYPES = {"main_unit", "depot_loader"}

    placed_lookup = {p.instance_id: p for p in placed}

    # Build connectivity maps from ROUTED (successful) flows only
    has_incoming = set()
    has_outgoing = set()
    for fp, path in routed:
        if path is not None:
            has_incoming.add(fp.dst_id)
            has_outgoing.add(fp.src_id)

    # Identify stranded machines
    stranded_no_input = []
    stranded_no_output = []
    for pm in placed:
        if pm.machine_type.id in EXEMPT_TYPES:
            continue
        if pm.machine_type.is_power_src:
            continue
        has_recipe = (pm.active_recipe is not None and pm.machine_type.recipes
                      and pm.active_recipe < len(pm.machine_type.recipes))

        # Depot unloaders / stashes are utility; they only need one direction
        if pm.machine_type.id == "depot_unloader":
            continue  # only needs outgoing
        if pm.machine_type.id == "stash":
            continue  # only needs incoming

        if has_recipe:
            recipe = pm.machine_type.recipes[pm.active_recipe]
            needs_input = len(recipe.inputs) > 0
            needs_output = len(recipe.outputs) > 0

            if needs_input and pm.instance_id not in has_incoming:
                stranded_no_input.append(pm)
            if needs_output and pm.instance_id not in has_outgoing:
                stranded_no_output.append(pm)

    print(f"  [CONNECTIVITY] {len(stranded_no_input)} machines missing INPUT, "
          f"{len(stranded_no_output)} machines missing OUTPUT")

    # --- Fix missing inputs: try re-routing, then spawn depot_unloader ---
    depot_type = MACHINE_CATALOGUE.get("depot_unloader")
    for pm in stranded_no_input:
        fixed = False

        # Strategy A: find an unrouted flow targeting this machine and re-route
        for idx, (fp, path) in enumerate(routed):
            if path is not None:
                continue
            if fp.dst_id != pm.instance_id:
                continue
            src_pm = placed_lookup.get(fp.src_id)
            if src_pm is None:
                continue
            outs = src_pm.output_belt_points()
            ins = pm.input_belt_points()
            pairs = [(o, i) for o in range(len(outs)) for i in range(len(ins))]
            rng.shuffle(pairs)
            for oi, ii in pairs[:max_attempts]:
                candidate = astar(grid, outs[oi], ins[ii])
                if candidate and grid.place_belt(candidate, f"{fp.src_id}->{fp.dst_id}"):
                    routed[idx] = (fp, candidate)
                    has_incoming.add(fp.dst_id)
                    has_outgoing.add(fp.src_id)
                    fixed = True
                    break
            if fixed:
                break

        # Strategy B: spawn a depot_unloader adjacent and route from it
        if not fixed and depot_type is not None:
            sw, sh = depot_type.size(Rotation.R0)
            cx, cy = pm.x + pm.w // 2, pm.y + pm.h // 2
            # depot_unloader is edge_only, so a +-8 sweep around an interior
            # machine produced only positions place_machine would reject --
            # Strategy B could essentially never fire and n_added stayed 0
            # (which also made the Phase 3 "5 passes" loop exit after one).
            # Search the grid EDGES, nearest-first, and keep a ranked list so a
            # rejected position falls through to the next candidate instead of
            # abandoning the machine.
            edge_positions = []
            for sx in range(0, grid.width - sw + 1):
                for sy in (0, grid.height - sh):
                    edge_positions.append((sx, sy))
            for sy in range(0, grid.height - sh + 1):
                for sx in (0, grid.width - sw):
                    edge_positions.append((sx, sy))
            candidates_ranked = sorted(
                {p for p in edge_positions
                 if grid.check_clearance(p[0], p[1], sw, sh, clearance=0)},
                key=lambda p: abs(p[0] - cx) + abs(p[1] - cy))[:12]
            for sx, sy in candidates_ranked:
                existing_ids = {p.instance_id for p in placed}
                didx = 0
                while f"depot_util_in_{didx}" in existing_ids:
                    didx += 1
                depot_id = f"depot_util_in_{didx}"
                depot_pm = PlacedMachine(machine_type=depot_type, x=sx, y=sy,
                                          rotation=Rotation.R0, instance_id=depot_id,
                                          active_recipe=None)
                if not grid.place_machine(depot_pm):
                    continue
                if True:
                    dep_outs = depot_pm.output_belt_points()
                    pm_ins = pm.input_belt_points()
                    routed_ok = False
                    for oi in range(len(dep_outs)):
                        for ii in range(len(pm_ins)):
                            candidate = astar(grid, dep_outs[oi], pm_ins[ii])
                            if candidate and grid.place_belt(candidate, f"{depot_id}->{pm.instance_id}"):
                                fp_new = FlowPair(src_id=depot_id, dst_id=pm.instance_id,
                                                   material="utility_supply", intensity=1.0,
                                                   src_node_idx=oi, dst_node_idx=ii)
                                routed.append((fp_new, candidate))
                                has_incoming.add(pm.instance_id)
                                placed.append(depot_pm)
                                placed_lookup[depot_id] = depot_pm
                                n_added += 1
                                routed_ok = True
                                fixed = True
                                break
                        if routed_ok:
                            break
                    if not routed_ok:
                        grid.remove_machine(depot_pm)
                    else:
                        break

        if fixed:
            print(f"    Fixed INPUT for {pm.instance_id}")

    # --- Fix missing outputs: try re-routing, then spawn stash ---
    stash_type = MACHINE_CATALOGUE.get("stash")
    for pm in stranded_no_output:
        # Skip if it was already fixed above (might now have output from input fix)
        if pm.instance_id in has_outgoing:
            continue
        fixed = False

        # Strategy A: find an unrouted flow from this machine and re-route
        for idx, (fp, path) in enumerate(routed):
            if path is not None:
                continue
            if fp.src_id != pm.instance_id:
                continue
            dst_pm = placed_lookup.get(fp.dst_id)
            if dst_pm is None:
                continue
            outs = pm.output_belt_points()
            ins = dst_pm.input_belt_points()
            pairs = [(o, i) for o in range(len(outs)) for i in range(len(ins))]
            rng.shuffle(pairs)
            for oi, ii in pairs[:max_attempts]:
                candidate = astar(grid, outs[oi], ins[ii])
                if candidate and grid.place_belt(candidate, f"{fp.src_id}->{fp.dst_id}"):
                    routed[idx] = (fp, candidate)
                    has_outgoing.add(fp.src_id)
                    has_incoming.add(fp.dst_id)
                    fixed = True
                    break
            if fixed:
                break

        # Strategy B (spawn a stash and route to it) used to run here for any
        # machine still missing an output after Strategy A. Removed: it
        # assumed a sellable output needs an explicit outgoing flow to
        # count for revenue, same wrong assumption connect_deep_polish's
        # old Stage 2 made -- _calculate_revenue credits output purely from
        # upstream supply efficiency, no outgoing flow required. Spawning a
        # stash "fixed" machines that were never broken, and the added
        # machine/flow could itself fail to route, net-negative for
        # feasibility. Missing OUTPUT now only gets Strategy A (re-route an
        # existing unrouted flow); if that doesn't find one, the machine is
        # left as-is.

        if fixed:
            print(f"    Fixed OUTPUT for {pm.instance_id}")

    flows = [fp for fp, _ in routed]
    print(f"  [CONNECTIVITY] Sweep complete: {n_added} utility machines added")
    return grid, placed, flows, routed, n_added

print("ensure_full_connectivity() defined — bidirectional belt enforcement.")


# ─── PHASE 3: CONNECTION DEEP POLISH ────────────────────────────────────────────
print("=" * 70)
print("PHASE 3: Connection Deep Polish -- one more, much deeper connection pass")
print("=" * 70)

polished = []
for entry in refined:
    ind, result, terminal_rate, total_genes, chromo, main_feas = entry

    grid, placed, flows, routed = result["grid"], result["placed"], result["flows"], result["routed"]
    # Seeded per individual so the polish pass is reproducible; both helpers
    # below otherwise fall back to the global `random` module.
    polish_rng = random.Random(individual_repair_seed(ind))
    n_stash_added = 0
    for _ in range(5):
        grid, placed, flows, routed, added1 = connect_deep_polish(
            grid, placed, flows, routed, rng=polish_rng,
            max_attempts=CFG.RECONNECT_MAX_ATTEMPTS, max_passes=CFG.RECONNECT_MAX_PASSES)
        # ── FULL CONNECTIVITY SWEEP ──────────────────────────────────────────────
        grid, placed, flows, routed, added2 = ensure_full_connectivity(
            grid, placed, flows, routed, rng=polish_rng,
            max_attempts=CFG.RECONNECT_MAX_ATTEMPTS)
        # Last resort: let still-blocked flows displace belts that are in the way.
        routed, _ripped = ripup_and_reroute(grid, placed, routed, rng=polish_rng,
                                            verbose=True)
        if _ripped:
            flows = [fp for fp, _ in routed]
        added_total = added1 + added2
        n_stash_added += added_total
        if added_total == 0:
            break

    paths_found = sum(path is not None for _, path in routed)
    mhc = sum(fp.intensity * (len(path) - 1) for fp, path in routed if path is not None)

    build_cost = sum(sum(q for _, q in pm.machine_type.build_cost) for pm in placed)
    demand = sum(pm.machine_type.power_cost for pm in placed)
    output = _power_output(placed, routed)
    power_ok = output > demand
    chain_time = _chain_time(placed, routed)
    sustain_ok = chain_time <= SIM_EQ_MAX_TICKS
    # all_placed carried over from Phase 2's own evaluate_layout, not
    # recomputed here. evaluate_layout computes it BEFORE its internal
    # cleanup step (which can remove machines that never got connected),
    # so len(placed) after that cleanup can be legitimately less than
    # total_genes even when all_placed was truly True -- recomputing it
    # as `len(placed) >= total_genes` disagreed with evaluate_layout's own
    # definition and flipped otherwise-unchanged, fully-feasible candidates
    # to infeasible for no real reason (confirmed on a real run: identical
    # power_ok/chain_time/paths before and after, only all_placed differed).
    # Safe to carry over unchanged: this function never adds or removes
    # machines, only routes.
    all_placed = result["all_placed"]
    feasible = all_placed and power_ok and sustain_ok
    if CFG.REQUIRE_FULL_ROUTING:
        feasible = feasible and paths_found == len(flows)
    revenue = _calculate_revenue(placed, routed)

    new_result = dict(grid=grid, placed=placed, flows=flows, routed=routed, mhc=mhc,
                       build_cost=build_cost, power_ok=power_ok, chain_time=chain_time,
                       feasible=feasible, paths_found=paths_found, flows_total=len(flows),
                       demand=demand, output=output, all_placed=all_placed, revenue=revenue,
                       machines_removed=result.get("machines_removed", 0), n_stash_added=n_stash_added)
    main_feas = _feasibility_ratio(new_result, total_genes)
    terminal_rate = new_result["revenue"] * main_feas
    polished.append((ind, new_result, terminal_rate, total_genes, chromo, main_feas))

polished.sort(key=lambda x: _calculate_weighted_score(_feasibility_ratio(x[1], x[3]), x[2]), reverse=True)
n_feasible = sum(1 for r in polished if r[1]["feasible"])
print(f"Phase 3 complete: {n_feasible}/{len(polished)} layout variants are fully feasible "
      f"after deep connection polish (was {sum(1 for r in refined if r[1]['feasible'])}/{len(refined)} after Phase 2).")

# ── Ranking report: make the Phase 3 -> Phase 4 cut explicit ─────────────────
# Candidates used to vanish silently here. Three separate mechanisms could drop
# one, none of which printed anything:
#   1. only the first CFG.PHASE4_TOP_N of CFG.PHASE2_CANDIDATES are bombed/plotted;
#   2. the ranking key is feasibility*WEIGHT_FEASIBILITY + terminal_rate*
#      WEIGHT_REVENUE, so feasibility does NOT strictly dominate -- a big enough
#      revenue lead outranks a higher-feasibility layout;
#   3. the saved filename depends on the BOOLEAN result["feasible"], not on the
#      graded percentage, so a 91%-feasible layout is still written as
#      "best_partial_N" rather than "top_N".
print()
print(f"{'Rank':>4} {'Feasibility':>12} {'TerminalRate':>13} {'Score':>9} {'Routed':>10} "
      f"{'Gate':>9}  {'place/route/power/sust':>26}  Bombed?")
print("-" * 104)
_cut = CFG.PHASE4_TOP_N
for _i, (_ind, _res, _tr, _tg, _ch, _mf) in enumerate(polished):
    _f = _feasibility_ratio(_res, _tg)
    _sc = _calculate_weighted_score(_f, _tr)
    _bomb = "yes" if _i < _cut else "NO  <- cut"
    _b = _feasibility_breakdown(_res, _tg)
    _gates = f"{_b['placement']:.2f}/{_b['routing']:.2f}/{_b['power']:.2f}/{_b['sustain']:.2f}"
    print(f"{_i+1:>4} {_f:>11.1%} {_tr:>13.4f} {_sc:>9.2f} "
          f"{str(_res['paths_found'])+'/'+str(_res['flows_total']):>10} "
          f"{'FEASIBLE' if _res['feasible'] else 'PARTIAL':>9}  {_gates:>26}  {_bomb}")
print("-" * 104)
print(f"  score = feasibility x {CFG.WEIGHT_FEASIBILITY} + terminal_rate x {CFG.WEIGHT_REVENUE}")
if len(polished) > _cut:
    _kept_min = min(_feasibility_ratio(e[1], e[3]) for e in polished[:_cut])
    _lost = [(_i+1, _feasibility_ratio(e[1], e[3]))
             for _i, e in enumerate(polished[_cut:], start=_cut)
             if _feasibility_ratio(e[1], e[3]) > _kept_min]
    if _lost:
        print(f"  [!] {len(_lost)} cut layout(s) have HIGHER feasibility than the weakest "
              f"one kept ({_kept_min:.1%}):")
        for _r, _fv in _lost:
            print(f"      rank {_r}: {_fv:.1%} -- outranked on revenue, not feasibility.")
        print("      Raise CFG.PHASE4_TOP_N, or lower CFG.WEIGHT_REVENUE, to keep them.")
    print(f"  {len(polished) - _cut} of {len(polished)} candidate(s) are NOT bombed and NOT plotted.")
print(f"  Figures are named from the boolean gate: FEASIBLE -> 'v4_layout_top_N_*', "
      f"PARTIAL -> 'v4_layout_best_partial_N_*'.")

refined = polished  # Phase 4 (ZPF bombing) reads from `refined`


In [ ]:
# ─── PHASE 4: ZPF DISASTER SIMULATION ───────────────────────────────────────────
print("=" * 70)
print("PHASE 4: ZPF Disaster Simulation — Bombing top layouts")
print("=" * 70)

# Dedupe again: Phase 3 can leave duplicates if two candidates converged, then
# drop anything that cannot power itself -- see CFG.PHASE4_REQUIRE_POWER.
_seen_layout, top_layouts, _unpowered = set(), [], []
for _entry in refined:
    if len(top_layouts) >= CFG.PHASE4_TOP_N:
        break
    _k = tuple((g.machine_id, g.recipe_idx) for g in _entry[4].genes)
    if _k in _seen_layout:
        continue
    _seen_layout.add(_k)
    _res = _entry[1]
    if CFG.PHASE4_REQUIRE_POWER and not _res.get("power_ok", True):
        _unpowered.append(_entry)
        continue
    top_layouts.append(_entry)

if _unpowered:
    _pr = lambda r: (r["output"] / r["demand"]) if r.get("demand") else 1.0
    print(f"  {len(_unpowered)} layout(s) skipped: generation does not meet demand "
          f"(power ratio {', '.join(f'{_pr(e[1]):.2f}' for e in _unpowered[:6])}). "
          f"CFG.PHASE4_REQUIRE_POWER=True.")
    print(f"  A brownout is not a recovery result -- these were previously bombed "
          f"and could report regret 0.0000 while the simulation produced nothing.")

# Fallback: never leave Phase 4 empty. If everything is unpowered, bomb the best
# of them and make the caveat impossible to miss in the log.
if len(top_layouts) < CFG.PHASE4_TOP_N and _unpowered:
    _need = CFG.PHASE4_TOP_N - len(top_layouts)
    _fill = _unpowered[:_need]
    top_layouts.extend(_fill)
    print(f"  [!] Only {len(top_layouts) - len(_fill)} powered layout(s) available; "
          f"filling {len(_fill)} slot(s) with UNPOWERED layouts to keep Phase 4 "
          f"populated. Their regret figures are not defensible on their own -- "
          f"cross-check against Phase 5.")

print(f"Bombing {len(top_layouts)} DISTINCT layout(s) of {len(refined)} polished candidates.")
print("Every scenario reports THREE states, not two:")
print("    pre        the layout before the bomb")
print("    post-bomb  the crater blocked and its machines gone, NOTHING recovered")
print("    post-rec   after the recovery search")
print("  regret = pre - post-rec        conflates how bad the bomb was with how")
print("                                 well the algorithm recovered")
print("  gain   = post-rec - post-bomb  what the recovery actually contributed")
print("  frac   = gain / (pre - post-bomb)   the same, normalised, so it is")
print("                                 comparable across layouts and damage levels")

for idx, (ind, result, terminal_rate, total_genes, chromo, main_feas) in enumerate(top_layouts):
    feas_tag = "FEASIBLE" if result["feasible"] else "PARTIAL"
    # One scenario set per LAYOUT, seeded from the layout itself so every
    # recovery algorithm faces exactly the same craters.
    _layout_seed = _chromosome_repair_seed(chromo)
    scenarios = make_bomb_scenarios(result["placed"], seed=_layout_seed)

    _pwr = (result["output"] / result["demand"]) if result.get("demand") else 1.0
    _pwr_tag = "" if result.get("power_ok", True) else f"  [UNPOWERED {_pwr:.2f}]"
    print(f"\n  Layout #{idx+1} [{feas_tag}]{_pwr_tag}  pre-bomb Rate={terminal_rate:.4f}, "
          f"Feasibility={main_feas:.1%}   [{_feas_str(result, total_genes)}]")
    # What that rate is actually made of, before anything is bombed.
    production_report(result["placed"], result["routed"],
                      title=f"Layout #{idx+1} PRE-BOMB", feasibility=main_feas,
                      indent="    ")
    check_raw_dependencies(result["placed"], result["routed"], indent="    ")
    print(f"    {len(scenarios)} scenario(s), rejection-sampled to guarantee damage")

    # The comparable baseline: the SAME layout, undamaged, re-routed on a fresh
    # grid exactly as everything downstream of the bomb is. The pre-bomb belts
    # were laid incrementally by blf_decode -> ALNS -> Tabu -> connect_or_remove
    # -> repair_feasibility; the recovery re-lays all of them at once. Comparing
    # a post-bomb rate to the pre-bomb rate mixes the crater with that
    # difference, and the difference is not small.
    ref = post_bomb_state(result, None)
    ref_feas = _feasibility_ratio(ref, total_genes)
    ref_rate = ref["revenue"] * ref_feas
    print(f"    clean re-route reference (no crater): Rate={ref_rate:.4f} "
          f"feas={ref_feas:.1%}   [{_feas_str(ref, total_genes)}]"
          f"   re-route effect {ref_rate - terminal_rate:+.4f}")

    runs = []
    for s_i, zone in enumerate(scenarios):
        destroyed, movable = scenario_damage(result["placed"], zone)

        # R1's missing baseline: the damaged layout with NO recovery applied,
        # built on the same preserved flow plan the recovery uses, so all three
        # states are the same graph and the differences mean something.
        pb = post_bomb_state(result, zone)
        pb_feas = _feasibility_ratio(pb, total_genes)
        pb_rate = pb["revenue"] * pb_feas

        rng = random.Random(_layout_seed ^ (s_i * 0x9E3779B1))
        zpf_result, _ = zpf_recovery(chromo, zone, iterations=CFG.ZPF_ITERATIONS,
                                     rng=rng, base=result)
        zfeas  = _feasibility_ratio(zpf_result, total_genes)
        zrate  = zpf_result["revenue"] * zfeas
        regret, gain, frac = recovery_metrics(terminal_rate, pb_rate, zrate,
                                              ref_rate=ref_rate)
        _rlog = zpf_result.get("recovery_log") or {}
        runs.append(dict(scenario=s_i, zone=zone, result=zpf_result, feas=zfeas,
                         reclog=_rlog, post_bomb=pb, pb_feas=pb_feas, pb_rate=pb_rate,
                         ref_rate=ref_rate,
                         rate=zrate, regret=regret, gain=gain, fraction=frac,
                         destroyed=destroyed, movable=movable,
                         disturbed=zpf_result.get("machines_disturbed", 0),
                         unplaced=len(zpf_result.get("unplaced_ids", []))))
        # The crater must be empty in BOTH downstream states. See
        # crater_violations for why this is asserted rather than assumed.
        for _state_name, _state in (("post-bomb", pb), ("post-recovery", zpf_result)):
            _bad_m, _bad_b = crater_violations(_state, zone)
            assert not _bad_m and not _bad_b, (
                f"crater violated in {_state_name} state, layout #{idx+1} "
                f"scenario {s_i+1} at ({zone.x},{zone.y}): "
                f"machines={_bad_m} belts={_bad_b}")

        _frac_s = "  n/a" if frac != frac else f"{frac:5.1%}"
        print(f"      #{s_i+1:>2} crater({zone.x:>2},{zone.y:>2})  destroyed={destroyed} "
              f"movable={movable:>3}  rate {terminal_rate:7.4f} -> {pb_rate:7.4f} -> {zrate:7.4f}"
              f"  regret={regret:7.4f}  gain={gain:+7.4f}  frac={_frac_s}"
              f"  disturbed={runs[-1]['disturbed']}")

    regrets = [r["regret"] for r in runs]
    gains   = [r["gain"] for r in runs]
    fracs   = [r["fraction"] for r in runs if r["fraction"] == r["fraction"]]
    mean_r  = sum(regrets) / len(regrets)
    mean_g  = sum(gains) / len(gains)
    mean_f  = (sum(fracs) / len(fracs)) if fracs else float("nan")
    worst   = max(runs, key=lambda r: r["regret"])
    median  = sorted(regrets)[len(regrets) // 2]
    zero    = sum(1 for r in regrets if r <= 1e-9)
    # A bomb that IMPROVES the layout is a modelling defect, not a result. It
    # used to be invisible because regret clamps at zero; counted explicitly
    # now, since it is the single most informative diagnostic in this phase.
    helped  = sum(1 for r in runs if r["pb_rate"] > ref_rate + 1e-9)
    print(f"    REGRET over {len(runs)} scenario(s): mean={mean_r:.4f}  median={median:.4f}  "
          f"worst={worst['regret']:.4f} (crater {worst['zone'].x},{worst['zone'].y})  "
          f"zero-regret={zero}/{len(runs)}")
    _mf = "n/a" if mean_f != mean_f else f"{mean_f:.1%}"
    print(f"    RECOVERY: mean gain={mean_g:+.4f}  mean recovered fraction={_mf}"
          f"   (scenarios where the BOMB IMPROVED the layout: {helped}/{len(runs)})")
    # The same ledger for the state Phase 5/6 carry forward, so the drop in
    # rate can be attributed to a material rather than left as a delta.
    production_report(worst["result"]["placed"], worst["result"]["routed"],
                      title=f"Layout #{idx+1} POST-RECOVERY "
                            f"(worst crater {worst['zone'].x},{worst['zone'].y})",
                      feasibility=worst["feas"], indent="    ")
    print(f"    damage: mean destroyed="
          f"{sum(r['destroyed'] for r in runs)/len(runs):.2f}, "
          f"mean movable={sum(r['movable'] for r in runs)/len(runs):.1f}")

    # Recovery EFFORT, averaged over the scenario set. Final quality alone does
    # not separate two methods that reach the same layout at different cost.
    _logs = [r["reclog"] for r in runs if r.get("reclog")]
    if _logs:
        _ev   = sum(l.get("evaluations", 0)    for l in _logs) / len(_logs)
        _etb  = sum(l.get("evals_to_best", 0)  for l in _logs) / len(_logs)
        _dist = [l.get("distinct_costs") for l in _logs if l.get("distinct_costs")]
        _line = (f"    effort: method={_logs[0].get('method','?')}  "
                 f"mean evaluations={_ev:.0f}  mean evals-to-best={_etb:.0f}  "
                 f"({(1 - _etb / max(1e-9, _ev)):.0%} of the budget bought nothing)")
        if _dist:
            # How much of the budget saw a layout it had never seen before. A
            # permutation neighbourhood over machines that all go straight back
            # to their original coordinates re-evaluates the same layout almost
            # every iteration; this is the number that exposes it.
            _line += (f"\n            distinct costs={sum(_dist)/len(_dist):.1f}"
                      f" of {_ev:.0f} evaluations")
        if "acceptance_rate" in _logs[0]:
            _ar = sum(l["acceptance_rate"] for l in _logs) / len(_logs)
            _ir = sum(l["improvement_rate"] for l in _logs) / len(_logs)
            _line += f"\n            acceptance={_ar:.1%}  improvement={_ir:.1%}"
        print(_line)

    # The WORST case is carried forward for Phase 5/6 -- a resilience claim
    # should be made against the hardest scenario, not a lucky one.
    zpf_result, zpf_feas, regret = worst["result"], worst["feas"], worst["regret"]
    ind._flp_cache = dict(result=result, zpf=zpf_result, terminal_rate=terminal_rate,
                           zone=worst["zone"],   # the crater that PRODUCED zpf_result
                           worst_run=worst,
                           post_bomb=worst["post_bomb"], pb_feas=worst["pb_feas"],
                           pb_rate=worst["pb_rate"], ref_rate=ref_rate,
                           ref_feas=ref_feas,
                           main_feas=main_feas, zpf_feas=zpf_feas, total_genes=total_genes,
                           runs=runs, mean_regret=mean_r, worst_regret=worst["regret"],
                           median_regret=median, mean_gain=mean_g, mean_fraction=mean_f,
                           bomb_helped=helped)

# ── Cross-layout summary ─────────────────────────────────────────────────────
print("\n" + "=" * 96)
print("PHASE 4 SUMMARY — regret and recovery distribution per layout")
print("=" * 96)
print(f"{'Layout':>8}{'scen':>6}{'pre-rate':>10}{'post-bomb':>11}{'mean reg':>10}"
      f"{'worst reg':>11}{'mean gain':>11}{'mean frac':>11}{'zero':>7}{'destroyed':>11}")
for idx, (ind, result, terminal_rate, *_rest) in enumerate(top_layouts):
    ch = getattr(ind, "_flp_cache", {})
    rs = ch.get("runs") or []
    if not rs:
        continue
    _pb = sum(r["pb_rate"] for r in rs) / len(rs)
    _mf = ch.get("mean_fraction", float("nan"))
    _mfs = "n/a" if _mf != _mf else f"{_mf:>10.1%}"
    print(f"{'#'+str(idx+1):>8}{len(rs):>6}{terminal_rate:>10.4f}{_pb:>11.4f}"
          f"{ch['mean_regret']:>10.4f}{ch['worst_regret']:>11.4f}"
          f"{ch['mean_gain']:>+11.4f}{_mfs:>11}"
          f"{sum(1 for r in rs if r['regret'] <= 1e-9):>4}/{len(rs):<2}"
          f"{sum(r['destroyed'] for r in rs)/len(rs):>11.2f}")
_helped_total = sum((getattr(i, '_flp_cache', {}) or {}).get('bomb_helped', 0)
                    for i, *_ in top_layouts)
if _helped_total:
    print(f"\n  [!] {_helped_total} scenario(s) where the UN-RECOVERED post-bomb layout scored")
    print( "      HIGHER than the CLEAN RE-ROUTE REFERENCE. That is a modelling defect --")
    print( "      destroying machines must not improve the factory. Check the sustain gate")
    print( "      (CFG.SUSTAIN_NORMALISE), the routing denominator (_zpf_flow_demand) and")
    print( "      the revenue attribution (CFG.REVENUE_AT_SINK_ONLY).")
print("  Phase 5/6 use the WORST scenario for each layout.")
_n_checked = sum(len((getattr(i, '_flp_cache', {}) or {}).get('runs') or ())
                 for i, *_ in top_layouts)
print(f"  Crater integrity: {2 * _n_checked} state(s) checked, 0 violations "
      f"(asserted per scenario -- see crater_violations).")


## Phase 5 — Discrete-time simulation (production timeline)

The static pipeline scores a layout geometrically: belt lengths, a topological
critical-path estimate (`_chain_time`), and an analytical revenue figure derived
from recipe rates and supply-chain efficiency. None of that observes a machine
actually *running*.

`SimEngine` steps the factory tick by tick — material moves along belts at one
cell per tick, machines consume from their input buffers, run for
`recipe.process_time` ticks, and emit into their output buffers — so it can show
the two things the static model cannot:

- **which machines starve, and when** (Gantt chart)
- **whether production actually reaches a steady state** within the tick budget,
  which is what Gate 2 claims to test

Each of the top `CFG.SIM_TOP_N` layouts is simulated twice: once on the
pre-bomb layout and once on the Phase 4 post-recovery layout. The paired charts
are the direct evidence of how much throughput the disruption cost and how much
the recovery search won back.

Raw material is injected **only** where a belt from `main_unit` / a depot
actually lands (see `build_injection_map`), so a machine whose real supply belt
failed to route will visibly starve rather than being quietly topped up.


In [ ]:
# ─── PHASE 5: DISCRETE-TIME SIMULATION — PRE-BOMB vs POST-RECOVERY ─────────────
print("=" * 70)
print("PHASE 5: Discrete-time simulation — production timeline & throughput")
print("=" * 70)

sim_top     = refined[:CFG.SIM_TOP_N]
sim_records = []

for idx, (ind, result, terminal_rate, total_genes, chromo, main_feas) in enumerate(sim_top):
    label      = f"Layout #{idx + 1}"
    zpf_result = getattr(ind, "_flp_cache", {}).get("zpf")
    print(f"\n--- {label} ---")

    pre_engine,  pre_eq  = simulate_layout(result, label=f"{label} PRE-BOMB")
    post_engine, post_eq = (None, None)
    if zpf_result is not None:
        post_engine, post_eq = simulate_layout(zpf_result, label=f"{label} POST-RECOVERY")

    pre_rev  = pre_engine.simulated_revenue_per_tick(window=CFG.SIM_EQ_WINDOW)
    post_rev = (post_engine.simulated_revenue_per_tick(window=CFG.SIM_EQ_WINDOW)
                if post_engine is not None else 0.0)

    # Regret measured on SIMULATED throughput rather than the analytical rate.
    sim_regret = max(0.0, pre_rev - post_rev)

    # Why the simulation produced what it produced. The old agreement check
    # compared equilibrium TIMING only, so a layout with an analytic rate of
    # 7.0000 and 0.000 simulated revenue/tick was reported as agreeing.
    pre_diag  = diagnose_simulation(pre_engine, pre_eq, terminal_rate)
    post_diag = (diagnose_simulation(post_engine, post_eq, terminal_rate)
                 if post_engine is not None else None)
    print("  [SIM] " + explain_simulation(pre_diag, f"{label} PRE-BOMB: "))
    if post_diag is not None:
        print("  [SIM] " + explain_simulation(post_diag, f"{label} POST-RECOVERY: "))

    sim_records.append(dict(
        label=label, idx=idx,
        pre_engine=pre_engine, pre_eq=pre_eq, pre_rev=pre_rev,
        post_engine=post_engine, post_eq=post_eq, post_rev=post_rev,
        sim_regret=sim_regret,
        static_chain_time=result["chain_time"],
        static_sustain_ok=result["chain_time"] <= SIM_EQ_MAX_TICKS,
        sim_sustain_ok=(pre_eq is not None and pre_eq <= SIM_EQ_MAX_TICKS),
        analytic_rate=terminal_rate,
        pre_diag=pre_diag, post_diag=post_diag,
    ))

    # ── Paired Gantt: pre-bomb above, post-recovery below ─────────────────────
    if HAS_MPL:
        engines = [(pre_engine, pre_eq, "PRE-BOMB", pre_rev)]
        if post_engine is not None:
            engines.append((post_engine, post_eq, "POST-RECOVERY", post_rev))

        rows = max(len(e.states) for e, _, _, _ in engines)
        fig, axes_g = plt.subplots(len(engines), 1, squeeze=False,
                                   figsize=(15, max(3.5, rows * 0.26) * len(engines)))
        for row_i, (eng, eq, tag, rev) in enumerate(engines):
            plot_gantt(eng, eq_tick=eq, ax=axes_g[row_i][0],
                       max_ticks=CFG.SIM_MAX_TICKS,
                       title=f"{label} — {tag}   "
                             f"(machines={len(eng.states)}, "
                             f"sim revenue/tick={rev:.3f}, eq_tick={eq})")
        plt.tight_layout()
        plt.savefig(f"v4_sim_gantt_layout_{idx + 1}.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  Saved v4_sim_gantt_layout_{idx + 1}.png")

        # ── Paired throughput curves ─────────────────────────────────────────
        fig, axes_t = plt.subplots(1, len(engines), squeeze=False,
                                   figsize=(8 * len(engines), 4.5))
        for col_i, (eng, eq, tag, rev) in enumerate(engines):
            plot_throughput(eng, ax=axes_t[0][col_i],
                            title=f"{label} — {tag}  (rev/tick={rev:.3f})")
        plt.tight_layout()
        plt.savefig(f"v4_sim_throughput_layout_{idx + 1}.png", dpi=150, bbox_inches="tight")
        plt.show()
        print(f"  Saved v4_sim_throughput_layout_{idx + 1}.png")


# ── Summary: analytical model vs simulation ───────────────────────────────────
print("\n" + "=" * 78)
print("PHASE 5 SUMMARY — analytical estimate vs simulated throughput")
print("=" * 78)
print(f"{'Layout':10s} {'AnalyticRate':>13s} {'SimRev/tick':>12s} "
      f"{'PostRev/tick':>13s} {'SimRegret':>10s} {'Gate2 static':>13s} {'Gate2 sim':>10s} "
      f"  {'cause':<11s}")
print("-" * 96)
for r in sim_records:
    print(f"{r['label']:10s} {r['analytic_rate']:13.4f} {r['pre_rev']:12.4f} "
          f"{r['post_rev']:13.4f} {r['sim_regret']:10.4f} "
          f"{str(r['static_sustain_ok']):>13s} {str(r['sim_sustain_ok']):>10s} "
          f"  {r['pre_diag']['cause']:<11s}")

# ── Check 1: equilibrium TIMING (what this section used to check, alone) ──────
disagree = [r for r in sim_records if r["static_sustain_ok"] != r["sim_sustain_ok"]]
if disagree:
    print(f"\n[!] {len(disagree)}/{len(sim_records)} layouts where the static Gate 2 "
          f"(_chain_time <= {SIM_EQ_MAX_TICKS}) and the simulation DISAGREE ON TIMING.")
    print("    The static gate is a critical-path estimate over the flow DAG; it")
    print("    assumes every machine is fed. The simulation does not. Where they")
    print("    differ, trust the simulation and treat the static gate as optimistic.")
else:
    print(f"\n[OK] Static Gate 2 and the simulation agree on TIMING for every layout.")

# ── Check 2: THROUGHPUT. This is the one that was missing. ───────────────────
# A layout can pass the timing check and produce nothing whatsoever: observed on
# a real run, analytic rates of 7.0000 and 6.2000 against 0.000 simulated
# revenue/tick, reported as "[OK] ... agree on every simulated layout".
dead  = [r for r in sim_records if not r["pre_diag"]["produced"]]
short = [r for r in sim_records
         if r["pre_diag"]["produced"] and r["pre_diag"]["cause"] != "none"]

if dead:
    print(f"\n[!!] {len(dead)}/{len(sim_records)} layouts produce NOTHING in simulation "
          f"while the analytical model reports a positive rate.")
    print("     The analytical rate is revenue x feasibility over the flow DAG. It")
    print("     assumes every routed flow delivers and every machine is powered;")
    print("     neither is checked. Where the simulation produces nothing, the")
    print("     analytical rate for that layout is not a throughput estimate.")
    for r in dead:
        print(f"     - {r['label']}: analytic {r['analytic_rate']:.4f} -> sim 0.000  |  "
              + explain_simulation(r["pre_diag"]))
# Reported separately rather than chained onto the block above: a run can hold
# BOTH a layout that produces nothing and one that merely under-produces, and an
# elif hides the second -- the same class of omission as the original defect.
if short:
    print(f"\n[!] {len(short)}/{len(sim_records)} layouts produce, but below "
          f"{CFG.SIM_AGREE_FRACTION:.0%} of the analytical rate.")
    for r in short:
        print(f"     - {r['label']}: " + explain_simulation(r["pre_diag"]))
if not dead and not short:
    print("[OK] Every simulated layout reaches at least "
          f"{CFG.SIM_AGREE_FRACTION:.0%} of its analytical rate.")

# ── Cause breakdown, so two different failures are never conflated ────────────
_causes = {}
for r in sim_records:
    _causes[r["pre_diag"]["cause"]] = _causes.get(r["pre_diag"]["cause"], 0) + 1
if set(_causes) - {"none"}:
    print("\n  Failure causes across the simulated layouts: "
          + ", ".join(f"{k}={v}" for k, v in sorted(_causes.items()) if k != "none"))
    print("    power      -- the factory cannot pay its own draw (see P5 in the register)")
    print("    starvation -- machines are fed too slowly, or never run at all")
    print("    no-supply  -- nothing is injected at the boundary")
    print("    throughput -- produces, but well under the analytical claim")

if HAS_MPL and sim_records:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    xs   = range(len(sim_records))
    ax.bar([x - 0.2 for x in xs], [r["pre_rev"] for r in sim_records],
           width=0.4, label="Pre-bomb", color="#27AE60")
    ax.bar([x + 0.2 for x in xs], [r["post_rev"] for r in sim_records],
           width=0.4, label="Post-recovery", color="#E67E22")
    ax.set_xticks(list(xs))
    ax.set_xticklabels([r["label"] for r in sim_records])
    ax.set_ylabel("Simulated revenue / tick (at equilibrium)")
    ax.set_title("Simulated throughput: disruption impact and recovery")
    ax.legend()
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig("v4_sim_regret_summary.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved v4_sim_regret_summary.png")


In [ ]:
# ─── VISUALIZATION ──────────────────────────────────────────────────────────────
# The crater is per-layout, not per-config: Phase 4 samples ZPF_SCENARIOS
# positions per layout and carries the WORST one into the cache. Drawing
# CFG.ZPF_X/ZPF_Y here put the rectangle and rings somewhere the bomb never
# landed, which is why recovered layouts appeared to have machines and belts
# sitting inside the crater.
_ZPF_DEFAULT = ZPF(CFG.ZPF_X, CFG.ZPF_Y, CFG.ZPF_W, CFG.ZPF_H, CFG.ZPF_R1, CFG.ZPF_R2)

for idx, (ind, result, terminal_rate, total_genes, chromo, main_feas) in enumerate(top_layouts):
    cache = ind._flp_cache
    zpf_result, zpf_feas = cache["zpf"], cache["zpf_feas"]
    zpf_zone_for_plot = cache.get("zone") or _ZPF_DEFAULT

    label = f"Top {idx+1}" if result["feasible"] else f"Best Partial {idx+1}"
    _w = cache.get("worst_run") or {}
    _frac = _w.get("fraction", float("nan"))
    _frac_s = "n/a" if _frac != _frac else f"{_frac:.0%}"
    _scenario_note = (
        f"{label} — worst of {len(cache.get('runs') or [1])} scenarios  |  "
        f"crater {zpf_zone_for_plot.w}x{zpf_zone_for_plot.h} at "
        f"({zpf_zone_for_plot.x},{zpf_zone_for_plot.y})  "
        f"r1={zpf_zone_for_plot.r1:g} r2={zpf_zone_for_plot.r2:g}  |  "
        f"destroyed={_w.get('destroyed', '?')}  "
        f"movable={_w.get('movable', '?')}  "
        f"relocated={_w.get('disturbed', '?')}  |  "
        f"regret={cache.get('worst_regret', 0.0):.4f} "
        f"(mean {cache.get('mean_regret', 0.0):.4f})  "
        f"recovered {_frac_s} of what the bomb cost"
    )
    print(f"  {label}: {_scenario_note}")
    fname_base = f"v4_layout_{label.lower().replace(' ', '_')}"

    # --- Multi-panel: Pre-Bomb | Post-Bomb (un-recovered) | Post-Recovery ---
    # The middle panel is the SAME object Phase 4 scored, not a fresh
    # reconstruction. It used to be rebuilt here with derive_flow_matrix() from
    # scratch, which re-runs a greedy order-dependent match on the surviving
    # machines: the panel showed a different supply plan, so the drop it
    # depicted was partly a re-match rather than the damage. See P8 and
    # post_bomb_state().
    try:
        post_bomb_result = cache.get("post_bomb")
        if post_bomb_result is None:
            post_bomb_result = post_bomb_state(result, zpf_zone_for_plot)
        post_bomb_feas = cache.get("pb_feas")
        if post_bomb_feas is None:
            post_bomb_feas = _feasibility_ratio(post_bomb_result, total_genes)
        post_bomb_rate = cache.get("pb_rate", post_bomb_result["revenue"] * post_bomb_feas)

        plot_zpf_sequence(
            pre_result=result,
            post_result=post_bomb_result,
            rec_result=zpf_result,
            zpf_zone=zpf_zone_for_plot,
            pre_feas=main_feas,
            post_feas=post_bomb_feas,
            rec_feas=zpf_feas,
            pre_rate=terminal_rate,
            post_rate=post_bomb_rate,
            rec_rate=zpf_result["revenue"] * zpf_feas,
            scenario_note=_scenario_note,
            save_path=f"{fname_base}_sequence.png",
        )
    except Exception:
        print(f"\n[MATPLOTLIB EXCEPTION IN plot_zpf_sequence for {label}]:")
        traceback.print_exc()


---

# APPENDIX — LEGACY (not part of the pipeline)

Everything below this line is **retained for reference only**. Nothing in
Sections 1–17 calls any of it, verified by call-graph reachability from the
cells that actually execute (`run_nsga2` through Phase 6).

It is kept because it documents the earlier v3 evaluation architecture that
this notebook's `evaluate_layout` replaced, and because the thesis text refers
to some of it. **Running these cells is optional and changes nothing.**

| Symbol | Superseded by |
|---|---|
| `FitnessResult`, `calculate_metrics`, `evaluate_fitness` | `evaluate_layout()` (Section 12) |
| `cull_unlinked_machines` | `connect_or_remove_unlinked()` |
| `evaluate_with_simulation`, `evaluate_search_fitness` | `simulate_layout()` (Phase 5) |
| `greedy_rotation_search` | nothing — rotation is fixed at `R0` throughout the v4 pipeline |
| `ProductSpec`, `ALL_PRODUCTS` | `derive_rate_balanced_bom()` + `TARGET_PRODUCTS` |
| `print_grid` | `plot_layout()` |
| `build_belt_paths`, `find_dag_roots` | `build_belt_paths_from_routed()`, `build_injection_map()` |

Two of these are worth flagging explicitly rather than just deleting:

- **`greedy_rotation_search` is unused, and that is a real modelling limitation.**
  Every chromosome in the v4 pipeline is built as `Gene(m.id, Rotation.R0, r)`,
  in `evaluate_nsga2`, in Phase 2, and in `zpf_recovery`. Since `Rotation`
  decides which face carries the I/O ports, machine orientation is a dimension
  of the layout problem that this notebook does not search. State that
  explicitly in the write-up rather than implying otherwise.
- **`evaluate_with_simulation` claimed Gate 2 was simulation-backed.** It never
  ran. Gate 2 in the live pipeline is the static `_chain_time()` critical-path
  DP; the simulation now runs in Phase 5 as post-hoc validation, and Phase 5
  reports where the two disagree.


## 10. Fitness Evaluation (3-Gate Model)

| Gate | Condition |
|---|---|
| **Gate 1 — Power** | `battery_power_output ≥ total_power_demand` |
| **Gate 2 — Self-sustaining** | `chain_time ≤ _sustain_budget(chain_ideal)`, which is `CFG.SUSTAIN_SLACK × chain_ideal` when `CFG.SUSTAIN_NORMALISE` is on and `SIM_EQ_MAX_TICKS` (300) otherwise |
| **Gate 3 — All placed** | Every gene in the chromosome was decoded to a placed machine |

The **chain time** is computed as the longest path through the production DAG
(topological sort + DP on arrival times, including belt travel time at `1 tick/cell`).


In [ ]:
from collections import defaultdict, deque

@dataclass
class FitnessResult:
    mhc:                  float
    revenue:              float
    build_cost:           dict[str, int]
    all_placed:           bool
    paths_found:          int
    paths_total:          int
    belt_cells_used:      int
    total_power_demand:   float
    battery_power_output: float
    power_feasible:       bool
    chain_time:           float
    startup_time:         float
    self_sustaining:      bool
    belt_budget_used:     float
    feasible:             bool

def cull_unlinked_machines(
    grid: FactoryGrid, placed: list[PlacedMachine], belt_results: list[tuple[FlowPair, Optional[list[tuple[int, int]]]]]
) -> list[PlacedMachine]:
    """Removes machines that have no successful belt connections.
    Exempts main_unit and ther_bank (power sources)."""
    connected_ids = set()
    for fp, path in belt_results:
        if path is not None:
            connected_ids.add(fp.src_id)
            connected_ids.add(fp.dst_id)

    culled_placed = []
    # Work on a copy of the list so we can remove safely
    for pm in list(placed):
        # Never cull the main unit or power sources
        if pm.machine_type.id == "main_unit" or pm.machine_type.is_power_src:
            culled_placed.append(pm)
            continue

        if pm.instance_id in connected_ids:
            culled_placed.append(pm)
        else:
            grid.remove_machine(pm)

    return culled_placed

def calculate_metrics(
    grid: FactoryGrid,
    placed: list[PlacedMachine],
    flow_matrix: list[FlowPair],
    belt_results: list[tuple[FlowPair, Optional[list[tuple[int, int]]]]],
    chromosome_len: int,
    battery_item: str = 'SC_bat'
) -> FitnessResult:
    placed_lookup = {pm.instance_id: pm for pm in placed}
    proc_time: dict[str, int] = {}
    for pm in placed:
        if pm.active_recipe is not None and pm.machine_type.recipes:
            if pm.active_recipe < len(pm.machine_type.recipes):
                proc_time[pm.instance_id] = pm.machine_type.recipes[pm.active_recipe].process_time
            else:
                proc_time[pm.instance_id] = 0
        else:
            proc_time[pm.instance_id] = 0

    path_cache: dict[tuple[str, str], int] = {}
    mhc = 0.0; paths_found = 0; belt_cells_used = 0

    for fp, path in belt_results:
        if path is not None:
            dist = len(path) - 1
            mhc += fp.intensity * dist
            paths_found    += 1
            belt_cells_used += len(path)
            path_cache[(fp.src_id, fp.dst_id)] = dist
        else:
            src_pm = placed_lookup.get(fp.src_id)
            dst_pm = placed_lookup.get(fp.dst_id)
            if src_pm and dst_pm:
                dist = (abs(src_pm.x + src_pm.w//2 - dst_pm.x - dst_pm.w//2)
                      + abs(src_pm.y + src_pm.h//2 - dst_pm.y - dst_pm.h//2))
                mhc += fp.intensity * dist * 2
                path_cache[(fp.src_id, fp.dst_id)] = dist * 2
            else:
                penalty = grid.width + grid.height
                mhc += fp.intensity * penalty
                path_cache[(fp.src_id, fp.dst_id)] = penalty

    # 4. Revenue
    revenue = 0.0
    for pm in placed:
        if pm.active_recipe is not None and pm.machine_type.recipes:
            if pm.active_recipe < len(pm.machine_type.recipes):
                r = pm.machine_type.recipes[pm.active_recipe]
                for out_name, out_qty in r.outputs:
                    if out_name in REVENUE_TABLE:
                        revenue += REVENUE_TABLE[out_name] * (out_qty / max(r.process_time, 1))

    # 5. Build cost
    build_cost: dict[str, int] = {}
    for pm in placed:
        for mat, qty in pm.machine_type.build_cost:
            build_cost[mat] = build_cost.get(mat, 0) + qty

    # 6. Power gate
    total_demand = sum(pm.machine_type.power_cost for pm in placed)
    battery_count = sum(1 for pm in placed if pm.machine_type.id == 'ther_bank')
    bat_power    = BATTERY_POWER.get(battery_item, 0) * max(battery_count, 1)
    power_ok     = bat_power >= total_demand

    # 7. Self-sustainability gate: critical path through production DAG
    in_nodes:  dict[str, list[str]] = defaultdict(list)
    out_nodes: dict[str, list[str]] = defaultdict(list)
    all_nodes: set[str]             = set()
    for fp in flow_matrix:
        in_nodes[fp.dst_id].append(fp.src_id)
        out_nodes[fp.src_id].append(fp.dst_id)
        all_nodes.update([fp.src_id, fp.dst_id])

    indeg = {n: len(in_nodes[n]) for n in all_nodes}
    queue = deque(n for n in all_nodes if indeg[n] == 0)
    topo: list[str] = []
    while queue:
        node = queue.popleft()
        topo.append(node)
        for dst in out_nodes[node]:
            indeg[dst] -= 1
            if indeg[dst] == 0:
                queue.append(dst)

    arrival: dict[str, float] = {}
    for node in topo:
        pt = proc_time.get(node, 0)
        if not in_nodes[node]:
            arrival[node] = pt
        else:
            best = max(
                arrival.get(src, 0)
                + path_cache.get((src, node), grid.width + grid.height) / max(1, BELT_SPEED)
                for src in in_nodes[node]
            )
            arrival[node] = best + pt

    ther_bank_times = [v for k, v in arrival.items() if k.startswith('ther_bank_')]
    chain_time  = min(ther_bank_times) if ther_bank_times else float('inf')
    sum_proc    = sum(proc_time.get(n, 0) for n in all_nodes
                      if not n.startswith('ther_bank_'))
    belt_secs   = max(0.0, chain_time - sum_proc)
    sustain_ok  = chain_time <= SIM_EQ_MAX_TICKS

    # 8. Aggregate
    # all_placed evaluates against the ORIGINAL chromosome len
    all_placed = len(placed) == chromosome_len
    paths_total = len(flow_matrix)
    feasible   = power_ok and sustain_ok and all_placed and (paths_found == paths_total)

    return FitnessResult(
        mhc=mhc, revenue=revenue, build_cost=build_cost,
        all_placed=all_placed, paths_found=paths_found,
        paths_total=paths_total, belt_cells_used=belt_cells_used,
        total_power_demand=total_demand,
        battery_power_output=bat_power, power_feasible=power_ok,
        chain_time=chain_time, startup_time=chain_time,
        self_sustaining=sustain_ok, belt_budget_used=belt_secs,
        feasible=feasible,
    )

def evaluate_fitness(
    chromosome:   Chromosome,
    battery_item: str = 'SC_bat',
) -> tuple[FitnessResult, FactoryGrid, list[FlowPair]]:
    """Full fitness evaluation with 3-gate feasibility model."""

    # 1. Decode chromosome -> grid
    grid   = FactoryGrid()
    placed = blf_decode(chromosome, grid)

    # 2. Derive flow matrix from recipes
    flow_matrix   = derive_flow_matrix(placed)
    placed_lookup = {pm.instance_id: pm for pm in placed}

    # 3. Route belts
    belt_results = route_belts(grid, placed_lookup, flow_matrix)

    # 3.5 Repair starved connections (magic depots allowed as per user request)
    rep_count, rep_machines, rep_belts = repair_feasibility(
        grid, placed, flow_matrix, belt_results,
        battery_item=battery_item, verbose=False)
    if rep_machines:
        placed.extend(rep_machines)          # repair_feasibility no longer self-appends
    flow_matrix, belt_results = apply_repair_belts(
        placed, flow_matrix, belt_results, rep_belts)

    # 3.6 Cull unlinked machines -- STILL BYPASSED (do not delete knapsack machines)

    # 4-8. Calculate metrics
    res = calculate_metrics(grid, placed, flow_matrix, belt_results, len(chromosome.genes), battery_item)
    return res, grid, flow_matrix

print("FitnessResult, calculate_metrics() and evaluate_fitness() defined.")




### 14.4  `evaluate_with_simulation()` — unified pipeline

In [ ]:
def evaluate_with_simulation(
    chromosome:    object,           # Chromosome
    battery_item:  str  = 'SC_bat',
    max_sim_ticks: int  = 350,
    eq_window:     int  = 30,
    eq_threshold:  float= 0.08,
    verbose:       bool = True,
) -> tuple:
    """
    Full evaluation = static fitness + discrete-time simulation.

    Returns
    -------
    (FitnessResult, FactoryGrid, flow_matrix, SimEngine, eq_tick)

    Gate 2 update
    -------------
    Gate 2 (self-sustainability) is now checked against the *simulation* result:
      equilibrium_tick <= SIM_EQ_MAX_TICKS ticks
    If no equilibrium is reached, Gate 2 fails regardless of analytical chain_time.
    The FitnessResult.self_sustaining and FitnessResult.feasible flags are
    updated in-place to reflect the simulation result.
    """
    # 1. Static evaluation (existing pipeline)
    result, grid, flow_matrix = evaluate_fitness(chromosome, battery_item)

    if verbose:
        print(f"Static evaluation done:")
        print(f"  MHC={result.mhc:.1f}  revenue={result.revenue:.2f}  "
              f"chain_time={result.chain_time:.1f} ticks  "
              f"feasible={result.feasible}")

    # 2. Build simulation inputs
    belt_paths      = build_belt_paths(grid, flow_matrix)
    raw_supply_ids  = find_dag_roots(grid.placed_machines, flow_matrix)

    if verbose:
        print(f"  Belt paths for simulation: {len(belt_paths)}")
        print(f"  Raw supply sources (DAG roots): {raw_supply_ids}")

    # 3. Run simulation
    engine = SimEngine(
        placed_machines = grid.placed_machines,
        belt_paths      = belt_paths,
        flow_matrix     = flow_matrix,
        revenue_table   = REVENUE_TABLE,
        raw_supply_ids  = raw_supply_ids,
        inject_every    = 1,
    )
    eq_tick = engine.run(
        max_ticks=max_sim_ticks,
        window=eq_window,
        threshold=eq_threshold,
        verbose=verbose,
    )

    # 4. Update Gate 2 with simulation result
    sim_sustain = eq_tick is not None and eq_tick <= SIM_EQ_MAX_TICKS
    result.self_sustaining = sim_sustain
    result.startup_time    = eq_tick if eq_tick else float('inf')
    result.feasible        = (result.power_feasible
                               and sim_sustain
                               and result.all_placed)

    if verbose:
        sim_rev = engine.simulated_revenue_per_tick(window=eq_window)
        print(f"  Sim revenue/tick (at eq): {sim_rev:.3f}  "
              f"(static: {result.revenue:.3f})")
        print(f"  Gate 2 (simulation):  "
              f"{'PASS' if sim_sustain else 'FAIL'}  "
              f"(eq_tick={eq_tick} ≤ {SIM_EQ_MAX_TICKS}?)")
        print(f"  Overall feasible: {result.feasible}")

    return result, grid, flow_matrix, engine, eq_tick


print("evaluate_with_simulation() defined.")
print(f"Gate 2 now: equilibrium_tick ≤ SIM_EQ_MAX_TICKS ({SIM_EQ_MAX_TICKS} ticks)")


### 14.5  Simulation-Aware Search Fitness

Lightweight wrapper that runs static evaluation first, then a short simulation
on feasible layouts to rank by actual simulated revenue.

In [ ]:
def evaluate_search_fitness(chromosome, battery_item):
    """
    Simulation-aware fitness for search functions.
    1. Run static evaluation (fast gate check)
    2. If all 3 static gates pass, run a short simulation
    3. Use -sim_revenue as the MHC value (so minimisation = maximise revenue)
    """
    res, grid, fm = evaluate_fitness(chromosome, battery_item)
    if res.feasible:
        sim_res, _, _, _, eq_tick = evaluate_with_simulation(
            chromosome, battery_item,
            max_sim_ticks=80, eq_window=15, verbose=False)
        if eq_tick is not None:
            sim_rev = sim_res.revenue
            res.mhc = -sim_rev   # negative so that minimisation maximises revenue
        else:
            res.mhc = 999999.0   # no equilibrium = worst score
    return res, grid, fm

print('evaluate_search_fitness() defined.')
print('  Static gates filter first, then short 80-tick simulation ranks survivors.')


## 15. Rotation Search

BLF with `R0` for all machines is a valid starting point but not optimal.
Rotating machines can orient I/O ports toward their neighbours, shortening belts.

This cell runs a **greedy rotation search**: for each machine in the chromosome,
try all 4 rotations and keep the one that gives the lowest MHC, holding all other
machines fixed. One pass over all genes (can be iterated).


In [ ]:

def greedy_rotation_search(
    base_chromo:  Chromosome,
    battery_item: str = 'SC_bat',
    n_passes:     int = 2,
) -> tuple[Chromosome, float]:
    """
    Greedy per-gene rotation optimisation.
    For each gene, try all 4 rotations, keep the best (lowest MHC).
    Repeat for n_passes.
    """
    best_chromo = base_chromo.copy()
    best_res, _, _ = evaluate_search_fitness(best_chromo, battery_item)
    best_mhc = best_res.mhc

    for pass_n in range(n_passes):
        improved = False
        for i in range(len(best_chromo.genes)):
            for rot in list(Rotation):
                candidate = best_chromo.copy()
                candidate.genes[i].rotation = rot
                res, _, _ = evaluate_search_fitness(candidate, battery_item)
                if res.mhc < best_mhc:
                    best_mhc    = res.mhc
                    best_chromo = candidate
                    improved    = True
        print(f"  Pass {pass_n + 1}: MHC = {best_mhc:.1f}  {'(improved)' if improved else '(no change)'}")
        if not improved:
            break

    return best_chromo, best_mhc




## 16. Gene Order Search

BLF placement order controls **where** each machine lands on the grid.
Machines placed earlier get the bottom-left positions; later machines fill in around them.

At `CFG.MAX_MACHINES` (160) an exhaustive permutation search is hopeless, so the ordering is derived from the production graph (`_order_instances`) and the arrangement is left to ALNS/Tabu.
Instead we use two fast heuristics:

1. **Neighbourhood swap**: randomly swap two genes, keep the swap if MHC improves
2. **Production-aware seed**: put upstream machines first so they land near depot
   loaders, and downstream machines (packaging, ther_bank) come last


In [ ]:
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional
import itertools


@dataclass
class ProductSpec:
    """
    Complete description of a production chain for one revenue product.

    Attributes
    ----------
    name            : output item name (must be in REVENUE_TABLE)
    revenue         : credits per unit of output
    chain_depth     : number of processing steps in the critical path
    raw_materials   : {material: qty_per_output_unit}  — depot loader inputs
    machine_types   : {machine_type_id: count}  — machines needed per line
    total_power     : total power demand of all machines in one line
    critical_path_s : critical path process time (excluding belt travel)
    terminal_recipe : (machine_type_id, recipe_idx) of the final machine
    power_battery   : which battery item feeds the ther_bank for this chain
    """
    name:             str
    revenue:          int
    chain_depth:      int
    raw_materials:    Dict[str, int]
    machine_types:    Dict[str, int]
    total_power:      int
    critical_path_s:  int
    terminal_recipe:  Tuple[str, int]
    power_battery:    str = 'SC_bat'

    def revenue_per_power(self) -> float:
        return self.revenue / max(self.total_power, 1)

    def revenue_per_depth(self) -> float:
        return self.revenue / max(self.chain_depth, 1)

    def footprint_cells(self) -> int:
        """Approximate grid cells consumed by one production line."""
        total = 0
        for mid, count in self.machine_types.items():
            mt = MACHINE_CATALOGUE[mid]
            total += mt.base_w * mt.base_h * count
        return total


# ── All viable product specs (derived from recipe graph) ─────────────────────
ALL_PRODUCTS: Dict[str, ProductSpec] = {

    # ── Filling products (rev=70) ─────────────────────────────────────────────
    # buck_cap_A: steel_bottle×10 + ground_buck_pow×10 -> buck_cap_A
    #   steel chain:  ferri_ore -> refin(r2) -> ferrium -> shred(r1) -> ferri_pow
    #                 -> grind(r3)×2ferri_pow -> dense_ferri_pow -> refin(r10) -> steel
    #                 -> mould(r0)×2steel -> steel_bottle
    #   buck chain:   buckflower -> shred(r5) -> buck_pow×2
    #                 sandleaf -> shred(r7) -> sandleaf_pow×2  (common_extra for grind r0)
    #                 grind(r0): buck_pow×2 + sandleaf_pow×1 -> ground_buck_pow
    'buck_cap_A': ProductSpec(
        name='buck_cap_A', revenue=70, chain_depth=5,
        raw_materials={'ferri_ore': 40, 'buckflower': 10, 'sandleaf': 5},
        machine_types={
            'refining_unit': 2, 'shredding_unit': 3,
            'grinding_unit': 2, 'moulding_unit': 1, 'filling_unit': 1,
        },
        total_power = 2*10 + 3*10 + 2*20 + 1*10 + 1*20,  # 120
        critical_path_s = 5 * 2,   # 5 processing steps × 2s each = 10s + belt
        terminal_recipe = ('filling_unit', 0),
        power_battery = 'SC_bat',
    ),

    # can_cit_A: replace ground_buck_pow with ground_citrome_pow
    #   citrome chain: citrome -> shred(r6) -> citrome_pow×2 -> grind(r5)×2 -> ground_citrome_pow
    'can_cit_A': ProductSpec(
        name='can_cit_A', revenue=70, chain_depth=5,
        raw_materials={'ferri_ore': 40, 'citrome': 10},
        machine_types={
            'refining_unit': 2, 'shredding_unit': 2,
            'grinding_unit': 2, 'moulding_unit': 1, 'filling_unit': 1,
        },
        total_power = 2*10 + 2*10 + 2*20 + 1*10 + 1*20,  # 110
        critical_path_s = 5 * 2,
        terminal_recipe = ('filling_unit', 1),
        power_battery = 'SC_bat',
    ),

    # ── Packaging products ────────────────────────────────────────────────────
    # SC_bat (rev=30): ferri_part×10 + ori_pow×15 -> SC_bat
    #   ferri_part: ferri_ore -> refin(r2) -> ferrium -> fit(r1) -> ferri_part
    #   ori_pow:    ori_ore -> shred(r0) -> ori_pow
    'SC_bat': ProductSpec(
        name='SC_bat', revenue=30, chain_depth=3,
        raw_materials={'ferri_ore': 10, 'ori_ore': 15},
        machine_types={
            'refining_unit': 1, 'fitting_unit': 1,
            'shredding_unit': 1, 'packaging_unit': 1,
        },
        total_power = 10 + 10 + 10 + 20,   # 50
        critical_path_s = 3 * 2,
        terminal_recipe = ('packaging_unit', 2),
        power_battery = 'SC_bat',
    ),

    # HC_bat (rev=27): steel_part×10 + dense_ori_pow×15 -> HC_bat
    #   steel: ferri_ore -> refin -> shred -> grind -> refin -> fit
    #   dense_ori_pow: ori_ore -> shred -> ori_pow -> grind(r1)×2 -> dense_ori_pow
    'HC_bat': ProductSpec(
        name='HC_bat', revenue=27, chain_depth=5,
        raw_materials={'ferri_ore': 10, 'ori_ore': 30},
        machine_types={
            'refining_unit': 2, 'shredding_unit': 2,
            'grinding_unit': 2, 'fitting_unit': 1, 'packaging_unit': 1,
        },
        total_power = 2*10 + 2*10 + 2*20 + 1*10 + 1*20,  # 110
        critical_path_s = 5 * 2,
        terminal_recipe = ('packaging_unit', 0),
        power_battery = 'HC_bat',  # HC_bat powers HC_bat chain (1100 >> 110)
    ),

    # LC_bat (rev=16): ame_part×5 + ori_pow×10 -> LC_bat
    #   ame_part: ame_ore -> refin(r1) -> ame_fiber -> fit(r3) -> ame_part
    #   ori_pow:  ori_ore -> shred(r0) -> ori_pow
    'LC_bat': ProductSpec(
        name='LC_bat', revenue=16, chain_depth=3,
        raw_materials={'ame_ore': 5, 'ori_ore': 10},
        machine_types={
            'refining_unit': 1, 'fitting_unit': 1,
            'shredding_unit': 1, 'packaging_unit': 1,
        },
        total_power = 10 + 10 + 10 + 20,   # 50
        critical_path_s = 3 * 2,
        terminal_recipe = ('packaging_unit', 1),
        power_battery = 'LC_bat',
    ),

    # buck_cap_B (rev=27): ferrium_bottle×10 + ground_buck_pow×10 -> buck_cap_B
    #   ferrium_bottle: ferri_ore -> refin -> ferrium -> mould(r1)×2 -> ferrium_bottle
    'buck_cap_B': ProductSpec(
        name='buck_cap_B', revenue=27, chain_depth=4,
        raw_materials={'ferri_ore': 20, 'buckflower': 10, 'sandleaf': 5},
        machine_types={
            'refining_unit': 1, 'shredding_unit': 2,
            'grinding_unit': 1, 'moulding_unit': 1, 'filling_unit': 1,
        },
        total_power = 10 + 2*10 + 20 + 10 + 20,  # 80
        critical_path_s = 4 * 2,
        terminal_recipe = ('filling_unit', 2),
        power_battery = 'SC_bat',
    ),

    # can_cit_B (rev=27): ferrium_bottle×10 + ground_citrome_pow×10 -> can_cit_B
    'can_cit_B': ProductSpec(
        name='can_cit_B', revenue=27, chain_depth=4,
        raw_materials={'ferri_ore': 20, 'citrome': 10},
        machine_types={
            'refining_unit': 1, 'shredding_unit': 2,
            'grinding_unit': 1, 'moulding_unit': 1, 'filling_unit': 1,
        },
        total_power = 10 + 2*10 + 20 + 10 + 20,  # 80
        critical_path_s = 4 * 2,
        terminal_recipe = ('filling_unit', 3),
        power_battery = 'SC_bat',
    ),
}

# ── Summarise ─────────────────────────────────────────────────────────────────
print(f"{'Product':15s} {'Rev':>5} {'Depth':>6} {'Power':>6} {'Rev/Pwr':>8} "
      f"{'Footprint':>10} {'Raw materials'}")
print("-" * 85)
for p in sorted(ALL_PRODUCTS.values(), key=lambda x: -x.revenue):
    rm  = ", ".join(f"{m}×{q}" for m, q in p.raw_materials.items())
    fp  = p.footprint_cells()
    rpp = p.revenue_per_power()
    print(f"  {p.name:13s} {p.revenue:>5} {p.chain_depth:>6} {p.total_power:>6} "
          f"{rpp:>8.3f} {fp:>10}  {rm}")


In [ ]:
# ─── LEGACY: superseded helpers ────────────────────────────────────────────────
# print_grid()      -> use plot_layout()
# build_belt_paths(), find_dag_roots()
#     -> replaced by build_belt_paths_from_routed() and build_injection_map().
#        build_belt_paths reconstructed belts by re-parsing "{src}->{dst}" grid
#        labels, which collapsed multiple materials between the same machine
#        pair and picked up superseded repair belts. find_dag_roots returned
#        "machines with no incoming flow that have a recipe" -- in the v4
#        pipeline main_unit feeds every raw material, so almost nothing
#        qualified, the injection set came back empty, and the simulation
#        produced literally nothing.

def print_grid(grid, flow_matrix):
    """Fallback ASCII representation of the grid."""
    print("\n--- ASCII GRID FALLBACK ---")

    # 1. Build a 2D array of chars
    arr = [['.' for _ in range(grid.width)] for _ in range(grid.height)]

    # 2. Mark belts
    for _, path in grid.belt_paths:
        for x, y in path:
            if grid.in_bounds(x, y):
                arr[y][x] = '*'

    # 3. Mark machines
    for pm in grid.placed_machines:
        # Use first letter of machine ID
        char = pm.machine_type.id[0].upper()
        for dy in range(pm.h):
            for dx in range(pm.w):
                if grid.in_bounds(pm.x + dx, pm.y + dy):
                    arr[pm.y + dy][pm.x + dx] = char

    # 4. Print
    # Print column numbers (tens)
    print("   " + "".join([str(i // 10) if i % 10 == 0 else " " for i in range(grid.width)]))
    # Print column numbers (units)
    print("   " + "".join([str(i % 10) for i in range(grid.width)]))

    for y in range(grid.height):
        row_str = "".join(arr[y])
        print(f"{y:2d} {row_str}")
    print("---------------------------\n")

def build_belt_paths(
    grid:        FactoryGrid,
    flow_matrix: list,          # list[FlowPair] — instance-level, from derive_flow_matrix()
) -> list:
    """Convert grid.belt_paths into BeltPath simulation objects.

    grid.belt_paths stores (label, path) where label = "{src_id}->{dst_id}".
    We cross-reference with flow_matrix to find the material each belt carries.
    """
    # Build lookup: (src_id, dst_id) -> FlowPair
    flow_lookup = {}
    for fp in flow_matrix:
        flow_lookup[(fp.src_id, fp.dst_id)] = fp

    belt_paths = []
    for label, path in grid.belt_paths:
        # Label format: "{src_instance_id}->{dst_instance_id}"
        if '->' not in label:
            continue
        src_id, dst_id = label.split('->', 1)
        fp = flow_lookup.get((src_id, dst_id))
        if fp is None:
            continue
        belt_paths.append(BeltPath(
            src_id    = src_id,
            dst_id    = dst_id,
            material  = fp.material,
            path      = list(path),
            intensity = fp.intensity,
        ))

    return belt_paths


def find_dag_roots(
    placed_machines: list,
    flow_matrix:     list,
) -> set:
    """Instance IDs that have NO incoming flow pairs.
    These are the start of production chains — they receive free raw
    material injection each tick (simulating depot loader supply)."""
    has_incoming = {fp.dst_id for fp in flow_matrix}
    return {
        pm.instance_id
        for pm in placed_machines
        if pm.instance_id not in has_incoming
        and pm.machine_type.recipes               # skip utilities
    }


print("build_belt_paths() and find_dag_roots() defined.")

print("[LEGACY] print_grid(), build_belt_paths(), find_dag_roots() defined (unused).")
